

# PoliMillionaire Hybrid RAG Pipeline - Notebook 12 V3

This notebook is a production-oriented continuation of notebook 11. It keeps the same local GGUF LLM, retrieval indexes, Colab paths, and PoliMillionaire API loop, but replaces the weak parts observed in the logs with a stricter and more auditable design.





## Design Notes

The pipeline follows the assignment constraints: all LLM inference is local, retrieval uses local indexes or raw evidence only, and tool use is executed by Python rather than by the model.

Compared with notebook 11, this version adds three production changes:

1. **Validated generic Maths tools.** Tool calls are parsed as JSON, checked against schemas, guarded against semantically incompatible calls, executed deterministically, and matched to options without relying on a free-form LLM matcher.
2. **Option-wise evidence retrieval.** Factual questions, especially Entertainment, can retrieve evidence for each answer option instead of relying only on one global query.
3. **Richer decision logging.** Logs include confidence, retrieval score summaries, option evidence scores, validated tool traces, rejection reasons, and fallback indicators.

The implementation intentionally avoids LangChain to keep the notebook self-contained, but follows the same best-practice pattern: structured tool call -> validation -> controlled execution -> structured result -> deterministic answer matching.

V2 changes: GBNF-constrained final option generation, adaptive option-wise retrieval, and conservative deterministic Maths fixes from V1 logs.

V3 changes: extended generic Maths tools, analysis-first JSON router, constrained Micro-CoT Maths fallback, clearer fallback logging, and deterministic tool asserts.





## 1. Install dependencies





### Optional fallback: rebuild llama-cpp-python with CUDA



In [2]:


# Run this ONLY if the wheel above fails or does not use the GPU.
# It can take several minutes.
# !CMAKE_ARGS="-DGGML_CUDA=on" FORCE_CMAKE=1 pip install --no-cache-dir --force-reinstall llama-cpp-python



In [12]:
import os

# Evita che transformers/sentence-transformers cerchino backend non necessari.
os.environ["USE_FLAX"] = "0"
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_NO_TF"] = "1"

# Se JAX fosse ancora presente, non deve usare GPU.
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["JAX_PLATFORM_NAME"] = "cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Backend guard set.")

Backend guard set.


In [13]:
%pip uninstall -y jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt flax optax chex orbax-checkpoint
%pip uninstall -y torchcodec

%pip install -q --no-cache-dir --force-reinstall \
  "numpy==2.0.2" \
  "scipy==1.14.1" \
  "scikit-learn==1.6.1"

%pip install -q --no-cache-dir --force-reinstall \
  "bm25s" \
  "hnswlib" \
  "diskcache" \
  "jinja2" \
  "typing-extensions" \
  "huggingface_hub"

%pip install -q --no-cache-dir --force-reinstall \
  torch torchvision torchaudio \
  --index-url https://download.pytorch.org/whl/cu124

%pip install -q --no-cache-dir --force-reinstall --no-deps \
  "sentence-transformers" \
  "transformers" \
  "huggingface_hub"

print("Cleaned JAX/Flax and repaired core deps. Restart runtime now.")
raise SystemExit("Restart runtime required.")

Found existing installation: jax 0.7.2
Uninstalling jax-0.7.2:
  Successfully uninstalled jax-0.7.2
Found existing installation: jaxlib 0.7.2
Uninstalling jaxlib-0.7.2:
  Successfully uninstalled jaxlib-0.7.2
Found existing installation: jax-cuda12-plugin 0.7.2
Uninstalling jax-cuda12-plugin-0.7.2:
  Successfully uninstalled jax-cuda12-plugin-0.7.2
Found existing installation: jax-cuda12-pjrt 0.7.2
Uninstalling jax-cuda12-pjrt-0.7.2:
  Successfully uninstalled jax-cuda12-pjrt-0.7.2
Found existing installation: flax 0.11.2
Uninstalling flax-0.11.2:
  Successfully uninstalled flax-0.11.2
Found existing installation: optax 0.2.8
Uninstalling optax-0.2.8:
  Successfully uninstalled optax-0.2.8
Found existing installation: orbax-checkpoint 0.11.40
Uninstalling orbax-checkpoint-0.11.40:
  Successfully uninstalled orbax-checkpoint-0.11.40
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dep

SystemExit: Restart runtime required.

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)




## 2. Mount Drive, paths, and token setup



In [1]:


from pathlib import Path
import os, sys, json, time, math, re, shutil, gc

try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    userdata = None

PROJECT_ROOT = Path('/content/drive/MyDrive/nlp26') if IN_COLAB else Path.cwd()
PROJECT_SRC_DIR = PROJECT_ROOT / 'project' / 'src'
LEGACY_SRC_DIR = PROJECT_ROOT / 'src'
SRC_DIR = PROJECT_SRC_DIR if PROJECT_SRC_DIR.exists() else LEGACY_SRC_DIR
API_BASE_DIR = PROJECT_ROOT / 'api_client'
# This is the directory containing the 'millionaire_client' package folder
API_CLIENT_DIR = API_BASE_DIR / 'NLP_assignment_api_client'
DRIVE_INDEX_DIR = PROJECT_ROOT / 'indexes'
LOG_DIR = PROJECT_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_ROOT = Path('/content/nlp26_runtime') if IN_COLAB else PROJECT_ROOT / '.runtime'
LOCAL_INDEX_DIR = LOCAL_ROOT / 'indexes'
LOCAL_MODEL_DIR = Path('/content/models') if IN_COLAB else PROJECT_ROOT / 'models'
LOCAL_HF_CACHE = Path('/content/hf_cache') if IN_COLAB else PROJECT_ROOT / '.hf_cache'

# Add all possible source directories to sys.path
# We ensure the parent of the package is in sys.path
for p in [SRC_DIR, PROJECT_SRC_DIR, LEGACY_SRC_DIR, API_BASE_DIR, API_CLIENT_DIR, PROJECT_ROOT]:
    if p.exists() and str(p) not in sys.path:
        sys.path.append(str(p))

if IN_COLAB:
    try:
        token = userdata.get('HF_TOKEN')
        if token:
            os.environ['HF_TOKEN'] = token
    except Exception as e:
        print('Could not read Colab secret HF_TOKEN:', repr(e))

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['HF_HOME'] = str(LOCAL_HF_CACHE)

print('API_CLIENT_DIR exists:', API_CLIENT_DIR.exists())
if API_CLIENT_DIR.exists():
    print('Contents of', API_CLIENT_DIR, ':', os.listdir(API_CLIENT_DIR))
print('PROJECT_SRC_DIR exists:', PROJECT_SRC_DIR.exists())
print('LEGACY_SRC_DIR exists:', LEGACY_SRC_DIR.exists())
print('Selected SRC_DIR:', SRC_DIR)
print('sys.path includes API_CLIENT_DIR:', str(API_CLIENT_DIR) in sys.path)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
API_CLIENT_DIR exists: True
Contents of /content/drive/MyDrive/nlp26/api_client/NLP_assignment_api_client : ['PoliMillionaire.ipynb', 'millionaire_client']
PROJECT_SRC_DIR exists: False
LEGACY_SRC_DIR exists: True
Selected SRC_DIR: /content/drive/MyDrive/nlp26/src
sys.path includes API_CLIENT_DIR: True


In [2]:


# The API client is a package folder under API_CLIENT_DIR, not an installable project.
# The path setup cell above adds API_CLIENT_DIR to sys.path, so a direct import is enough.
print('API_CLIENT_DIR:', API_CLIENT_DIR)
print('millionaire_client package exists:', (API_CLIENT_DIR / 'millionaire_client').exists())

from millionaire_client import MillionaireClient
print('millionaire_client import OK:', MillionaireClient)



API_CLIENT_DIR: /content/drive/MyDrive/nlp26/api_client/NLP_assignment_api_client
millionaire_client package exists: True
millionaire_client import OK: <class 'millionaire_client.client.MillionaireClient'>


In [3]:


# Optional Drive cleanup. Keep commented during normal runs.
# from google.colab import drive
# drive.flush_and_unmount()





## 3. Memory helpers



In [4]:


import psutil

try:
    import torch
except Exception:
    torch = None

def mem_report(label=''):
    print(f"\n[MEM] {label}")
    vm = psutil.virtual_memory()
    print(f"CPU RAM: {vm.used/1024**3:.2f} / {vm.total/1024**3:.2f} GiB ({vm.percent:.1f}%)")
    if torch is not None and torch.cuda.is_available():
        print(f"GPU torch allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GiB")
        print(f"GPU torch reserved:  {torch.cuda.memory_reserved()/1024**3:.2f} GiB")

def cleanup_memory():
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

mem_report('initial')




[MEM] initial
CPU RAM: 1.60 / 52.96 GiB (4.2%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB




## 4. Copy index files from Drive to local Colab disk



In [5]:


INDEX_FILES = {
    'simplewiki_bm25': 'simplewiki_160w_title2_stop_bm25.joblib',
    'simplewiki_dense_index': 'simplewiki_160w_dense_hnsw.index',
    'simplewiki_dense_meta': 'simplewiki_160w_dense_meta.joblib',
    'kelm_bm25': 'kelm_500k_stop_bm25.joblib',
    'kelm_dense_index': 'kelm_500k_dense_hnsw.index',
    'kelm_dense_meta': 'kelm_500k_dense_meta.joblib',
    'textbook_introductory_statistics': 'introductory_statistics_2e_200w_section2_stop_ngram2_bm25.joblib',
    'textbook_algebra_trigonometry': 'algebra_trigonometry_2e_200w_section2_stop_ngram2_bm25.joblib',
    'textbook_calculus_volume_1': 'calculus_volume_1_200w_section2_stop_ngram2_bm25.joblib',
    'textbook_discrete_math': 'discrete_math_open_intro_200w_section2_stop_ngram2_bm25.joblib',
    'textbook_abstract_algebra': 'abstract_algebra_judson_200w_section2_stop_ngram2_bm25.joblib',
    'textbook_basic_analysis': 'basic_analysis_1_200w_section2_stop_ngram2_bm25.joblib',
    'textbook_topology': 'topology_without_tears_200w_section2_stop_ngram2_bm25.joblib',
    'textbook_introductory_statistics_dense_index': 'introductory_statistics_2e_200w_dense_hnsw.index',
    'textbook_introductory_statistics_dense_meta': 'introductory_statistics_2e_200w_dense_meta.joblib',
    'textbook_algebra_trigonometry_dense_index': 'algebra_trigonometry_2e_200w_dense_hnsw.index',
    'textbook_algebra_trigonometry_dense_meta': 'algebra_trigonometry_2e_200w_dense_meta.joblib',
    'textbook_calculus_volume_1_dense_index': 'calculus_volume_1_200w_dense_hnsw.index',
    'textbook_calculus_volume_1_dense_meta': 'calculus_volume_1_200w_dense_meta.joblib',
    'textbook_discrete_math_dense_index': 'discrete_math_open_intro_200w_dense_hnsw.index',
    'textbook_discrete_math_dense_meta': 'discrete_math_open_intro_200w_dense_meta.joblib',
    'textbook_abstract_algebra_dense_index': 'abstract_algebra_judson_200w_dense_hnsw.index',
    'textbook_abstract_algebra_dense_meta': 'abstract_algebra_judson_200w_dense_meta.joblib',
    'textbook_basic_analysis_dense_index': 'basic_analysis_1_200w_dense_hnsw.index',
    'textbook_basic_analysis_dense_meta': 'basic_analysis_1_200w_dense_meta.joblib',
    'textbook_topology_dense_index': 'topology_without_tears_200w_dense_hnsw.index',
    'textbook_topology_dense_meta': 'topology_without_tears_200w_dense_meta.joblib',
}

def copy_indexes_to_local():
    LOCAL_INDEX_DIR.mkdir(parents=True, exist_ok=True)
    out = {}
    for key, filename in INDEX_FILES.items():
        src = DRIVE_INDEX_DIR / filename
        dst = LOCAL_INDEX_DIR / filename
        if not src.exists():
            raise FileNotFoundError(f'Missing index file on Drive: {src}')
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            print(f'Copying {filename} -> {dst}')
            shutil.copy2(src, dst)
        out[key] = dst
    return out

LOCAL_INDEX_FILES = copy_indexes_to_local()
for key, path in LOCAL_INDEX_FILES.items():
    print(f'{key:28s}', path.exists(), f'{path.stat().st_size/1024**2:.1f} MB', path)

mem_report('after local index cache')



simplewiki_bm25              True 244.0 MB /content/nlp26_runtime/indexes/simplewiki_160w_title2_stop_bm25.joblib
simplewiki_dense_index       True 750.3 MB /content/nlp26_runtime/indexes/simplewiki_160w_dense_hnsw.index
simplewiki_dense_meta        True 121.0 MB /content/nlp26_runtime/indexes/simplewiki_160w_dense_meta.joblib
kelm_bm25                    True 40.5 MB /content/nlp26_runtime/indexes/kelm_500k_stop_bm25.joblib
kelm_dense_index             True 864.2 MB /content/nlp26_runtime/indexes/kelm_500k_dense_hnsw.index
kelm_dense_meta              True 21.6 MB /content/nlp26_runtime/indexes/kelm_500k_dense_meta.joblib
textbook_introductory_statistics True 3.1 MB /content/nlp26_runtime/indexes/introductory_statistics_2e_200w_section2_stop_ngram2_bm25.joblib
textbook_algebra_trigonometry True 2.9 MB /content/nlp26_runtime/indexes/algebra_trigonometry_2e_200w_section2_stop_ngram2_bm25.joblib
textbook_calculus_volume_1   True 1.5 MB /content/nlp26_runtime/indexes/calculus_volume_1_200


## 5. Download and load Qwen3.5-9B Q8_0 GGUF — unified model for every section


In [6]:

from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# Unified LLM backend for ALL sections, including Maths.
# The current bartowski repo exposes the Q8 quant as Qwen3.5-9B-Q8_0.gguf.
# Keep a legacy filename candidate as a safety fallback in case an older mirror is used.
MODEL_REPO = 'bartowski/Qwen_Qwen3.5-9B-GGUF'
MODEL_REVISION = 'main'
MODEL_FILE_CANDIDATES = [
    ('Qwen3.5-9B-Q8_0.gguf', int(8.8 * 1024**3)),       # about 9.8 GB decimal / 9.1 GiB
    ('Qwen_Qwen3.5-9B-Q8_0.gguf', int(8.8 * 1024**3)),  # legacy/alternate naming fallback
]
MODEL_FILE = MODEL_FILE_CANDIDATES[0][0]
MODEL_MIN_BYTES = MODEL_FILE_CANDIDATES[0][1]


def _gguf_status(path, min_bytes=MODEL_MIN_BYTES):
    path = Path(path)
    if not path.exists():
        return {'exists': False, 'size': 0, 'magic': None, 'valid': False}
    size = path.stat().st_size
    with path.open('rb') as f:
        magic = f.read(4)
    return {
        'exists': True,
        'size': size,
        'magic': magic,
        'valid': magic == b'GGUF' and size >= min_bytes,
    }


def download_gguf(force=False):
    last_error = None
    for filename, min_bytes in MODEL_FILE_CANDIDATES:
        try:
            path = Path(hf_hub_download(
                repo_id=MODEL_REPO,
                filename=filename,
                revision=MODEL_REVISION,
                local_dir=str(LOCAL_MODEL_DIR),
                token=os.environ.get('HF_TOKEN'),
                force_download=force,
            ))
            status = _gguf_status(path, min_bytes=min_bytes)
            print('MODEL_REPO:', MODEL_REPO)
            print('MODEL_FILE:', filename)
            print('MODEL_PATH:', path)
            print('MODEL_REVISION:', MODEL_REVISION)
            print('Model file size:', status['size'] / 1024**3, 'GiB')
            print('GGUF magic:', status['magic'])
            if status['valid']:
                globals()['MODEL_FILE'] = filename
                globals()['MODEL_MIN_BYTES'] = min_bytes
                return path, status
            last_error = RuntimeError(f'Invalid GGUF candidate {filename}: {status}')
        except Exception as exc:
            last_error = exc
            print(f'Could not use GGUF candidate {filename}:', repr(exc))

    raise RuntimeError(
        f'Could not download a valid Qwen3.5-9B Q8_0 GGUF from {MODEL_REPO}. '
        f'Last error: {last_error}'
    )


MODEL_PATH, model_status = download_gguf(force=False)
if not model_status['valid']:
    print('Model file is missing, incomplete, or not a GGUF file. Removing it and forcing a clean download...')
    try:
        Path(MODEL_PATH).unlink()
    except FileNotFoundError:
        pass
    MODEL_PATH, model_status = download_gguf(force=True)

if not model_status['valid']:
    raise RuntimeError(
        f'Invalid GGUF after download: path={MODEL_PATH}, '
        f"size={model_status['size']}, magic={model_status['magic']}. "
        'Restart the Colab runtime, delete /content/models, and rerun the download cell.'
    )

mem_report('after Qwen3.5-9B Q8_0 GGUF download')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:277: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


Could not use GGUF candidate Qwen3.5-9B-Q8_0.gguf: RemoteEntryNotFoundError('404 Client Error. (Request ID: Root=1-6a19a3bf-084833ec1c628ed93b47fc00;72f29354-14f7-43b7-82bd-36b69b03a0fb)\n\nEntry Not Found for url: https://huggingface.co/bartowski/Qwen_Qwen3.5-9B-GGUF/resolve/main/Qwen3.5-9B-Q8_0.gguf.')
MODEL_REPO: bartowski/Qwen_Qwen3.5-9B-GGUF
MODEL_FILE: Qwen_Qwen3.5-9B-Q8_0.gguf
MODEL_PATH: /content/models/Qwen_Qwen3.5-9B-Q8_0.gguf
MODEL_REVISION: main
Model file size: 9.131191283464432 GiB
GGUF magic: b'GGUF'

[MEM] after Qwen3.5-9B Q8_0 GGUF download
CPU RAM: 1.66 / 52.96 GiB (4.4%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB


In [7]:

# Start conservative on a T4. Increase n_gpu_layers only after checking nvidia-smi.
# Unified backend: the same Qwen3.5-9B Q8_0 model is used for Maths and non-Maths.
# No separate Maths 7B model is downloaded or loaded.
import llama_cpp
import gc
import psutil

N_CTX = 4096
N_GPU_LAYERS = -1      # try 35, then -1 if memory is stable
N_BATCH = 256
N_THREADS = 2

# Compatibility variables kept for the existing logging code.
# Maths now uses the unified Qwen3.5-9B Q8_0 model, not a separate specialized Maths model.
MATH_LLM_IS_SPECIALIZED = False
MATH_LLM_LOAD_DEVICE = 'unified_qwen35_q8'
qwen_math_llm = None

print('llama-cpp-python:', getattr(llama_cpp, '__version__', 'unknown'))
print('Loading unified Qwen3.5-9B Q8_0 GGUF from:', MODEL_PATH)


def _available_ram_gb():
    return psutil.virtual_memory().available / 1024**3


def _cleanup_after_model_change(label=''):
    cleanup_memory() if 'cleanup_memory' in globals() else gc.collect()
    if label:
        mem_report(label) if 'mem_report' in globals() else None


def _load_llama_model(model_path, n_gpu_layers=N_GPU_LAYERS, label='model'):
    print(f'Loading {label}:', model_path)
    print(f'Available CPU RAM before {label}: {_available_ram_gb():.2f} GiB')
    return Llama(
        model_path=str(model_path),
        n_ctx=N_CTX,
        n_gpu_layers=n_gpu_layers,
        n_batch=N_BATCH,
        n_threads=N_THREADS,
        logits_all=False,
        verbose=False,
    )


def load_general_llm(unload_math_if_low_ram=True):
    """Return the unified Qwen3.5-9B Q8_0 LLM."""
    global qwen35_llm
    if 'qwen35_llm' in globals() and qwen35_llm is not None:
        return qwen35_llm
    qwen35_llm = _load_llama_model(MODEL_PATH, n_gpu_layers=N_GPU_LAYERS, label='unified Qwen3.5-9B Q8_0')
    _cleanup_after_model_change('after unified Qwen3.5-9B Q8_0 load')
    return qwen35_llm


def unload_general_llm():
    """Free the unified model if RAM cleanup is needed."""
    global qwen35_llm
    if 'qwen35_llm' in globals() and qwen35_llm is not None:
        print('Unloading unified Qwen3.5-9B Q8_0 model...')
        try:
            qwen35_llm.close()
        except Exception:
            pass
        qwen35_llm = None
        _cleanup_after_model_change('after unloading unified model')


def unload_math_llm():
    """Compatibility no-op: Maths uses the unified Qwen3.5-9B Q8_0 model."""
    global qwen_math_llm, MATH_LLM_IS_SPECIALIZED, MATH_LLM_LOAD_DEVICE
    qwen_math_llm = None
    MATH_LLM_IS_SPECIALIZED = False
    MATH_LLM_LOAD_DEVICE = 'unified_qwen35_q8'


def get_general_llm():
    """Use this in non-Maths sections."""
    return load_general_llm()


def load_math_llm(unload_general_if_low_ram=True):
    """Compatibility wrapper: Maths also uses the unified Qwen3.5-9B Q8_0 model."""
    global MATH_LLM_IS_SPECIALIZED, MATH_LLM_LOAD_DEVICE
    MATH_LLM_IS_SPECIALIZED = False
    MATH_LLM_LOAD_DEVICE = 'unified_qwen35_q8'
    return load_general_llm(unload_math_if_low_ram=False)


def get_math_llm():
    """Use this in Maths code paths; it returns the same unified Qwen3.5-9B Q8_0 model."""
    return load_math_llm(unload_general_if_low_ram=False)


try:
    qwen35_llm = _load_llama_model(MODEL_PATH, n_gpu_layers=N_GPU_LAYERS, label='unified Qwen3.5-9B Q8_0')
except ValueError as exc:
    status = _gguf_status(MODEL_PATH)
    raise RuntimeError(
        'llama-cpp-python failed to load the unified Qwen3.5-9B Q8_0 GGUF. '
        f"File status: size={status['size'] / 1024**3:.2f} GiB, magic={status['magic']}. "
        'If size is below the expected value or magic is not GGUF, delete /content/models and rerun the download cell. '
        'If the file is valid, restart the runtime and rerun the install cell so Colab loads the freshly installed llama-cpp-python wheel.'
    ) from exc

mem_report('after unified Qwen3.5-9B Q8_0 GGUF load')
!nvidia-smi


llama-cpp-python: 0.3.23
Loading unified Qwen3.5-9B Q8_0 GGUF from: /content/models/Qwen_Qwen3.5-9B-Q8_0.gguf
Loading unified Qwen3.5-9B Q8_0: /content/models/Qwen_Qwen3.5-9B-Q8_0.gguf
Available CPU RAM before unified Qwen3.5-9B Q8_0: 50.65 GiB


llama_context: n_ctx_seq (4096) < n_ctx_train (262144) -- the full capacity of the model will not be utilized



[MEM] after unified Qwen3.5-9B Q8_0 GGUF load
CPU RAM: 1.85 / 52.96 GiB (4.8%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB
Fri May 29 14:33:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   75C    P0             34W /   72W |    8936MiB /  23034MiB |      1%      Default |




## 6. GGUF LLM wrapper



In [8]:


try:
    from llama_cpp import LlamaGrammar
    FINAL_OPTION_GRAMMAR = LlamaGrammar.from_string('root ::= [0-3]')
except Exception as exc:
    FINAL_OPTION_GRAMMAR = None
    print('GBNF final-option grammar unavailable; falling back to stop-token parsing:', repr(exc))

FINAL_CHOICE_STOP = [
    '<|im_end|>',
    '<|endoftext|>',
    '\nWait',
    '\nExplanation',
    '\nReasoning',
    'Option text:',
    'Reasoning:',
    'Explanation:',
]


def _resolve_llm(llm=None):
    """Default to the general model. Maths code passes llm=get_math_llm() explicitly."""
    if llm is not None:
        return llm
    if 'get_general_llm' in globals():
        return get_general_llm()
    return qwen35_llm


def run_local_llm(prompt: str, max_new_tokens: int = 8, stop=None, temperature: float = 0.0, top_p: float = 1.0, top_k: int = 40, repeat_penalty: float = 1.05, llm=None) -> str:
    if stop is None:
        stop = ['<|im_end|>', '<|endoftext|>']
    model = _resolve_llm(llm)
    out = model(
        prompt,
        max_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repeat_penalty=repeat_penalty,
        stop=stop,
    )
    return out['choices'][0]['text'].strip()


def run_local_choice(prompt: str, valid_ids=None, llm=None):
    """Return (option_id, raw, parsed). Generic calls use the general model; Maths can pass its own model."""
    valid_ids = {0, 1, 2, 3} if valid_ids is None else {int(x) for x in valid_ids}
    model = _resolve_llm(llm)
    raw = ''
    if FINAL_OPTION_GRAMMAR is not None and valid_ids.issubset({0, 1, 2, 3}):
        try:
            out = model(
                prompt,
                max_tokens=1,
                temperature=0.0,
                top_p=1.0,
                top_k=40,
                repeat_penalty=1.0,
                stop=['<|im_end|>', '<|endoftext|>'],
                grammar=FINAL_OPTION_GRAMMAR,
            )
            raw = out['choices'][0]['text'].strip()
            if re.fullmatch(r'[0-3]', raw):
                option_id = int(raw)
                return (option_id if option_id in valid_ids else None), raw, option_id in valid_ids
        except Exception as exc:
            raw = f'[gbnf_error] {type(exc).__name__}: {exc}'

    fallback_raw = run_local_llm(
        prompt,
        max_new_tokens=4,
        stop=FINAL_CHOICE_STOP,
        temperature=0.0,
        top_p=1.0,
        top_k=40,
        repeat_penalty=1.0,
        llm=model,
    )
    option_id = option_id_from_text(fallback_raw, valid_ids) if 'option_id_from_text' in globals() else None
    combined_raw = fallback_raw if not raw else f'{raw}\n[fallback_raw] {fallback_raw}'
    return option_id, combined_raw, option_id is not None

# Smoke test uses the general model.
prompt = """You are answering a multiple-choice question.
Return ONLY the numeric option id.

Question:
Who was the first president of the United States?

Options:
0. Abraham Lincoln
1. George Washington
2. Thomas Jefferson
3. John Adams

Answer:"""
print(run_local_llm(prompt, max_new_tokens=4))
mem_report('after general LLM smoke test')



1

[MEM] after general LLM smoke test
CPU RAM: 2.14 / 52.96 GiB (5.3%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB




## 7. Load retrieval stack: embedding model, BM25, HNSW dense, reranker



In [9]:


import numpy as np
import pandas as pd
import joblib
import hnswlib
from collections import defaultdict
from sentence_transformers import SentenceTransformer, CrossEncoder

EMBEDDING_MODEL_NAME = 'sentence-transformers/multi-qa-MiniLM-L6-cos-v1'
RERANKER_MODEL_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

TOP_K_BM25 = 60
TOP_K_TEXTBOOK_BM25 = 40
TOP_K_DENSE = 40
RRF_K = 60
RRF_TOP_K = 30
RERANK_TOP_K = 12
LLM_CONTEXT_K = 4
DOC_MAX_CHARS = 500
MAX_NEW_TOKENS_FINAL = 4
MAX_NEW_TOKENS_ROUTER = 80
MATH_COMPETITION_NAME = 'Maths'
PROMPT_VERSION = 'qwen35_general_lazy_qwen25_math7b_option_substitution_semantic_router_v13c_answer_id_tool_recall'

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device='cpu')
mem_report('after embedding model')



/usr/local/lib/python3.12/dist-packages/sklearn/utils/_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.4.4)
  from scipy.sparse import csr_matrix, issparse


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


[MEM] after embedding model
CPU RAM: 2.36 / 52.96 GiB (5.7%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB


In [10]:


def normalize_text(x):
    if x is None:
        return ''
    if isinstance(x, str):
        return x
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return str(x)

def simple_tokenize(text):
    return re.findall(r"[A-Za-z0-9_]+", normalize_text(text).lower())

def extract_doc_text(doc):
    if isinstance(doc, str):
        return doc
    if isinstance(doc, dict):
        for key in ['text', 'contents', 'content', 'passage', 'document', 'body', 'chunk']:
            if key in doc and doc[key]:
                return normalize_text(doc[key])
        return normalize_text(doc)
    return normalize_text(doc)

def extract_docs_from_loaded(obj):
    if isinstance(obj, dict):
        for key in ['docs', 'documents', 'corpus', 'texts', 'chunks', 'passages']:
            if key in obj and obj[key] is not None:
                return list(obj[key])
        for key in ['metadata', 'metas', 'meta']:
            if key in obj and isinstance(obj[key], (list, tuple)):
                return list(obj[key])
    if isinstance(obj, (list, tuple)):
        return list(obj)
    return None

def make_doc_id(source, idx):
    return f'{source}:{int(idx)}'

def make_result_item(source, idx, text, score=None, rank=None, method=None):
    return {
        'doc_id': make_doc_id(source, idx),
        'source': source,
        'idx': int(idx),
        'text': extract_doc_text(text),
        'score': float(score) if score is not None else None,
        'rank': int(rank) if rank is not None else None,
        'method': method,
    }

class SparseIndexAdapter:
    def __init__(self, path, source):
        self.path = Path(path)
        self.source = source
        self.obj = joblib.load(self.path)
        self.docs = extract_docs_from_loaded(self.obj)
        self.bm25 = None
        self.vectorizer = None
        self.matrix = None
        if isinstance(self.obj, dict):
            self.bm25 = self.obj.get('bm25') or self.obj.get('index') or self.obj.get('bm25_index')
            self.vectorizer = self.obj.get('vectorizer')
            self.matrix = self.obj.get('matrix') or self.obj.get('X') or self.obj.get('tfidf_matrix')
        else:
            self.bm25 = self.obj
        if self.docs is None:
            raise ValueError(f'Could not extract docs from {path}')
        print(f'[SparseIndexAdapter] {source}: docs={len(self.docs)} bm25={self.bm25 is not None} vectorizer={self.vectorizer is not None}')

    def search(self, query, top_k=50):
        tokens = simple_tokenize(query)
        # bm25s style or custom bm25 object
        if self.bm25 is not None:
            # Try bm25s retrieve API variants.
            for call in [
                lambda: self.bm25.retrieve([tokens], k=top_k),
                lambda: self.bm25.retrieve(tokens, k=top_k),
                lambda: self.bm25.get_top_n(tokens, self.docs, n=top_k),
            ]:
                try:
                    res = call()
                    # bm25s often returns (results, scores) arrays.
                    if isinstance(res, tuple) and len(res) == 2:
                        indices, scores = res
                        indices = np.array(indices).reshape(-1)[:top_k]
                        scores = np.array(scores).reshape(-1)[:top_k]
                        return [make_result_item(self.source, int(i), self.docs[int(i)], score=s, rank=r, method='bm25')
                                for r, (i, s) in enumerate(zip(indices, scores), start=1)]
                    # If returns docs directly, map by identity is impossible; return text-only pseudo indices.
                    if isinstance(res, list) and res and not isinstance(res[0], (int, np.integer)):
                        return [make_result_item(self.source, i, d, score=None, rank=i+1, method='bm25')
                                for i, d in enumerate(res[:top_k])]
                except Exception:
                    pass
            # rank_bm25/get_scores style
            try:
                scores = np.asarray(self.bm25.get_scores(tokens))
                idx = np.argsort(-scores)[:top_k]
                return [make_result_item(self.source, int(i), self.docs[int(i)], score=scores[int(i)], rank=r, method='bm25')
                        for r, i in enumerate(idx, start=1)]
            except Exception as e:
                raise RuntimeError(f'BM25 search failed for {self.source}: {e}')
        # sklearn TF-IDF fallback
        if self.vectorizer is not None and self.matrix is not None:
            qv = self.vectorizer.transform([query])
            scores = (self.matrix @ qv.T).toarray().reshape(-1)
            idx = np.argsort(-scores)[:top_k]
            return [make_result_item(self.source, int(i), self.docs[int(i)], score=scores[int(i)], rank=r, method='tfidf')
                    for r, i in enumerate(idx, start=1)]
        raise RuntimeError(f'No searchable sparse index found for {self.source}')

class DenseIndexAdapter:
    def __init__(self, index_path, meta_path, source, shared_docs=None, dim=384, space='cosine'):
        self.source = source
        self.index_path = Path(index_path)
        self.meta_path = Path(meta_path)
        meta = joblib.load(self.meta_path)
        meta_docs = extract_docs_from_loaded(meta)
        if shared_docs is not None and meta_docs is not None and len(shared_docs) == len(meta_docs):
            self.docs = shared_docs
            del meta_docs, meta
            gc.collect()
            print(f'[DenseIndexAdapter] {source}: reusing BM25 docs; dense meta docs released')
        else:
            self.docs = meta_docs
        if self.docs is None:
            raise ValueError(f'Could not extract dense docs from {meta_path}')
        self.index = hnswlib.Index(space=space, dim=dim)
        self.index.load_index(str(self.index_path))
        self.index.set_ef(128)
        print(f'[DenseIndexAdapter] {source}: docs={len(self.docs)} dim={dim} space={space} ef=128')

    def search(self, query, top_k=40):
        vec = embedding_model.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
        labels, distances = self.index.knn_query(vec, k=top_k)
        labels = labels.reshape(-1)
        distances = distances.reshape(-1)
        # cosine distance: lower is better. Convert to similarity-ish score.
        scores = 1.0 - distances
        return [make_result_item(self.source, int(i), self.docs[int(i)], score=s, rank=r, method='dense')
                for r, (i, s) in enumerate(zip(labels, scores), start=1)]



In [11]:


simplewiki_sparse = SparseIndexAdapter(LOCAL_INDEX_FILES['simplewiki_bm25'], source='simplewiki')
mem_report('after SimpleWiki BM25')
kelm_sparse = SparseIndexAdapter(LOCAL_INDEX_FILES['kelm_bm25'], source='kelm')
mem_report('after KELM BM25')

TEXTBOOK_INDEX_SOURCES = {
    'textbook_introductory_statistics': 'textbook_introductory_statistics',
    'textbook_algebra_trigonometry': 'textbook_algebra_trigonometry',
    'textbook_calculus_volume_1': 'textbook_calculus_volume_1',
    'textbook_discrete_math': 'textbook_discrete_math',
    'textbook_abstract_algebra': 'textbook_abstract_algebra',
    'textbook_basic_analysis': 'textbook_basic_analysis',
    'textbook_topology': 'textbook_topology',
}
textbook_sparse_indexes = {
    source: SparseIndexAdapter(LOCAL_INDEX_FILES[key], source=source)
    for key, source in TEXTBOOK_INDEX_SOURCES.items()
}
mem_report('after textbook BM25 indexes')

simplewiki_dense = DenseIndexAdapter(
    LOCAL_INDEX_FILES['simplewiki_dense_index'],
    LOCAL_INDEX_FILES['simplewiki_dense_meta'],
    source='simplewiki',
    shared_docs=simplewiki_sparse.docs,
)
mem_report('after SimpleWiki dense')

kelm_dense = DenseIndexAdapter(
    LOCAL_INDEX_FILES['kelm_dense_index'],
    LOCAL_INDEX_FILES['kelm_dense_meta'],
    source='kelm',
    shared_docs=kelm_sparse.docs,
)
mem_report('after KELM dense')

TEXTBOOK_DENSE_INDEX_FILES = {
    'textbook_introductory_statistics': ('textbook_introductory_statistics_dense_index', 'textbook_introductory_statistics_dense_meta'),
    'textbook_algebra_trigonometry': ('textbook_algebra_trigonometry_dense_index', 'textbook_algebra_trigonometry_dense_meta'),
    'textbook_calculus_volume_1': ('textbook_calculus_volume_1_dense_index', 'textbook_calculus_volume_1_dense_meta'),
    'textbook_discrete_math': ('textbook_discrete_math_dense_index', 'textbook_discrete_math_dense_meta'),
    'textbook_abstract_algebra': ('textbook_abstract_algebra_dense_index', 'textbook_abstract_algebra_dense_meta'),
    'textbook_basic_analysis': ('textbook_basic_analysis_dense_index', 'textbook_basic_analysis_dense_meta'),
    'textbook_topology': ('textbook_topology_dense_index', 'textbook_topology_dense_meta'),
}
textbook_dense_indexes = {
    source: DenseIndexAdapter(
        LOCAL_INDEX_FILES[index_key],
        LOCAL_INDEX_FILES[meta_key],
        source=source,
        shared_docs=textbook_sparse_indexes[source].docs,
    )
    for source, (index_key, meta_key) in TEXTBOOK_DENSE_INDEX_FILES.items()
}
mem_report('after textbook dense indexes')

reranker = CrossEncoder(RERANKER_MODEL_NAME, device='cpu')
mem_report('after CPU reranker')
print('Embedding device:', getattr(embedding_model, 'device', 'unknown'))
print('Reranker device:', reranker.model.device)



[SparseIndexAdapter] simplewiki: docs=434093 bm25=True vectorizer=False

[MEM] after SimpleWiki BM25
CPU RAM: 3.63 / 52.96 GiB (8.1%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB
[SparseIndexAdapter] kelm: docs=500000 bm25=True vectorizer=False

[MEM] after KELM BM25
CPU RAM: 4.00 / 52.96 GiB (8.8%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB
[SparseIndexAdapter] textbook_introductory_statistics: docs=2168 bm25=True vectorizer=False
[SparseIndexAdapter] textbook_algebra_trigonometry: docs=2845 bm25=True vectorizer=False
[SparseIndexAdapter] textbook_calculus_volume_1: docs=1310 bm25=True vectorizer=False
[SparseIndexAdapter] textbook_discrete_math: docs=1163 bm25=True vectorizer=False
[SparseIndexAdapter] textbook_abstract_algebra: docs=1054 bm25=True vectorizer=False
[SparseIndexAdapter] textbook_basic_analysis: docs=578 bm25=True vectorizer=False
[SparseIndexAdapter] textbook_topology: docs=619 bm25=True vectorizer=False

[MEM] after textbook BM25 index

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


[MEM] after CPU reranker
CPU RAM: 6.02 / 52.96 GiB (12.6%)
GPU torch allocated: 0.00 GiB
GPU torch reserved:  0.00 GiB
Embedding device: cpu
Reranker device: cpu




## 8. Hybrid retrieval, RRF, reranker



In [12]:


def hybrid_retrieve(query, top_k_bm25=TOP_K_BM25, top_k_dense=TOP_K_DENSE, include_textbooks=False):
    result_lists = []
    result_lists.append(simplewiki_sparse.search(query, top_k=top_k_bm25))
    result_lists.append(kelm_sparse.search(query, top_k=top_k_bm25))
    result_lists.append(simplewiki_dense.search(query, top_k=top_k_dense))
    result_lists.append(kelm_dense.search(query, top_k=top_k_dense))
    if include_textbooks:
        for index in textbook_sparse_indexes.values():
            result_lists.append(index.search(query, top_k=TOP_K_TEXTBOOK_BM25))
        for index in textbook_dense_indexes.values():
            result_lists.append(index.search(query, top_k=TOP_K_DENSE))
    return result_lists

def rrf_fusion(result_lists, k=RRF_K, top_k=RRF_TOP_K):
    scores = defaultdict(float)
    docs = {}
    sources = defaultdict(list)
    for results in result_lists:
        for rank, item in enumerate(results, start=1):
            doc_id = item['doc_id']
            scores[doc_id] += 1.0 / (k + rank)
            if doc_id not in docs:
                docs[doc_id] = dict(item)
            sources[doc_id].append(item.get('method'))
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    fused = []
    for doc_id, score in ranked:
        item = dict(docs[doc_id])
        item['rrf_score'] = float(score)
        item['matched_methods'] = sorted(set(m for m in sources[doc_id] if m))
        fused.append(item)
    return fused

def rerank(query, docs, top_k=LLM_CONTEXT_K):
    if not docs:
        return []
    docs = docs[:RERANK_TOP_K]
    pairs = [(query, d['text'][:1200]) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: float(x[1]), reverse=True)
    out = []
    for doc, score in ranked[:top_k]:
        item = dict(doc)
        item['reranker_score'] = float(score)
        out.append(item)
    return out

def retrieve_and_rerank(query, include_textbooks=False):
    result_lists = hybrid_retrieve(query, include_textbooks=include_textbooks)
    fused = rrf_fusion(result_lists)
    return rerank(query, fused)



In [13]:


# Smoke test retrieval
query = 'Who was the first president of the United States?'
docs = retrieve_and_rerank(query)
for i, d in enumerate(docs[:5], start=1):
    print('='*80)
    print(i, d.get('source'), d.get('method'), d.get('matched_methods'), d.get('reranker_score'))
    print(d['text'][:500])



BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/9 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/9 [00:00<?, ?it/s]

1 simplewiki bm25 ['bm25', 'dense'] 10.440590858459473
The first inauguration of George Washington as the president of the United States took place on April 30, 1789. The inauguration was the beginning of the first term of George Washington as president. John Adams had already taken office as vice president on April 21. Washington was sworn in by Chancellor of New York Robert Livingston. Washington became the first president of the United States following the ratification of the Constitution.
2 simplewiki dense ['dense'] 9.524194717407227
wrote the Constitution of the United States, and all of the states eventually agreed to it and joined the new government. of President George Washington]] Presidency On January 7, 1789, aged 56, Washington was elected as the first president of the United States. He did not want the job but thought that the country might fall apart unless he took it. John Adams (1735–1826), who received the second-largest number of votes, became the first vice presiden



## 9. Prompting, answer parsing, and option-wise retrieval



In [14]:


def get_question_text(question):
    """Return the canonical text field used by all strategies."""
    return getattr(question, 'text', None) or getattr(question, 'question_text', None) or str(question)


def get_options(question):
    """Return the API option objects."""
    return getattr(question, 'options')


def _clean_answer_text(text):
    text = normalize_text(text).strip()
    text = re.sub(r'<think>.*?(?:</think>|$)', ' ', text, flags=re.I | re.S).strip()
    text = re.sub(r'^(?:answer|option|choice)\s*[:#\-]?\s*', '', text, flags=re.I).strip()
    return text.strip(' .,:;\n\t')


def _normalize_for_text_match(text):
    text = normalize_text(text).lower().strip()
    text = text.replace('−', '-').replace('–', '-').replace('—', '-')
    text = text.replace('π', 'pi')
    text = text.replace('$', '')
    text = re.sub(r'\\left|\\right|\\,', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip(' .,:;')


def _math_text_for_parse(text):
    text = normalize_text(text)
    text = text.replace('−', '-').replace('–', '-').replace('—', '-')
    text = text.replace('π', 'pi')
    text = text.replace('^', '**')
    text = text.replace('$', '')
    text = text.replace('\\times', '*').replace('\\cdot', '*').replace('×', '*')
    text = text.replace('\\div', '/').replace('÷', '/')
    text = re.sub(r'\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}', r'((\1)/(\2))', text)
    text = re.sub(r'frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}', r'((\1)/(\2))', text)
    text = re.sub(r'\\sqrt\s*\{([^{}]+)\}', r'sqrt(\1)', text)
    text = re.sub(r'\\sqrt\s*\(([^()]+)\)', r'sqrt(\1)', text)
    text = re.sub(r'\\overline\s*\{([^{}]+)\}', r'\1', text)
    return text.strip()


def _try_parse_math_value(text):
    try:
        import sympy as sp
        from sympy.parsing.sympy_parser import (
            convert_xor,
            implicit_multiplication_application,
            parse_expr,
            standard_transformations,
        )
        cleaned = _math_text_for_parse(text).replace(',', '')
        if not cleaned or re.search(r'[^0-9a-zA-Z_+\-*/().\s=]', cleaned):
            return None
        if '=' in cleaned:
            return None
        transformations = standard_transformations + (implicit_multiplication_application, convert_xor)
        local_dict = {'sqrt': sp.sqrt, 'pi': sp.pi, 'e': sp.E, 'E': sp.E, 'i': sp.I, 'I': sp.I}
        return sp.simplify(parse_expr(cleaned, local_dict=local_dict, transformations=transformations, evaluate=True))
    except Exception:
        return None


def _numeric_values_close(a, b, tolerance=1e-6):
    try:
        import sympy as sp
        return abs(float(sp.N(sp.sympify(a) - sp.sympify(b)))) <= tolerance
    except Exception:
        return False


def _extract_number_sequence(text):
    return [float(x) for x in re.findall(r'[-+]?\d+(?:\.\d+)?', normalize_text(text).replace(',', ''))]


def extract_display_math_expression(question):
    """Extract the longest LaTeX/math span from a question when present."""
    text = get_question_text(question)
    matches = re.findall(r'\$\$(.*?)\$\$|\$(.*?)\$', text, flags=re.S)
    chunks = [a or b for a, b in matches if (a or b)]
    if chunks:
        return max(chunks, key=len)
    match = re.search(r'(?:expression|evaluate|simplify)\s*:?\s*(.+?)(?:\?|\.|$)', text, flags=re.I | re.S)
    if match:
        return match.group(1)
    return None


def option_id_from_value(value, question, tolerance=1e-6):
    for opt in get_options(question):
        parsed = _try_parse_math_value(opt.text)
        if parsed is not None and _numeric_values_close(parsed, value, tolerance=tolerance):
            return int(opt.id)
        if '%' in normalize_text(opt.text):
            pct = _try_parse_math_value(normalize_text(opt.text).replace('%', ''))
            if pct is not None and (_numeric_values_close(pct, value, tolerance=tolerance) or _numeric_values_close(pct / 100, value, tolerance=tolerance)):
                return int(opt.id)
    return None


def option_id_from_number_sequence(values, question, tolerance=1e-3):
    values = [float(v) for v in values]
    for opt in get_options(question):
        nums = _extract_number_sequence(opt.text)
        if len(nums) != len(values):
            continue
        if all(abs(a - b) <= tolerance for a, b in zip(nums, values)):
            return int(opt.id)
    return None


def option_id_from_text(text, valid_ids, question=None):
    """Parse an LLM answer. This accepts an explicit id, letter, exact option text, or option value."""
    original = normalize_text(text).strip()
    m = re.search(r'(?is)\b(?:final\s+)?answer\s*[:#\-]?\s*([0-3])\b', original)
    if m:
        val = int(m.group(1))
        if val in valid_ids:
            return val
    raw = _clean_answer_text(original)
    if not raw:
        return None
    m = re.match(r'^\s*([0-3])(?:\s|$)', raw)
    if m:
        val = int(m.group(1))
        if val in valid_ids:
            return val
    if re.fullmatch(r'[-+]?\d+', raw):
        val = int(raw)
        if val in valid_ids:
            return val
    letter_map_zero = {'A': 0, 'B': 1, 'C': 2, 'D': 3}
    letter_map_one = {'A': 1, 'B': 2, 'C': 3, 'D': 4}
    m = re.fullmatch(r'([ABCD])', raw.upper())
    if m:
        letter = m.group(1)
        for val in [letter_map_zero[letter], letter_map_one[letter]]:
            if val in valid_ids:
                return val
    if question is not None:
        raw_norm = _normalize_for_text_match(raw)
        for opt in get_options(question):
            if raw_norm == _normalize_for_text_match(opt.text):
                return int(opt.id)
        raw_value = _try_parse_math_value(raw)
        if raw_value is not None:
            option_id = option_id_from_value(raw_value, question)
            if option_id is not None:
                return option_id
        raw_numbers = _extract_number_sequence(raw)
        if raw_numbers:
            option_id = option_id_from_number_sequence(raw_numbers, question)
            if option_id is not None:
                return option_id
    m = re.search(r'\b([0-3])\b', raw)
    if m:
        val = int(m.group(1))
        if val in valid_ids:
            return val
    return None


def retrieval_score_summary(docs):
    scores = []
    for doc in docs or []:
        try:
            scores.append(float(doc.get('reranker_score')))
        except Exception:
            pass
    scores = sorted(scores, reverse=True)
    top = scores[0] if scores else None
    second = scores[1] if len(scores) > 1 else None
    margin = (top - second) if top is not None and second is not None else None
    return {
        'retrieval_top_score': top,
        'retrieval_second_score': second,
        'retrieval_margin': margin,
    }


def _confidence_from_retrieval(summary, parsed=True):
    if not parsed:
        return 0.2
    top = summary.get('retrieval_top_score')
    margin = summary.get('retrieval_margin')
    confidence = 0.55
    if top is not None:
        confidence += max(-0.15, min(0.25, top / 20.0))
    if margin is not None:
        confidence += max(-0.05, min(0.15, margin / 12.0))
    return round(max(0.25, min(0.9, confidence)), 3)


def _compact_doc(doc, max_chars=420):
    return {
        'source': doc.get('source'),
        'idx': doc.get('idx'),
        'reranker_score': doc.get('reranker_score'),
        'text': normalize_text(doc.get('text', ''))[:max_chars],
    }


def build_rag_prompt(question, docs, competition_name):
    qtext = get_question_text(question)
    options = '\n'.join(f'{int(opt.id)}. {opt.text}' for opt in get_options(question))
    context = '\n\n'.join(
        f'[DOC {i} | {doc.get("source", "unknown")} | score={doc.get("reranker_score", 0):.3f}]\n{doc["text"][:DOC_MAX_CHARS]}'
        for i, doc in enumerate(docs[:LLM_CONTEXT_K], start=1)
    )
    return f"""/no_think
You are answering a multiple-choice quiz question.

Use ONLY the context below. If the context is weak, choose the option that is best supported by general factual knowledge, but do not invent details.
Do not choose an answer only because it shares words with the context.
Return ONLY the numeric option id.

Competition: {competition_name}
Question:
{qtext}

Options:
{options}

Context:
{context}

/no_think
Answer:"""


def llm_choose_option(question, docs, competition_name):
    valid_ids = {int(opt.id) for opt in get_options(question)}
    prompt = build_rag_prompt(question, docs, competition_name)
    option_id, raw, parsed = run_local_choice(prompt, valid_ids)
    if option_id is None:
        option_id = int(get_options(question)[0].id)
    summary = retrieval_score_summary(docs)
    return option_id, {
        'strategy': 'hybrid_rag_rrf_rerank_qwen35_gguf_gbnf',
        'decision_source': 'rag_global',
        'confidence': _confidence_from_retrieval(summary, parsed=parsed),
        'raw_llm_output': raw,
        'retrieved_context': docs,
        'fallback_used': None if parsed else 'first_option_invalid_llm_output',
        **summary,
    }


OPTION_RETRIEVAL_ALWAYS_COMPETITIONS = {'Entertainment', 'Ancient History and Politics'}
OPTION_RETRIEVAL_LOW_SCORE_THRESHOLD = 0.25
OPTION_RETRIEVAL_LOW_MARGIN_THRESHOLD = 0.35
OPTION_RETRIEVAL_SCIENCE_LOW_MARGIN_THRESHOLD = 0.25
OPTION_EVIDENCE_TOP_DOCS = 2


def _competition_family(competition_name):
    name = normalize_text(competition_name).lower()
    if 'math' in name:
        return 'maths'
    if 'entertainment' in name:
        return 'entertainment'
    if 'history' in name or 'politics' in name:
        return 'history'
    if 'science' in name or 'nature' in name:
        return 'science'
    return 'other'


def should_use_option_retrieval(competition_name, docs):
    """Adaptive option-wise retrieval: always for Entertainment/History, cautious for Science."""
    family = _competition_family(competition_name)
    if family == 'maths':
        return False
    if competition_name in OPTION_RETRIEVAL_ALWAYS_COMPETITIONS or family in {'entertainment', 'history'}:
        return True

    summary = retrieval_score_summary(docs)
    top = summary.get('retrieval_top_score')
    margin = summary.get('retrieval_margin')
    if top is None:
        return True
    if top < OPTION_RETRIEVAL_LOW_SCORE_THRESHOLD:
        return True
    if family == 'science':
        return margin is not None and margin < OPTION_RETRIEVAL_SCIENCE_LOW_MARGIN_THRESHOLD
    return margin is not None and margin < OPTION_RETRIEVAL_LOW_MARGIN_THRESHOLD


def retrieve_option_evidence(question, top_docs_per_option=OPTION_EVIDENCE_TOP_DOCS):
    """Retrieve evidence separately for each option using query = question + option text."""
    qtext = get_question_text(question)
    evidence = []
    for opt in get_options(question):
        query = f'{qtext} {opt.text}'
        docs = retrieve_and_rerank(query)
        summary = retrieval_score_summary(docs)
        evidence.append({
            'option_id': int(opt.id),
            'option_text': opt.text,
            'query': query,
            'top_score': summary.get('retrieval_top_score'),
            'second_score': summary.get('retrieval_second_score'),
            'margin': summary.get('retrieval_margin'),
            'docs': [_compact_doc(d) for d in docs[:top_docs_per_option]],
        })
    scores = [row['top_score'] for row in evidence if row.get('top_score') is not None]
    sorted_scores = sorted(scores, reverse=True)
    return evidence, {
        'option_retrieval_top_score': sorted_scores[0] if sorted_scores else None,
        'option_retrieval_second_score': sorted_scores[1] if len(sorted_scores) > 1 else None,
        'option_retrieval_margin': (sorted_scores[0] - sorted_scores[1]) if len(sorted_scores) > 1 else None,
    }


def build_option_rag_prompt(question, global_docs, option_evidence, competition_name):
    qtext = get_question_text(question)
    options = '\n'.join(f'{int(opt.id)}. {opt.text}' for opt in get_options(question))
    global_context = '\n\n'.join(
        f'[GLOBAL DOC {i} | {doc.get("source", "unknown")} | score={doc.get("reranker_score", 0):.3f}]\n{doc["text"][:DOC_MAX_CHARS]}'
        for i, doc in enumerate((global_docs or [])[:LLM_CONTEXT_K], start=1)
    )
    option_blocks = []
    for row in option_evidence:
        docs_text = '\n'.join(
            f'- [{doc.get("source")} | score={doc.get("reranker_score")}] {doc.get("text", "")[:DOC_MAX_CHARS]}'
            for doc in row.get('docs', [])
        )
        option_blocks.append(
            f'Option {row["option_id"]}. {row["option_text"]}\nOption evidence top score: {row.get("top_score")}\n{docs_text}'
        )
    option_context = '\n\n'.join(option_blocks)
    return f"""/no_think
You are answering a multiple-choice factual quiz question.

Use the global evidence and the option-specific evidence. Prefer the option with direct support, not the option that merely repeats words from the question.
Return ONLY the numeric option id.

Competition: {competition_name}
Question:
{qtext}

Options:
{options}

Global evidence:
{global_context}

Option-specific evidence:
{option_context}

/no_think
Answer:"""


def _confidence_from_option_retrieval(global_summary, option_summary, parsed=True):
    confidence = _confidence_from_retrieval(global_summary, parsed=parsed)
    margin = option_summary.get('option_retrieval_margin')
    if not parsed:
        return 0.2
    if margin is None:
        return min(confidence, 0.65)
    if margin < 0.25:
        return min(confidence, 0.55)
    if margin < 0.75:
        return min(confidence, 0.70)
    if margin < 1.5:
        return min(confidence, 0.82)
    return min(0.90, confidence + max(0.0, min(0.05, margin / 60.0)))


def llm_choose_option_with_option_evidence(question, global_docs, option_evidence, option_summary, competition_name):
    valid_ids = {int(opt.id) for opt in get_options(question)}
    prompt = build_option_rag_prompt(question, global_docs, option_evidence, competition_name)
    option_id, raw, parsed = run_local_choice(prompt, valid_ids)
    if option_id is None:
        option_id = int(get_options(question)[0].id)
    global_summary = retrieval_score_summary(global_docs)
    confidence = _confidence_from_option_retrieval(global_summary, option_summary, parsed=parsed)
    return option_id, {
        'strategy': 'hybrid_rag_option_evidence_qwen35_gguf_gbnf_adaptive',
        'decision_source': 'rag_option_evidence',
        'confidence': round(confidence, 3),
        'raw_llm_output': raw,
        'retrieved_context': global_docs,
        'option_evidence': option_evidence,
        'option_evidence_json': json.dumps(option_evidence, ensure_ascii=False),
        'option_evidence_scores_json': json.dumps([
            {'option_id': row['option_id'], 'top_score': row.get('top_score'), 'margin': row.get('margin')}
            for row in option_evidence
        ], ensure_ascii=False),
        'fallback_used': None if parsed else 'first_option_invalid_llm_output',
        **global_summary,
        **option_summary,
    }





## 10. Validated generic Maths tool layer

This section implements a self-contained tool-calling layer without LangChain. The model may propose a JSON tool call, but Python validates the schema, checks semantic guards, executes the tool, and matches the result to an answer option deterministically.



In [15]:


import sympy as sp
import math
import re
import json
import time
from dataclasses import dataclass, field
from statistics import NormalDist
from typing import Any, Callable, Optional
from sympy.parsing.sympy_parser import (
    convert_xor,
    implicit_multiplication_application,
    parse_expr,
    standard_transformations,
)

# This layer intentionally mirrors modern tool-calling practice without adding a framework:
# a model may propose a JSON call, but Python validates, guards, executes, and matches it.

@dataclass
class ToolDecision:
    option_id: int
    strategy: str
    confidence: float
    explanation: str
    raw_tool_call: Optional[str] = None
    validated_tool_call: Optional[dict] = None


@dataclass
class ToolExecution:
    tool: str
    value: Any
    explanation: str
    confidence: float = 0.9
    candidate_values: list[Any] = field(default_factory=list)
    candidate_sequences: list[list[float]] = field(default_factory=list)
    option_id: Optional[int] = None
    answer_text: Optional[str] = None


@dataclass
class ToolSpec:
    name: str
    description: str
    required: dict[str, str]
    optional: dict[str, str]
    execute: Callable[[Any, dict], ToolExecution]
    guard: Optional[Callable[[Any, dict], tuple[bool, str]]] = None


MATH_COMPETITION_NAME = 'Maths'
PROMPT_VERSION = 'qwen35_general_lazy_qwen25_math7b_option_substitution_semantic_router_v13c_answer_id_tool_recall'
MATH_STRUCTURED_MAX_NEW_TOKENS = 96
MATH_DIRECT_MAX_NEW_TOKENS = 32
MATH_MICRO_COT_MAX_TOKENS = 48
LAST_MATH_TOOL_TRACE = []
TOOL_SPECS = {}


class ToolValidationError(ValueError):
    pass


def register_tool(spec: ToolSpec):
    TOOL_SPECS[spec.name] = spec
    return spec


def _make_tool_decision(option_id, strategy, confidence, explanation, raw_tool_call=None, validated_tool_call=None):
    return ToolDecision(int(option_id), str(strategy), float(confidence), str(explanation), raw_tool_call, validated_tool_call)



def _append_tool_trace(tool, matched, start, error=None, call=None, raw=None, validation=None, explanation=None):
    item = {
        'tool': tool,
        'matched': bool(matched),
        'latency': time.time() - start,
        'error': error,
    }
    if call is not None:
        item['call'] = call
    if raw is not None:
        item['raw'] = str(raw)[:800]
    if validation is not None:
        item['validation'] = validation
    if explanation is not None:
        item['explanation'] = str(explanation)[:800]
    LAST_MATH_TOOL_TRACE.append(item)

def _safe_float(x):
    try:
        return float(str(x).replace(',', '').strip())
    except Exception:
        return None


def _safe_int(x):
    try:
        return int(float(str(x).replace(',', '').strip()))
    except Exception:
        return None


def _sympy_local_dict():
    return {
        'sqrt': sp.sqrt, 'log': sp.log, 'ln': sp.log,
        'sin': sp.sin, 'cos': sp.cos, 'tan': sp.tan, 'exp': sp.exp,
        'pi': sp.pi, 'e': sp.E, 'E': sp.E, 'i': sp.I, 'I': sp.I,
        'gcd': sp.gcd, 'lcm': sp.lcm, 'divisor_count': sp.divisor_count,
        'factorial': sp.factorial, 'binomial': sp.binomial,
        'Abs': sp.Abs, 'abs': sp.Abs,
        'floor': sp.floor, 'ceil': sp.ceiling, 'ceiling': sp.ceiling,
    }


def parse_math_expression(text):
    cleaned = _math_text_for_parse(str(text))
    cleaned = cleaned.replace('y =', '').replace('y=', '').replace('f(x) =', '').replace('f(x)=', '')
    cleaned = re.sub(r'(?<=\d),(?=\d{3}\b)', '', cleaned)
    cleaned = cleaned.replace('ln', 'log')
    cleaned = re.sub(r'\be\b', 'E', cleaned)
    if not cleaned or re.search(r'[^0-9a-zA-Z_,+\-*/().\s=]', cleaned):
        return None
    if '=' in cleaned:
        return None
    transformations = standard_transformations + (implicit_multiplication_application, convert_xor)
    try:
        return sp.simplify(parse_expr(cleaned, local_dict=_sympy_local_dict(), transformations=transformations, evaluate=True))
    except Exception:
        return None


def parse_equation(equation, variable='x'):
    equation = _math_text_for_parse(str(equation))
    equation = re.sub(r'(?<=\d),(?=\d{3}\b)', '', equation)
    variable_symbol = sp.Symbol(str(variable))
    if '=' in equation:
        lhs, rhs = equation.split('=', 1)
        lhs_expr = parse_math_expression(lhs)
        rhs_expr = parse_math_expression(rhs)
        if lhs_expr is None or rhs_expr is None:
            return None, variable_symbol
        return sp.Eq(lhs_expr, rhs_expr), variable_symbol
    expr = parse_math_expression(equation)
    if expr is None:
        return None, variable_symbol
    return sp.Eq(expr, 0), variable_symbol


def option_id_by_text(question, include, exclude=()):
    include = [str(x).lower() for x in include]
    exclude = [str(x).lower() for x in exclude]
    for opt in get_options(question):
        low = _normalize_for_text_match(opt.text)
        if all(token in low for token in include) and not any(token in low for token in exclude):
            return int(opt.id)
    return None


def option_id_by_any_value(values, question, tolerance=1e-6):
    for value in values:
        option_id = option_id_from_value(value, question, tolerance=tolerance)
        if option_id is not None:
            return option_id
        for opt in get_options(question):
            opt_norm = _normalize_for_text_match(opt.text)
            if value == sp.I and opt_norm == 'i':
                return int(opt.id)
            parsed = parse_math_expression(opt.text)
            if parsed is not None:
                try:
                    if sp.simplify(parsed - value) == 0:
                        return int(opt.id)
                except Exception:
                    pass
                if _numeric_values_close(parsed, value, tolerance=tolerance):
                    return int(opt.id)
    return None


def option_id_by_any_sequence(sequences, question, tolerance=1e-3):
    for seq in sequences:
        option_id = option_id_from_number_sequence(seq, question, tolerance=tolerance)
        if option_id is not None:
            return option_id
    return None


def decision_from_execution(question, execution):
    option_id = execution.option_id
    if option_id is None and execution.candidate_values:
        option_id = option_id_by_any_value(execution.candidate_values, question, tolerance=0.02)
    if option_id is None and execution.candidate_sequences:
        option_id = option_id_by_any_sequence(execution.candidate_sequences, question, tolerance=1500 if 'normal' in execution.tool else 0.03)
    if option_id is None and execution.answer_text:
        option_id = option_id_by_text(question, [execution.answer_text.lower()])
    if option_id is None:
        return None, 'tool result did not match any option deterministically'
    return _make_tool_decision(option_id, f'tool_{execution.tool}', execution.confidence, execution.explanation), None


def _coerce_arg(value, kind):
    if kind == 'int':
        out = _safe_int(value)
        if out is None:
            raise ToolValidationError(f'expected int, got {value!r}')
        return out
    if kind == 'float':
        out = _safe_float(value)
        if out is None:
            raise ToolValidationError(f'expected float, got {value!r}')
        return out
    if kind == 'number':
        out = _safe_float(value)
        if out is None:
            parsed = parse_math_expression(value)
            if parsed is None:
                raise ToolValidationError(f'expected number, got {value!r}')
            return parsed
        return out
    if kind == 'str':
        if value is None:
            raise ToolValidationError('expected string, got None')
        return str(value)
    if kind == 'list':
        if not isinstance(value, list):
            raise ToolValidationError(f'expected list, got {type(value).__name__}')
        return value
    if kind == 'bool':
        if isinstance(value, bool):
            return value
        if str(value).lower() in {'true', '1', 'yes'}:
            return True
        if str(value).lower() in {'false', '0', 'no'}:
            return False
        raise ToolValidationError(f'expected bool, got {value!r}')
    return value



def validate_tool_call(question, call):
    if not isinstance(call, dict):
        raise ToolValidationError('tool call is not a dict')
    tool = str(call.get('tool') or call.get('tool_name') or '').strip()
    if tool in {'', 'no_tool', 'none', 'null'}:
        raise ToolValidationError('no_tool_selected')
    spec = TOOL_SPECS.get(tool)
    if spec is None:
        raise ToolValidationError(f'unknown tool: {tool}')
    raw_args = call.get('args')
    if raw_args is None:
        raw_args = call.get('arguments', {})
    if not isinstance(raw_args, dict):
        raise ToolValidationError('args must be a dict')
    args = {}
    for name, kind in spec.required.items():
        if name not in raw_args:
            raise ToolValidationError(f'missing required argument: {name}')
        args[name] = _coerce_arg(raw_args[name], kind)
    for name, kind in spec.optional.items():
        if name in raw_args and raw_args[name] is not None:
            args[name] = _coerce_arg(raw_args[name], kind)
    if spec.guard is not None:
        ok, reason = spec.guard(question, args)
        if not ok:
            raise ToolValidationError(f'semantic guard rejected call: {reason}')
    return spec, args

def extract_json_objects(text):
    raw = normalize_text(text)
    objects = []
    i = 0
    while i < len(raw):
        start = raw.find('{', i)
        if start < 0:
            break
        depth = 0
        in_string = False
        escape = False
        end = None
        for index in range(start, len(raw)):
            char = raw[index]
            if in_string:
                if escape:
                    escape = False
                elif char == '\\':
                    escape = True
                elif char == '"':
                    in_string = False
                continue
            if char == '"':
                in_string = True
            elif char == '{':
                depth += 1
            elif char == '}':
                depth -= 1
                if depth == 0:
                    end = index + 1
                    break
        if end is None:
            break
        objects.append(raw[start:end])
        i = end
    return objects



def parse_validated_tool_call(question, raw):
    candidates = []
    full = normalize_text(raw).strip()
    try:
        obj = json.loads(full)
        if isinstance(obj, dict) and ('tool' in obj or 'tool_name' in obj):
            candidates.append(obj)
    except Exception:
        pass
    for chunk in extract_json_objects(full):
        try:
            obj = json.loads(chunk)
            if isinstance(obj, dict) and ('tool' in obj or 'tool_name' in obj):
                candidates.append(obj)
        except Exception:
            pass
    valid = []
    errors = []
    seen = set()
    for call in candidates:
        key = json.dumps(call, sort_keys=True, ensure_ascii=False)
        if key in seen:
            continue
        seen.add(key)
        try:
            spec, args = validate_tool_call(question, call)
            valid.append((call, spec, args))
        except ToolValidationError as exc:
            errors.append(str(exc))
    if not valid:
        reason = '; '.join(errors) if errors else 'no valid JSON tool call found'
        if 'no_tool_selected' in reason:
            reason = 'no_tool_selected'
        return None, None, None, reason
    normalized = {json.dumps({'tool': spec.name, 'args': args}, sort_keys=True, ensure_ascii=False) for _, spec, args in valid}
    if len(normalized) > 1:
        return None, None, None, 'multiple different valid tool calls found'
    return valid[0][0], valid[0][1], valid[0][2], None

def execute_validated_tool_call(question, call, raw=None):
    try:
        spec, args = validate_tool_call(question, call)
        execution = spec.execute(question, args)
        decision, error = decision_from_execution(question, execution)
        if decision is not None:
            decision.raw_tool_call = raw if raw is not None else json.dumps(call, ensure_ascii=False)
            decision.validated_tool_call = {'tool': spec.name, 'args': args}
        return decision, error
    except Exception as exc:
        return None, repr(exc)


def _text_guard(required=(), any_of=()):
    def guard(question, args):
        low = _normalize_for_text_match(get_question_text(question) + ' ' + ' '.join(str(o.text) for o in get_options(question)))
        missing = [token for token in required if token not in low]
        if missing:
            return False, 'missing trigger(s): ' + ', '.join(missing)
        if any_of and not any(token in low for token in any_of):
            return False, 'none of the expected trigger groups is present'
        return True, ''
    return guard


# Generic executable math tools.
def tool_math_evaluate_expression(question, args):
    expression = args.get('expression')
    value = parse_math_expression(expression)
    if value is None:
        raise ValueError('could not parse expression')
    modulus = args.get('modulus')
    if modulus is not None:
        value = sp.Mod(value, int(modulus))
    candidates = [sp.simplify(value)]
    try:
        candidates.append(float(sp.N(value)))
    except Exception:
        pass
    return ToolExecution('math_evaluate_expression', value, f'Evaluated {expression!r} = {value}.', 0.96, candidates)



def _split_equation_text(equations):
    if equations is None:
        return []
    if isinstance(equations, list):
        return [str(x) for x in equations if str(x).strip()]
    text = str(equations)
    parts = re.split(r'\s*[;,]\s*', text)
    return [p for p in parts if p.strip()]


def _parse_variable_list(raw, default='x'):
    if isinstance(raw, list):
        names = [str(x).strip() for x in raw if str(x).strip()]
    else:
        names = [x.strip() for x in re.split(r'[,;\s]+', str(raw or default)) if x.strip()]
    return [sp.Symbol(name) for name in names]


def tool_math_solve_equation(question, args):
    equations = args.get('equations')
    equation = args.get('equation')
    equation_texts = _split_equation_text(equations) or _split_equation_text(equation)
    variables = _parse_variable_list(args.get('variables', args.get('variable', 'x')))
    target = str(args.get('target', '')).strip()

    if len(equation_texts) > 1:
        parsed_equations = []
        for item in equation_texts:
            eq, _ = parse_equation(item, str(variables[0]))
            if eq is None:
                raise ValueError(f'could not parse equation in system: {item!r}')
            parsed_equations.append(eq)
        solutions = sp.solve(parsed_equations, variables, dict=True)
        if not solutions:
            raise ValueError('no symbolic solution')
        solution = solutions[0]
        candidates = []
        explanation_parts = []
        if target:
            target_symbol = sp.Symbol(target)
            if target_symbol in solution:
                candidates.append(sp.simplify(solution[target_symbol]))
        for symbol in variables:
            if symbol in solution:
                value = sp.simplify(solution[symbol])
                explanation_parts.append(f'{symbol}={value}')
                candidates.append(value)
        return ToolExecution(
            'math_solve_equation',
            solution,
            'Solved system: ' + ', '.join(explanation_parts) + '.',
            0.94,
            candidates,
        )

    eq, symbol = parse_equation(equation_texts[0] if equation_texts else equation, str(variables[0]))
    if eq is None:
        raise ValueError('could not parse equation')
    solutions = [sp.simplify(s) for s in sp.solve(eq, symbol)]
    if not solutions:
        raise ValueError('no symbolic solution')
    return ToolExecution('math_solve_equation', solutions[0] if len(solutions) == 1 else solutions, f'Solved {sp.sstr(eq)} for {symbol}: {solutions}.', 0.94, solutions)

def tool_math_modular_arithmetic(question, args):
    # Guard against composite expressions such as "2^87 + 3 divided by 7".
    # The simple modular tool only represents base^exponent mod m; full expressions
    # must be parsed by the deterministic modular-expression handler.
    q_raw = normalize_text(get_question_text(question)).replace('−', '-').replace('–', '-').replace('—', '-')
    if re.search(r'remainder\s+when\s+.+?[+\-*/()].+?\s+is\s+divided\s+by\s+\d+', q_raw, flags=re.I):
        raise ValueError('composite modular expression requires math_modular_expression_remainder, not base^exponent modular_arithmetic')
    base = int(args['base'])
    exponent = int(args['exponent'])
    modulus = int(args['modulus'])
    value = pow(base, exponent, modulus)
    return ToolExecution('math_modular_arithmetic', value, f'Computed pow({base}, {exponent}, {modulus}) = {value}.', 0.98, [sp.Integer(value)])


def tool_math_repeating_decimal_to_fraction(question, args):
    non_repeating = str(args.get('non_repeating', ''))
    repeating = str(args.get('repeating', ''))
    if not repeating:
        match = re.search(r'0\.(\d*)\\overline\{?(\d+)\}?', normalize_text(get_question_text(question)))
        if not match:
            raise ValueError('repeating decimal not found')
        non_repeating, repeating = match.groups()
    numerator = int((non_repeating or '0') + repeating) - int(non_repeating or '0')
    denominator = (10 ** len(non_repeating)) * (10 ** len(repeating) - 1)
    value = sp.Rational(numerator, denominator)
    return ToolExecution('math_repeating_decimal_to_fraction', value, f'Repeating decimal equals {value}.', 0.98, [value])


def tool_math_finite_power_sum(question, args):
    base = parse_math_expression(args['base'])
    start = int(args['start'])
    end = int(args['end'])
    if base is None:
        raise ValueError('could not parse base')
    if start > end:
        start, end = end, start
    value = sp.simplify(sum(base ** k for k in range(start, end + 1)))
    return ToolExecution('math_finite_power_sum', value, f'Summed {args["base"]}^k from k={start} to {end}: {value}.', 0.98, [value])


def tool_math_independent_trials_probability(question, args):
    n = int(args['n'])
    p = float(args.get('p', 0.5))
    sequence = str(args.get('target_sequence', '')).strip()
    successes = args.get('successes')
    if sequence:
        if abs(p - 0.5) < 1e-12 and successes is None:
            value = sp.Rational(1, 2) ** len(sequence)
        else:
            success_symbol = str(args.get('success_symbol', sequence[0]))
            k = sequence.count(success_symbol)
            value = sp.Rational(str(p)) ** k * sp.Rational(str(1 - p)) ** (len(sequence) - k)
    elif successes is not None:
        k = int(successes)
        value = sp.binomial(n, k) * sp.Rational(str(p)) ** k * sp.Rational(str(1 - p)) ** (n - k)
    else:
        raise ValueError('target_sequence or successes is required')
    return ToolExecution('math_independent_trials_probability', value, f'Independent trial probability = {value}.', 0.97, [sp.simplify(value), float(sp.N(value))])



def tool_math_binomial_probability(question, args):
    operation = str(args.get('operation', 'mean_std')).lower()
    operation = {
        'tail_probability': 'at_most',
        'cdf': 'at_most',
        'less_equal': 'at_most',
        'at most': 'at_most',
        'sf': 'greater_than',
        'survival': 'greater_than',
        'at least': 'at_least',
    }.get(operation, operation)
    n = int(args['n'])
    p = float(args['p'])
    if operation in {'mean_std', 'mean_and_std'}:
        mean = n * p
        std = math.sqrt(n * p * (1 - p))
        return ToolExecution('math_binomial_probability', (mean, std), f'Binomial mean={mean:g}; std={std:.6g}.', 0.95, candidate_sequences=[[mean, std]])
    k = int(args['k'])
    p_rat = sp.Rational(str(p))
    if operation == 'exact':
        value = sp.binomial(n, k) * p_rat ** k * (1 - p_rat) ** (n - k)
        trace = f'P(X={k}) for Bin({n},{p}) = {value}.'
    elif operation == 'at_most':
        value = sum(sp.binomial(n, i) * p_rat ** i * (1 - p_rat) ** (n - i) for i in range(k + 1))
        trace = f'P(X<={k}) for Bin({n},{p}) = {value}.'
    elif operation == 'at_least':
        value = sum(sp.binomial(n, i) * p_rat ** i * (1 - p_rat) ** (n - i) for i in range(k, n + 1))
        trace = f'P(X>={k}) for Bin({n},{p}) = {value}.'
    elif operation == 'greater_than':
        value = sum(sp.binomial(n, i) * p_rat ** i * (1 - p_rat) ** (n - i) for i in range(k + 1, n + 1))
        trace = f'P(X>{k}) for Bin({n},{p}) = {value}.'
    else:
        raise ValueError(f'unsupported binomial operation: {operation}')
    value = sp.simplify(value)
    return ToolExecution('math_binomial_probability', value, trace, 0.95, [value, float(sp.N(value))])

def tool_math_proportion_z_test(question, args):
    p0 = float(args['p0'])
    phat = float(args['phat'])
    n = int(args['n'])
    alternative = str(args.get('alternative', 'greater')).lower()
    se = math.sqrt(p0 * (1 - p0) / n)
    z = (phat - p0) / se
    dist = NormalDist()
    if alternative in {'greater', 'right', '>'}:
        p_value = 1 - dist.cdf(z)
    elif alternative in {'less', 'left', '<'}:
        p_value = dist.cdf(z)
    else:
        p_value = 2 * min(dist.cdf(z), 1 - dist.cdf(z))
    return ToolExecution('math_proportion_z_test', p_value, f'z={z:.4g}; p-value={p_value:.6g}.', 0.95, [p_value])



def tool_math_normal_distribution(question, args):
    operation = str(args.get('operation', 'upper_tail')).lower()
    operation = {
        'tail_probability': 'upper_tail',
        'right_tail': 'upper_tail',
        'greater_than': 'upper_tail',
        'lower_tail': 'cdf',
        'left_tail': 'cdf',
    }.get(operation, operation)
    mean = float(args['mean'])
    std = float(args['std'])
    if std <= 0:
        raise ValueError('std must be positive')
    dist = NormalDist(mu=mean, sigma=std)
    if operation == 'upper_tail':
        score = float(args['score'])
        value = 1 - dist.cdf(score)
        return ToolExecution('math_normal_distribution', value, f'z=({score:g}-{mean:g})/{std:g}; upper-tail normal probability = {value:.6g}.', 0.94, [value, value * 100])
    if operation == 'cdf':
        score = float(args['score'])
        value = dist.cdf(score)
        return ToolExecution('math_normal_distribution', value, f'z=({score:g}-{mean:g})/{std:g}; normal CDF = {value:.6g}.', 0.94, [value, value * 100])
    if operation == 'iqr':
        q1, q3 = dist.inv_cdf(0.25), dist.inv_cdf(0.75)
        return ToolExecution('math_normal_distribution', (q1, q3), f'Normal IQR endpoints are {q1:.6g}, {q3:.6g}.', 0.94, candidate_sequences=[[q1, q3], [q3, q1]])
    raise ValueError(f'unsupported normal operation: {operation}')

def _integer_partitions(n, max_part=None):
    if max_part is None or max_part > n:
        max_part = n
    if n == 0:
        yield []
    else:
        for first in range(max_part, 0, -1):
            for rest in _integer_partitions(n - first, first):
                yield [first] + rest


def tool_math_permutation_max_order(question, args):
    n = int(args['n'])
    best = 1
    best_partition = []
    for part in _integer_partitions(n):
        value = 1
        for cycle in part:
            value = int(sp.ilcm(value, cycle))
        if value > best:
            best = value
            best_partition = part
    return ToolExecution('math_permutation_max_order', best, f'Maximum order in S_{n} is lcm of partition {best_partition}: {best}.', 0.98, [sp.Integer(best)])


def tool_math_finite_abelian_group_count(question, args):
    n = int(args['n'])
    factors = sp.factorint(n)
    value = 1
    pieces = []
    for prime, exponent in factors.items():
        count = int(sp.partition(exponent))
        value *= count
        pieces.append(f'p({exponent})={count}')
    return ToolExecution('math_finite_abelian_group_count', value, f'Number of Abelian groups of order {n}: ' + ' * '.join(pieces) + f' = {value}.', 0.98, [sp.Integer(value)])


def _stirling_second(n, k):
    return sum((-1) ** (k - i) * math.comb(k, i) * (i ** n) for i in range(k + 1)) // math.factorial(k)


def tool_math_combinatorics_count(question, args):
    object_type = str(args['object_type']).lower()
    n = int(args.get('n', 0))
    if object_type == 'complete_graph_edges':
        value = n * (n - 1) // 2
        return ToolExecution('math_combinatorics_count', value, f'Complete graph K_{n} has n(n-1)/2 = {value} edges.', 0.97, [sp.Integer(value)])
    if object_type == 'morse_sequences':
        max_len = int(args.get('max_len', n))
        value = sum(2 ** k for k in range(1, max_len + 1))
        return ToolExecution('math_combinatorics_count', value, f'Binary strings of lengths 1..{max_len}: {value}.', 0.96, [sp.Integer(value)])
    if object_type == 'distinguishable_balls_indistinguishable_boxes':
        k = int(args['k'])
        value = sum(_stirling_second(n, used_boxes) for used_boxes in range(1, k + 1))
        return ToolExecution('math_combinatorics_count', value, f'Partitions of {n} distinguishable balls into at most {k} boxes: {value}.', 0.94, [sp.Integer(value)])
    if object_type == 'unlabeled_trees' and n == 5:
        return ToolExecution('math_combinatorics_count', 3, 'There are 3 nonisomorphic trees on 5 vertices.', 0.9, [sp.Integer(3)])
    raise ValueError(f'unsupported combinatorics object_type: {object_type}')



def _net_displacement_from_movements(text):
    dx = 0.0
    dy = 0.0
    low = normalize_text(text).lower()
    segments = re.split(r',\s*(?:and\s+)?(?:then\s+)?|(?:then\s+)|(?:and\s+finally\s+)', low)
    for seg in segments:
        dir_m = re.search(r'(east|west|north|south)', seg)
        num_m = re.search(r'(\d+(?:\.\d+)?)', seg)
        if dir_m and num_m:
            value = float(num_m.group(1))
            d = dir_m.group(1)
            if d == 'east': dx += value
            elif d == 'west': dx -= value
            elif d == 'north': dy += value
            elif d == 'south': dy -= value
    return dx, dy


def tool_math_geometry(question, args):
    operation = str(args['operation']).lower()
    if operation == 'slope_points':
        p1, p2 = args['points']
        value = sp.Rational(str(p2[1] - p1[1])) / sp.Rational(str(p2[0] - p1[0]))
        return ToolExecution('math_geometry', value, f'Slope between points is {value}.', 0.95, [value, float(sp.N(value))])
    if operation == 'equilateral_triangle_area':
        side = sp.Rational(str(args['side']))
        value = sp.sqrt(3) * side ** 2 / 4
        return ToolExecution('math_geometry', value, f'Equilateral triangle area = {value}.', 0.95, [value, float(sp.N(value)), round(float(sp.N(value)))])
    if operation in {'distance_from_origin', 'cardinal_walk_distance'}:
        if args.get('movements'):
            dx, dy = _net_displacement_from_movements(args['movements'])
        elif args.get('points'):
            points = args['points']
            last = points[-1]
            dx, dy = float(last[0]), float(last[1])
        else:
            dx, dy = _net_displacement_from_movements(get_question_text(question))
        value = sp.sqrt(sp.Rational(str(dx)) ** 2 + sp.Rational(str(dy)) ** 2)
        return ToolExecution(
            'math_geometry',
            value,
            f'Net displacement dx={dx:g}, dy={dy:g}; distance=sqrt(dx^2+dy^2)={sp.N(value, 6)}.',
            0.97,
            [value, float(sp.N(value)), round(float(sp.N(value)), 1), round(float(sp.N(value)))],
        )
    raise ValueError(f'unsupported geometry operation: {operation}')

def tool_math_concept_classifier(question, args):
    concept = str(args['concept']).lower()
    low = _normalize_for_text_match(get_question_text(question))
    if concept in {'linear_transformation_mean_std_range', 'mean and standard deviation under linear transformation'}:
        option_id = option_id_by_text(
            question,
            ['mean price', 'increase by 50 cents', 'standard deviation', 'remain the same'],
            exclude=['range']
        ) or option_id_by_text(question, ['mean', 'standard deviation', 'remain the same'], exclude=['range'])
        return ToolExecution(
            'math_concept_classifier',
            'mean shifts, spread unchanged',
            'Adding a constant to every value shifts the mean by that constant; standard deviation and range do not change.',
            0.92,
            option_id=option_id,
        )
    if concept == 'experimental_design':
        option_id = option_id_by_text(question, ['completely randomized', '24 treatment groups'])
        return ToolExecution(
            'math_concept_classifier',
            'completely randomized design with 24 treatment groups',
            'All 4 x 2 x 3 factor combinations are treatment groups; no blocking factor is specified.',
            0.90,
            option_id=option_id,
        )
    if concept == 'correlation_coefficient':
        option_id = option_id_by_text(question, ['+0.87', '-0.87', 'same degree'])
        return ToolExecution(
            'math_concept_classifier',
            'same magnitude correlation',
            'Correlation magnitude controls clustering strength; the sign only changes direction.',
            0.90,
            option_id=option_id,
        )
    if concept in {'mutually_exclusive_vs_independent', 'mutually_exclusive_independent'}:
        option_id = option_id_by_text(question, ['p(a ∩ b) = 0', 'mutually exclusive']) or option_id_by_text(question, ['mutually exclusive'], exclude=['independent'])
        return ToolExecution(
            'math_concept_classifier',
            'zero intersection means mutually exclusive',
            'P(A intersection B)=0 is the definition of mutually exclusive events in this quiz context.',
            0.90,
            option_id=option_id,
        )
    if concept == 'confidence_interval_width':
        option_id = option_id_by_text(question, ['95', 'wider'])
        return ToolExecution('math_concept_classifier', '95 wider', 'Higher confidence produces a wider interval.', 0.9, option_id=option_id)
    if concept == 'type_ii_error':
        if 'probability' in low and 'significance level' in low:
            option_id = option_id_by_text(question, ['insufficient information'])
            return ToolExecution('math_concept_classifier', 'insufficient information', 'Alpha alone does not determine beta.', 0.88, option_id=option_id)
        option_id = option_id_by_text(question, ['continue', 'heartaid', 'more effective'])
        return ToolExecution('math_concept_classifier', 'fail to reject false null', 'Type II error means failing to reject a false null hypothesis.', 0.88, option_id=option_id)
    if concept == 'sampling_error':
        option_id = option_id_by_text(question, ['sample statistic', 'population parameter'])
        return ToolExecution('math_concept_classifier', 'sample statistic estimates parameter', 'Sampling error comes from using a statistic to estimate a population parameter.', 0.88, option_id=option_id)
    if concept == 'blocking':
        option_id = option_id_by_text(question, ['reduce variation within treatments'])
        return ToolExecution('math_concept_classifier', 'reduce within-treatment variation', 'Blocking groups similar units to reduce unexplained variation.', 0.86, option_id=option_id)
    if concept == 'binomial_applicability':
        option_id = option_id_by_text(question, ['none of the above'])
        return ToolExecution('math_concept_classifier', 'binomial criteria', 'A binomial model needs fixed n, binary outcomes, independence, and constant probability.', 0.78, option_id=option_id)
    if concept == 'observational_study':
        option_id = option_id_by_text(question, ['observational study'])
        return ToolExecution('math_concept_classifier', 'observational study', 'No treatment is imposed by the researcher.', 0.86, option_id=option_id)
    raise ValueError(f'unsupported concept: {concept}')


def tool_math_quadratic_threshold_duration(question, args):
    a = sp.Rational(str(args['a']))
    b = sp.Rational(str(args['b']))
    c = sp.Rational(str(args['c']))
    threshold = sp.Rational(str(args['threshold']))
    t = sp.symbols('t', real=True)
    roots = [sp.simplify(r) for r in sp.solve(sp.Eq(a * t**2 + b * t + c, threshold), t)]
    real_roots = sorted([r for r in roots if sp.im(r) == 0], key=lambda x: float(sp.N(x)))
    if len(real_roots) < 2:
        raise ValueError('quadratic threshold crossing needs two real roots')
    duration = sp.simplify(real_roots[-1] - real_roots[0])
    return ToolExecution(
        'math_quadratic_threshold_duration',
        duration,
        f'Time above threshold is the distance between roots {real_roots}: {duration}.',
        0.97,
        [duration, float(sp.N(duration))],
    )


def tool_math_integer_abs_inequality_sum(question, args):
    shift = int(args.get('shift', 3))
    bound = int(args.get('bound', 9))
    search = max(50, abs(shift) + bound + 5)
    values = [n for n in range(-search, search + 1) if abs(n) < abs(n - shift) < bound]
    value = sum(values)
    return ToolExecution(
        'math_integer_abs_inequality_sum',
        value,
        f'Integer solutions are {values}; their sum is {value}.',
        0.98,
        [sp.Integer(value)],
    )


def tool_math_equal_piles_remaining(question, args):
    remaining = sp.Rational(str(args['remaining']))
    piles = int(args.get('piles', 2))
    take_numerator = int(args.get('take_numerator', 1))
    take_denominator = int(args.get('take_denominator', 6))
    taken_from_one_pile = sp.Rational(take_numerator, take_denominator)
    removed_fraction_total = taken_from_one_pile / piles
    original = sp.simplify(remaining / (1 - removed_fraction_total))
    return ToolExecution(
        'math_equal_piles_remaining',
        original,
        f'Remaining amount is (1 - {removed_fraction_total}) of the original, so original = {original}.',
        0.96,
        [original, float(sp.N(original))],
    )


register_tool(ToolSpec('math_evaluate_expression', 'Evaluate a numeric or symbolic expression.', {'expression': 'str'}, {'modulus': 'int'}, tool_math_evaluate_expression))
register_tool(ToolSpec('math_solve_equation', 'Solve one equation or a small system of equations.', {}, {'equation': 'str', 'equations': 'list', 'variable': 'str', 'variables': 'list', 'target': 'str'}, tool_math_solve_equation))
register_tool(ToolSpec('math_quadratic_threshold_duration', 'Duration for which a quadratic trajectory is above a threshold.', {'a': 'number', 'b': 'number', 'c': 'number', 'threshold': 'number'}, {}, tool_math_quadratic_threshold_duration, _text_guard(any_of=('trajectory', 'height', 'above'))))
register_tool(ToolSpec('math_integer_abs_inequality_sum', 'Sum integer n satisfying |n| < |n-shift| < bound.', {'shift': 'int', 'bound': 'int'}, {}, tool_math_integer_abs_inequality_sum, _text_guard(any_of=('integer solutions', '|n|'))))
register_tool(ToolSpec('math_equal_piles_remaining', 'Recover original count after taking a fraction of one of equal piles.', {'remaining': 'number'}, {'piles': 'int', 'take_numerator': 'int', 'take_denominator': 'int'}, tool_math_equal_piles_remaining, _text_guard(any_of=('two piles', 'one pile', 'pins left'))))
register_tool(ToolSpec('math_modular_arithmetic', 'Compute base^exponent modulo modulus.', {'base': 'int', 'exponent': 'int', 'modulus': 'int'}, {}, tool_math_modular_arithmetic, _text_guard(any_of=('remainder', 'divided by', 'units digit', 'mod'))))
register_tool(ToolSpec('math_repeating_decimal_to_fraction', 'Convert a repeating decimal to a common fraction.', {}, {'non_repeating': 'str', 'repeating': 'str'}, tool_math_repeating_decimal_to_fraction, _text_guard(any_of=('overline', 'repeating'))))
register_tool(ToolSpec('math_finite_power_sum', 'Sum base^k for integer k in a finite range.', {'base': 'str', 'start': 'int', 'end': 'int'}, {}, tool_math_finite_power_sum, _text_guard(any_of=('cdots', 'sum', 'compute'))))
register_tool(ToolSpec('math_independent_trials_probability', 'Probability of a fixed sequence or k successes in independent trials.', {'n': 'int'}, {'p': 'float', 'target_sequence': 'str', 'success_symbol': 'str', 'successes': 'int'}, tool_math_independent_trials_probability, _text_guard(any_of=('probability', 'coin', 'sequence'))))
register_tool(ToolSpec('math_binomial_probability', 'Binomial mean/std, exact, at_most, at_least, and tail probabilities.', {'n': 'int', 'p': 'float'}, {'operation': 'str', 'k': 'int'}, tool_math_binomial_probability, _text_guard(any_of=('binomial', 'success', 'probability', 'roll', 'die'))))
register_tool(ToolSpec('math_proportion_z_test', 'One-sample z-test for a population proportion.', {'p0': 'float', 'phat': 'float', 'n': 'int'}, {'alternative': 'str'}, tool_math_proportion_z_test, _text_guard(any_of=('p-value', 'significance test', 'hypothesis'))))
register_tool(ToolSpec('math_normal_distribution', 'Normal CDF, tail probability, or IQR.', {'operation': 'str', 'mean': 'float', 'std': 'float'}, {'score': 'float'}, tool_math_normal_distribution, _text_guard(any_of=('normal', 'normally distributed'))))
register_tool(ToolSpec('math_permutation_max_order', 'Maximum order of an element in S_n.', {'n': 'int'}, {}, tool_math_permutation_max_order, _text_guard(required=('permutation', 'order'))))
register_tool(ToolSpec('math_finite_abelian_group_count', 'Count structurally distinct finite Abelian groups of order n.', {'n': 'int'}, {}, tool_math_finite_abelian_group_count, _text_guard(required=('abelian', 'order'))))
register_tool(ToolSpec('math_combinatorics_count', 'Generic counting: complete graphs, Morse strings, balls into boxes, small tree counts.', {'object_type': 'str'}, {'n': 'int', 'k': 'int', 'max_len': 'int'}, tool_math_combinatorics_count))
register_tool(ToolSpec('math_geometry', 'Geometry computations: slopes, equilateral area, point distance, and cardinal walk distance.', {'operation': 'str'}, {'points': 'list', 'side': 'number', 'movements': 'str'}, tool_math_geometry))
register_tool(ToolSpec('math_concept_classifier', 'Conservative conceptual statistics classifier.', {'concept': 'str'}, {}, tool_math_concept_classifier))


def route_math_tool_deterministically(question):
    text = get_question_text(question)
    raw_text = normalize_text(text).replace('−', '-').replace('–', '-').replace('—', '-')
    low = _normalize_for_text_match(text)
    compact = low.replace(' ', '')

    if 'trajectory' in low and 'above a height' in low and 'h(t)' in raw_text:
        quad = re.search(r'h\(t\)\s*=\s*([-+]?\d+(?:\.\d+)?)\s*t\^2\s*([+-]\s*\d+(?:\.\d+)?)\s*t\s*([+-]\s*\d+(?:\.\d+)?)', raw_text)
        threshold = re.search(r'above a height of\s*\$?\s*(\d+(?:\.\d+)?)', raw_text, flags=re.I)
        if quad and threshold:
            a, b, c = [x.replace(' ', '') for x in quad.groups()]
            return {'tool': 'math_quadratic_threshold_duration', 'args': {'a': a, 'b': b, 'c': c, 'threshold': threshold.group(1)}}
    if 'sum of all integer solutions' in low and '|n|' in raw_text and re.search(r'\|n\s*-\s*\d+\|', raw_text):
        shift_match = re.search(r'\|n\s*-\s*(\d+)\|', raw_text)
        bound_match = re.search(r'\|n\s*-\s*\d+\|\s*<\s*(\d+)', raw_text)
        if shift_match and bound_match:
            return {'tool': 'math_integer_abs_inequality_sum', 'args': {'shift': int(shift_match.group(1)), 'bound': int(bound_match.group(1))}}
    if all(word in low for word in ['east', 'north', 'west', 'south']) and any(word in low for word in ['walked', 'starting point', 'from his starting point']):
        return {'tool': 'math_geometry', 'args': {'operation': 'cardinal_walk_distance', 'movements': raw_text}}
    if 'two piles' in low and 'equal number of pins' in low and 'one-half of one-third of one pile' in low:
        nums = _extract_number_sequence(low)
        if nums:
            return {'tool': 'math_equal_piles_remaining', 'args': {'remaining': nums[-1], 'piles': 2, 'take_numerator': 1, 'take_denominator': 6}}
    if 'increase the prices of all items by 50 cents' in low:
        return {'tool': 'math_concept_classifier', 'args': {'concept': 'linear_transformation_mean_std_range'}}
    if 'in all combinations' in low and 'temperature' in low and 'pans' in low and 'ovens' in low:
        return {'tool': 'math_concept_classifier', 'args': {'concept': 'experimental_design'}}
    if 'correlation coefficient' in low:
        return {'tool': 'math_concept_classifier', 'args': {'concept': 'correlation_coefficient'}}
    if 'any two events a and b' in low and ('mutually exclusive' in low or 'independent' in low):
        return {'tool': 'math_concept_classifier', 'args': {'concept': 'mutually_exclusive_vs_independent'}}

    if 'common fraction' in low and 'overline' in low:
        return {'tool': 'math_repeating_decimal_to_fraction', 'args': {}}
    if re.search(r'i\s*\+\s*i\^', low) and ('cdots' in low or '...' in low):
        exponents = [int(x) for x in re.findall(r'i\^\{?(\d+)\}?', low)]
        end = max(exponents) if exponents else 1
        return {'tool': 'math_finite_power_sum', 'args': {'base': 'i', 'start': 1, 'end': end}}
    m = re.search(r'largest order .* permutations? of (\d+) objects', low)
    if m:
        return {'tool': 'math_permutation_max_order', 'args': {'n': int(m.group(1))}}
    m = re.search(r'abelian groups? have order (\d+)', low)
    if m:
        return {'tool': 'math_finite_abelian_group_count', 'args': {'n': int(m.group(1))}}
    m = re.search(r'complete graph with (\d+) vertices', low)
    if m:
        return {'tool': 'math_combinatorics_count', 'args': {'object_type': 'complete_graph_edges', 'n': int(m.group(1))}}
    if 'morse code' in low and '1, 2, 3, or 4' in low:
        return {'tool': 'math_combinatorics_count', 'args': {'object_type': 'morse_sequences', 'max_len': 4}}
    m = re.search(r'put\s+(\d+)\s+distinguishable balls into\s+(\d+)\s+indistinguishable boxes', low)
    if m:
        return {'tool': 'math_combinatorics_count', 'args': {'object_type': 'distinguishable_balls_indistinguishable_boxes', 'n': int(m.group(1)), 'k': int(m.group(2))}}
    if 'nonisomorphic trees with 5 vertices' in low:
        return {'tool': 'math_combinatorics_count', 'args': {'object_type': 'unlabeled_trees', 'n': 5}}
    m = re.search(r'remainder when\s+(\d+)\^(\d+)\s+is divided by\s+(\d+)', low)
    if m:
        return {'tool': 'math_modular_arithmetic', 'args': {'base': int(m.group(1)), 'exponent': int(m.group(2)), 'modulus': int(m.group(3))}}
    m = re.search(r'units digit .* number\s+(\d+)\^(\d+)', low)
    if m:
        return {'tool': 'math_modular_arithmetic', 'args': {'base': int(m.group(1)), 'exponent': int(m.group(2)), 'modulus': 10}}
    seq_match = re.search(r'\b([TF]{4,})\b', text)
    if seq_match and ('coin' in low or 'true-false' in low or 'probability' in low):
        seq = seq_match.group(1)
        return {'tool': 'math_independent_trials_probability', 'args': {'n': len(seq), 'p': 0.5, 'target_sequence': seq}}
    if 'significance test' in low and 'p-value' in low and 'p>' in compact:
        nums = _extract_number_sequence(low)
        if len(nums) >= 3:
            return {'tool': 'math_proportion_z_test', 'args': {'p0': nums[0], 'phat': nums[1], 'n': int(nums[2]), 'alternative': 'greater'}}
    if 'binomial experiment' in low and 'mean' in low and 'standard deviation' in low:
        nums = _extract_number_sequence(low)
        if len(nums) >= 2:
            return {'tool': 'math_binomial_probability', 'args': {'operation': 'mean_std', 'p': nums[0], 'n': int(nums[1])}}
    if 'normally distributed' in low or 'normal distribution' in low:
        if 'more than' in low or 'contain more than' in low:
            nums = _extract_number_sequence(low)
            if len(nums) >= 3:
                return {'tool': 'math_normal_distribution', 'args': {'operation': 'upper_tail', 'mean': nums[0], 'std': nums[1], 'score': nums[2]}}
        if 'interquartile range' in low:
            nums = _extract_number_sequence(low)
            if len(nums) >= 2:
                return {'tool': 'math_normal_distribution', 'args': {'operation': 'iqr', 'mean': nums[0], 'std': nums[1]}}
    if 'confidence interval' in low and '90' in low and '95' in low and ('length' in low or 'wider' in low):
        return {'tool': 'math_concept_classifier', 'args': {'concept': 'confidence_interval_width'}}
    if 'type ii error' in low:
        return {'tool': 'math_concept_classifier', 'args': {'concept': 'type_ii_error'}}
    if 'sampling error occurs' in low:
        return {'tool': 'math_concept_classifier', 'args': {'concept': 'sampling_error'}}
    if 'main purpose of blocking' in low:
        return {'tool': 'math_concept_classifier', 'args': {'concept': 'blocking'}}
    if 'binomial distribution is an appropriate model' in low:
        return {'tool': 'math_concept_classifier', 'args': {'concept': 'binomial_applicability'}}
    if 'relative maximum' in low and 'ln x' in low and '/x' in low:
        return {'tool': 'math_evaluate_expression', 'args': {'expression': '1/e'}}
    if 'equilateral triangle' in low and 'area' in low:
        nums = _extract_number_sequence(low)
        if nums:
            return {'tool': 'math_geometry', 'args': {'operation': 'equilateral_triangle_area', 'side': nums[0]}}
    if 'slope of the line' in low:
        nums = _extract_number_sequence(low)
        if len(nums) >= 4:
            return {'tool': 'math_geometry', 'args': {'operation': 'slope_points', 'points': [[nums[0], nums[1]], [nums[2], nums[3]]]}}
    if any(t in low for t in ['value of the expression', 'evaluate the expression', 'counting number is equivalent to the expression']):
        expression = extract_display_math_expression(question)
        if expression:
            return {'tool': 'math_evaluate_expression', 'args': {'expression': expression}}
    return None


# ── Maths V6: deterministic shortcuts + option substitution ───────────────
# Goal: solve more quiz-style Maths questions before falling back to Micro-CoT.
# These handlers are intentionally conservative: they return an answer only when
# exactly one option matches a deterministic computation.
USE_MATH_DETERMINISTIC_SHORTCUTS = True
USE_MATH_OPTION_SUBSTITUTION = True
USE_MATH_MICRO_COT_VERIFY_OVERRIDE = False  # Previous verifier was too aggressive in logs.


def _option_sympy_values(question):
    values = []
    for opt in get_options(question):
        text = normalize_text(str(opt.text)).replace('$', '').replace(',', '').strip()
        if text.endswith('%'):
            pct = parse_math_expression(text[:-1])
            if pct is not None:
                values.append((int(opt.id), opt.text, sp.simplify(pct / 100)))
                values.append((int(opt.id), opt.text, sp.simplify(pct)))
            continue
        parsed = parse_math_expression(text)
        if parsed is not None:
            values.append((int(opt.id), opt.text, parsed))
    return values


def _unique_option_from_value(question, value, tolerance=1e-6):
    matches = []
    for opt_id, opt_text, opt_val in _option_sympy_values(question):
        try:
            if sp.simplify(opt_val - value) == 0:
                matches.append(opt_id)
                continue
        except Exception:
            pass
        if _numeric_values_close(opt_val, value, tolerance=tolerance):
            matches.append(opt_id)
    matches = sorted(set(matches))
    return matches[0] if len(matches) == 1 else None


def _decision_from_option(label, option_id, explanation, confidence=0.94, args=None):
    call = {'tool': label, 'args': args or {}}
    decision = _make_tool_decision(
        option_id,
        f'deterministic_{label}',
        confidence,
        explanation,
        raw_tool_call=json.dumps(call, ensure_ascii=False),
        validated_tool_call=call,
    )
    return decision


def _roman_labels_from_option_text(text):
    raw = normalize_text(text).upper()
    return re.findall(r'\b(?:IV|V|III|II|I)\b', raw)


def _match_roman_statement_option(question, true_labels):
    true_set = set(true_labels)
    candidates = []
    for opt in get_options(question):
        labels = set(_roman_labels_from_option_text(opt.text))
        low = _normalize_for_text_match(opt.text)
        if not true_set and ('none' in low or 'neither' in low):
            candidates.append(int(opt.id))
        elif labels == true_set:
            candidates.append(int(opt.id))
    candidates = sorted(set(candidates))
    return candidates[0] if len(candidates) == 1 else None


def _parse_statement_blocks(text):
    blocks = []
    pattern = re.compile(r'\b(IV|V|III|II|I)\.\s*(.*?)(?=\s+\b(?:IV|V|III|II|I)\.|$)', re.S)
    for label, body in pattern.findall(normalize_text(text)):
        blocks.append((label, body.strip()))
    return blocks


def _parse_poly_in_lambda(poly_text):
    # Use L internally because 'lambda' is a Python keyword and cannot be parsed as a symbol.
    lam = sp.Symbol('L')
    cleaned = normalize_text(poly_text)
    cleaned = cleaned.replace('λ', 'L').replace('lambda', 'L').replace('−', '-').replace('^', '**')
    cleaned = cleaned.replace(' ', '')
    cleaned = re.sub(r'(\d)(L)', r'\1*\2', cleaned)
    try:
        expr = parse_expr(cleaned, local_dict={'L': lam}, transformations=standard_transformations + (implicit_multiplication_application, convert_xor), evaluate=True)
        return sp.expand(expr), lam
    except Exception:
        return None, lam


def _shortcut_matrix_characteristic(question):
    text = get_question_text(question)
    low = _normalize_for_text_match(text)
    raw_low = normalize_text(text).lower()
    if 'det' not in low or ('lambda' not in low and 'λ' not in raw_low) or 'trace' not in low:
        return None
    m = re.search(r'det\s*\([^)]*lambda[^)]*\)\s*=\s*([^,\n]+)', normalize_text(text), flags=re.I)
    if not m:
        m = re.search(r'det\s*\([^)]*λ[^)]*\)\s*=\s*([^,\n]+)', normalize_text(text), flags=re.I)
    if not m:
        return None
    poly, lam = _parse_poly_in_lambda(m.group(1))
    if poly is None:
        return None
    degree = int(sp.Poly(poly, lam).degree())
    coeff = sp.Poly(poly, lam).coeff_monomial(lam ** (degree - 1))
    trace = sp.simplify(coeff / ((-1) ** (degree + 1)))
    determinant = sp.simplify(poly.subs(lam, 0))
    roots = [sp.simplify(r) for r in sp.solve(sp.Eq(poly, 0), lam)]

    true_labels = []
    for label, body in _parse_statement_blocks(text):
        b = _normalize_for_text_match(body)
        is_true = None
        nums = [sp.Integer(int(x)) for x in re.findall(r'[-+]?\d+', b)]
        if 'trace' in b and nums:
            is_true = bool(sp.simplify(trace - nums[-1]) == 0)
        elif ('determinant' in b or 'determinate' in b) and nums:
            is_true = bool(sp.simplify(determinant - nums[-1]) == 0)
        elif 'eigenvalue' in b and nums:
            is_true = all(any(sp.simplify(root - n) == 0 for root in roots) for n in nums)
        if is_true:
            true_labels.append(label)
    if not true_labels and _parse_statement_blocks(text):
        return None
    opt_id = _match_roman_statement_option(question, true_labels)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_matrix_characteristic_statements',
        opt_id,
        f'For det(A-lambda I)={sp.sstr(poly)}, trace={trace}, det(A)={determinant}, eigenvalues={roots}; true statements={true_labels}.',
        0.97,
        {'poly': sp.sstr(poly), 'trace': sp.sstr(trace), 'det': sp.sstr(determinant), 'roots': [sp.sstr(r) for r in roots]},
    )


def _shortcut_ring_characteristic(question):
    text = get_question_text(question)
    low = _normalize_for_text_match(text)
    if 'characteristic' not in low or 'ring' not in low or 'z_' not in low:
        return None
    nums = [int(x) for x in re.findall(r'Z[_\s]*\{?(\d+)\}?', normalize_text(text), flags=re.I)]
    if not nums:
        nums = [int(x) for x in re.findall(r'Z_(\d+)', normalize_text(text), flags=re.I)]
    if not nums:
        return None
    value = int(nums[0])
    for n in nums[1:]:
        value = int(sp.ilcm(value, n))
    opt_id = _unique_option_from_value(question, sp.Integer(value))
    if opt_id is None:
        return None
    return _decision_from_option('math_ring_characteristic', opt_id, f'Characteristic of a product of Z_n rings is lcm({nums})={value}.', 0.98, {'moduli': nums})


def _shortcut_rectangle_diagonal(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if 'rectangle' not in low or 'twice its width' not in low or 'diagonal' not in low:
        return None
    m = re.search(r'diagonal\s+is\s+\$?([^,$.]+(?:sqrt\{?\d+\}?|\\sqrt\{?\d+\}?|√\d+)?[^,.?]*)', text, flags=re.I)
    diag_expr = m.group(1).strip() if m else None
    if not diag_expr:
        spans = re.findall(r'\$(.*?)\$', text)
        diag_expr = spans[-1] if spans else None
    if not diag_expr:
        return None
    d = parse_math_expression(diag_expr)
    if d is None:
        return None
    area = sp.simplify(2 * d ** 2 / 5)
    opt_id = _unique_option_from_value(question, area, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option('math_rectangle_twice_width_diagonal', opt_id, f'If L=2w, then d^2=5w^2 and area=2w^2={area}.', 0.98, {'diagonal': diag_expr})


def _shortcut_positive_integer_sum_cubic(question):
    text = normalize_text(get_question_text(question))
    low_compact = _normalize_for_text_match(text).replace(' ', '')
    if 'positiveintegers' not in low_compact or '(a+b+c)^3' not in low_compact or 'a+b+c' not in low_compact:
        return None
    m = re.search(r'=\s*([-+]?\d+)', text)
    if not m:
        return None
    target = int(m.group(1))
    matches = []
    for opt_id, opt_text, opt_val in _option_sympy_values(question):
        if not bool(opt_val.is_integer):
            continue
        S = int(opt_val)
        if S < 3 or S > 200:
            continue
        found = False
        for a in range(1, S - 1):
            for b in range(1, S - a):
                c = S - a - b
                if c >= 1 and S ** 3 - a ** 3 - b ** 3 - c ** 3 == target:
                    found = True
                    break
            if found:
                break
        if found:
            matches.append(int(opt_id))
    matches = sorted(set(matches))
    if len(matches) != 1:
        return None
    return _decision_from_option('math_positive_integer_sum_cubic', matches[0], f'Tried option sums S and found the unique positive-integer sum satisfying the equation = {target}.', 0.96, {'target': target})


def _parse_custom_star_definition(text):
    m = re.search(r'([a-z])\s+star\s+([a-z])\s*=\s*(.*?)(?:\.|;|,|\n|\s+if\s+)', normalize_text(text), flags=re.I)
    if not m:
        return None
    left_var, right_var, expr_text = m.groups()
    expr_text = expr_text.strip()
    expr_text = re.sub(fr'\b{left_var}{right_var}\b', f'{left_var}*{right_var}', expr_text)
    expr_text = expr_text.replace('^', '**')
    a, b = sp.Symbol(left_var), sp.Symbol(right_var)
    try:
        expr = parse_expr(expr_text, local_dict={left_var: a, right_var: b}, transformations=standard_transformations + (implicit_multiplication_application, convert_xor), evaluate=True)
        return left_var, right_var, sp.simplify(expr)
    except Exception:
        return None


def _shortcut_custom_star_operation(question):
    text = normalize_text(get_question_text(question))
    if ' star ' not in _normalize_for_text_match(text):
        return None
    parsed = _parse_custom_star_definition(text)
    if parsed is None:
        return None
    left_var, right_var, expr = parsed
    m = re.search(r'([-+]?\d+)\s+star\s+([a-z])\s*=\s*([-+]?\d+)', text, flags=re.I)
    if not m:
        return None
    left_value = sp.Integer(int(m.group(1)))
    unknown = m.group(2)
    target = sp.Integer(int(m.group(3)))
    matches = []
    for opt_id, opt_text, opt_val in _option_sympy_values(question):
        try:
            value = sp.simplify(expr.subs({sp.Symbol(left_var): left_value, sp.Symbol(right_var): opt_val}))
            if sp.simplify(value - target) == 0:
                matches.append(opt_id)
        except Exception:
            pass
    matches = sorted(set(matches))
    if len(matches) != 1:
        return None
    return _decision_from_option('math_custom_star_option_substitution', matches[0], f'Substituted each option into {left_value} star {unknown}; unique match gives {target}.', 0.98, {'expr': sp.sstr(expr), 'target': int(target)})


def _parse_one_variable_expr(expr_text, variable='x'):
    x = sp.Symbol(variable)
    cleaned = _math_text_for_parse(expr_text).replace(' ', '')
    try:
        expr = parse_expr(cleaned, local_dict={variable: x, **_sympy_local_dict()}, transformations=standard_transformations + (implicit_multiplication_application, convert_xor), evaluate=True)
        return sp.simplify(expr), x
    except Exception:
        return None, x


def _shortcut_quadratic_extremum(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if 'maximum value' not in low and 'minimum value' not in low:
        return None
    m = re.search(r'(?:maximum|minimum) value of\s+\$?(.+?)(?:\$|\?|\.)', text, flags=re.I | re.S)
    if not m:
        return None
    expr, x = _parse_one_variable_expr(m.group(1), 'x')
    if expr is None:
        return None
    poly = sp.Poly(sp.expand(expr), x)
    if poly.degree() != 2:
        return None
    a = poly.coeff_monomial(x ** 2)
    b = poly.coeff_monomial(x)
    if a == 0:
        return None
    want_max = 'maximum value' in low
    if want_max and a >= 0:
        return None
    if (not want_max) and a <= 0:
        return None
    xv = sp.simplify(-b / (2 * a))
    value = sp.simplify(expr.subs(x, xv))
    opt_id = _unique_option_from_value(question, value, tolerance=1e-6)
    if opt_id is None:
        return None
    kind = 'maximum' if want_max else 'minimum'
    return _decision_from_option('math_quadratic_extremum', opt_id, f'Quadratic {kind} at x={xv}; value={value}.', 0.97, {'expr': sp.sstr(expr), 'x_vertex': sp.sstr(xv)})


def _shortcut_function_composition(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text).replace(' ', '')
    if 'f(g(h(x)))' not in low:
        return None
    x = sp.Symbol('x')
    defs = {}
    for name in ('f', 'g', 'h'):
        m = re.search(fr'{name}\s*\(\s*x\s*\)\s*=\s*([^,;.]+)', text, flags=re.I)
        if not m:
            return None
        expr, _ = _parse_one_variable_expr(m.group(1), 'x')
        if expr is None:
            return None
        defs[name] = expr
    value = sp.simplify(defs['f'].subs(x, defs['g'].subs(x, defs['h'])))
    opt_id = option_id_by_any_value([value], question, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option('math_function_composition', opt_id, f'Composed f(g(h(x))) = {sp.sstr(value)}.', 0.97, {'composition': sp.sstr(value)})


def _shortcut_coin_turn_game(question):
    low = _normalize_for_text_match(get_question_text(question))
    if 'coin' not in low or 'take turns' not in low or ('first player' not in low and 'player a' not in low):
        return None
    if 'fair' not in low:
        return None
    value = sp.Rational(2, 3)
    opt_id = _unique_option_from_value(question, value, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option('math_coin_turn_game_first_player', opt_id, 'Alternating fair-coin first-success game gives 1/2 + 1/8 + ... = 2/3.', 0.88, {'p': '1/2'})


def _shortcut_compound_doubling(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if 'double' not in low or 'year' not in low:
        return None
    m = re.search(r'(?:takes?|take|every)\s+(\d+)\s+years?\s+to\s+double|double\s+every\s+(\d+)\s+years?', low)
    if not m:
        return None
    doubling_years = int(next(g for g in m.groups() if g))
    money = [float(x.replace(',', '')) for x in re.findall(r'\$\s*([0-9][0-9,]*(?:\.\d+)?)', text)]
    if len(money) >= 2 and any(t in low for t in ['how long', 'how many years', 'how many year']):
        ratio = money[-1] / money[0]
        if ratio > 0:
            k = math.log(ratio, 2)
            if abs(k - round(k)) < 1e-9:
                years = int(round(k)) * doubling_years
                opt_id = _unique_option_from_value(question, sp.Integer(years), tolerance=1e-6)
                if opt_id is not None:
                    return _decision_from_option('math_compound_doubling_time', opt_id, f'Amount changes by factor {ratio:g}; years={years}.', 0.95, {'doubling_years': doubling_years})
    y = re.search(r'after\s+(\d+)\s+years?', low)
    if money and y:
        years = int(y.group(1))
        k = years / doubling_years
        if abs(k - round(k)) < 1e-9:
            amount = sp.Integer(int(round(money[0] * (2 ** int(round(k))))))
            opt_id = _unique_option_from_value(question, amount, tolerance=1e-6)
            if opt_id is not None:
                return _decision_from_option('math_compound_doubling_value', opt_id, f'{money[0]:g} doubles {int(round(k))} times in {years} years, giving {amount}.', 0.95, {'doubling_years': doubling_years})
    return None


def _try_option_substitute_single_equation(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if not any(t in low for t in ['find x', 'solve for x', 'what is x', 'value of x']):
        return None
    equations = []
    spans = re.findall(r'\$(.*?)\$', text)
    equations.extend([s for s in spans if '=' in s and re.search(r'\bx\b', s)])
    equations.extend(re.findall(r'([^.?;\n]*x[^.?;\n]*=[^.?;\n]*)', text, flags=re.I))
    seen = set()
    for eq_text in equations:
        eq_text = eq_text.strip(' ,')
        if eq_text in seen:
            continue
        seen.add(eq_text)
        eq, symbol = parse_equation(eq_text, variable='x')
        if eq is None:
            continue
        matches = []
        for opt_id, opt_text, opt_val in _option_sympy_values(question):
            try:
                residual = sp.simplify(eq.lhs.subs(symbol, opt_val) - eq.rhs.subs(symbol, opt_val))
                if residual == 0:
                    matches.append(opt_id)
            except Exception:
                pass
        matches = sorted(set(matches))
        if len(matches) == 1:
            return _decision_from_option('math_option_substitution_equation', matches[0], f'Substituted numeric options into equation {eq_text!r}; unique solution matched.', 0.96, {'equation': eq_text})
    return None



# ── Maths V8: low-risk canonical handlers ───────────────────────────────
# This keeps the V6 option-substitution pipeline as the base and adds only
# conservative handlers observed in logs. No LLM is used in this block.

def _option_id_by_insufficient_information(question):
    patterns = [
        ['cannot', 'determin'],
        ['can', 'not', 'determin'],
        ['insufficient', 'information'],
        ['not', 'enough', 'information'],
        ['cannot', 'be', 'computed'],
    ]
    for opt in get_options(question):
        low = _normalize_for_text_match(opt.text)
        if any(all(tok in low for tok in pat) for pat in patterns):
            return int(opt.id)
    return None


def _shortcut_mean_of_numbers(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if not any(token in low for token in ['mean', 'average', 'arithmetic mean']):
        return None
    if any(token in low for token in ['standard deviation', 'variance', 'weighted average', 'weighted mean']):
        return None

    nums = [sp.Rational(str(x)) for x in _extract_number_sequence(text)]
    # Avoid very small lists because those are often parameters rather than a data set.
    if len(nums) < 3:
        return None
    value = sp.simplify(sum(nums) / len(nums))
    opt_id = _unique_option_from_value(question, value, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_mean_of_numbers',
        opt_id,
        f'Mean of {len(nums)} values is {sp.sstr(value)}.',
        0.98,
        {'numbers': [sp.sstr(n) for n in nums], 'mean': sp.sstr(value)},
    )


def _shortcut_lcm_from_factors(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    compact = low.replace(' ', '')
    trigger = (
        'least common multiple' in low
        or 'lcm' in low
        or (
            any(t in low for t in ['smallest positive integer', 'least positive integer', 'least number', 'smallest number'])
            and any(t in low for t in ['factor', 'factors', 'divisible', 'multiple'])
        )
    )
    if not trigger:
        return None
    nums = [int(float(x)) for x in _extract_number_sequence(text)]
    nums = [n for n in nums if n > 1]
    if len(nums) < 2:
        return None
    value = int(nums[0])
    for n in nums[1:]:
        value = int(sp.ilcm(value, int(n)))
    opt_id = _unique_option_from_value(question, sp.Integer(value), tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_lcm_from_factors',
        opt_id,
        f'Least positive integer divisible by {nums} is lcm={value}.',
        0.98,
        {'numbers': nums, 'lcm': value},
    )


def _shortcut_constant_function(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if not any(t in low for t in ['for all real', 'for every real', 'for any real', 'constant function']):
        return None
    m = re.search(r'([a-z])\s*\(\s*x\s*\)\s*=\s*([-+]?\d+(?:\.\d+)?|[-+]?\d+\s*/\s*\d+)', text, flags=re.I)
    if not m:
        return None
    fname, const_text = m.groups()
    if f'{fname.lower()}(' not in low and 'constant function' not in low:
        return None
    value = parse_math_expression(const_text)
    if value is None:
        return None
    opt_id = _unique_option_from_value(question, value, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_constant_function',
        opt_id,
        f'{fname}(x) is constant and equal to {sp.sstr(value)}, so every input has that value.',
        0.98,
        {'function': fname, 'constant': sp.sstr(value)},
    )


def _normalize_star_question_text(text):
    text = normalize_text(text)
    text = text.replace('⋆', ' star ').replace('★', ' star ').replace('✶', ' star ')
    text = re.sub(r'\\star\b', ' star ', text)
    text = re.sub(r'\s+', ' ', text)
    return text


def _parse_custom_star_definition_v8(text):
    text = _normalize_star_question_text(text)
    # Accept both "a star b = ..." and "a * b is defined as ..." only for explicit star text.
    m = re.search(r'\b([a-z])\s+star\s+([a-z])\s*(?:=|is\s+defined\s+as)\s*(.*?)(?:\.|;|,|\n|\s+if\s+)', text, flags=re.I)
    if not m:
        return None
    left_var, right_var, expr_text = m.groups()
    expr_text = expr_text.strip()
    expr_text = expr_text.replace('−', '-').replace('^', '**')
    expr_text = re.sub(fr'\b{left_var}\s*{right_var}\b', f'{left_var}*{right_var}', expr_text, flags=re.I)
    a, b = sp.Symbol(left_var), sp.Symbol(right_var)
    try:
        expr = parse_expr(
            expr_text,
            local_dict={left_var: a, right_var: b, **_sympy_local_dict()},
            transformations=standard_transformations + (implicit_multiplication_application, convert_xor),
            evaluate=True,
        )
        return left_var, right_var, sp.simplify(expr)
    except Exception:
        return None


# Override the V6 custom-star shortcut with a more tolerant implementation.
def _shortcut_custom_star_operation(question):
    text = _normalize_star_question_text(get_question_text(question))
    if ' star ' not in _normalize_for_text_match(text):
        return None
    parsed = _parse_custom_star_definition_v8(text)
    if parsed is None:
        return None
    left_var, right_var, expr = parsed
    m = re.search(r'([-+]?\d+(?:\.\d+)?)\s+star\s+([a-z])\s*=\s*([-+]?\d+(?:\.\d+)?)', text, flags=re.I)
    if not m:
        return None
    left_value = parse_math_expression(m.group(1))
    unknown = m.group(2)
    target = parse_math_expression(m.group(3))
    if left_value is None or target is None:
        return None
    matches = []
    for opt_id, opt_text, opt_val in _option_sympy_values(question):
        try:
            value = sp.simplify(expr.subs({sp.Symbol(left_var): left_value, sp.Symbol(right_var): opt_val}))
            if sp.simplify(value - target) == 0:
                matches.append(opt_id)
        except Exception:
            pass
    matches = sorted(set(matches))
    if len(matches) != 1:
        return None
    return _decision_from_option(
        'math_custom_star_option_substitution',
        matches[0],
        f'Substituted each option into {sp.sstr(expr)} with {left_var}={sp.sstr(left_value)}; unique match gives {sp.sstr(target)}.',
        0.98,
        {'expr': sp.sstr(expr), 'target': sp.sstr(target), 'unknown': unknown},
    )


def _normalize_sqrt_expression_text(text):
    text = normalize_text(text).replace('−', '-').replace('–', '-').replace('—', '-')
    text = re.sub(r'\\sqrt\s*\{([^{}]+)\}', r'sqrt(\1)', text)
    text = re.sub(r'√\s*\(([^()]+)\)', r'sqrt(\1)', text)
    text = re.sub(r'√\s*([A-Za-z0-9_+\-*/^]+)', r'sqrt(\1)', text)
    return text


def _extract_balanced_function_args(text, func_name='sqrt'):
    args = []
    pattern = func_name + '('
    i = 0
    while True:
        start = text.find(pattern, i)
        if start < 0:
            break
        j = start + len(pattern)
        depth = 1
        while j < len(text) and depth > 0:
            if text[j] == '(':
                depth += 1
            elif text[j] == ')':
                depth -= 1
            j += 1
        if depth == 0:
            args.append(text[start + len(pattern): j - 1].strip())
            i = j
        else:
            break
    return args


def _shortcut_sqrt_domain_width(question):
    text = _normalize_sqrt_expression_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if 'sqrt' not in text.lower() or 'domain' not in low or not any(t in low for t in ['width', 'length', 'range of x', 'interval length']):
        return None
    radicands = _extract_balanced_function_args(text, 'sqrt')
    if not radicands:
        return None
    domain = sp.Interval(-sp.oo, sp.oo)
    used = []
    for arg in radicands:
        expr, symbol = _parse_one_variable_expr(arg, 'x')
        if expr is None or not expr.has(symbol):
            continue
        sol = sp.solve_univariate_inequality(expr >= 0, symbol, relational=False)
        domain = domain.intersect(sol)
        used.append(sp.sstr(expr))
    if not used or not isinstance(domain, sp.Interval):
        return None
    if domain.start in (sp.S.NegativeInfinity, -sp.oo) or domain.end in (sp.S.Infinity, sp.oo):
        return None
    width = sp.simplify(domain.end - domain.start)
    opt_id = _unique_option_from_value(question, width, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_sqrt_domain_width',
        opt_id,
        f'Domain is {domain}; width is {sp.sstr(width)}.',
        0.96,
        {'radicands': used, 'domain': sp.sstr(domain), 'width': sp.sstr(width)},
    )


def _shortcut_variance_without_independence(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if not any(t in low for t in ['var(', 'variance']) or not any(t in low for t in ['x+y', 'x + y', 'sum of x and y']):
        return None
    if any(t in low for t in ['independent', 'uncorrelated', 'covariance', 'cov(']):
        return None
    # E(X+Y) is computable, but Var(X+Y) needs covariance. Prefer the explicit insufficient-information option.
    opt_id = _option_id_by_insufficient_information(question)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_variance_unknown_covariance',
        opt_id,
        'E(X+Y) is determined by linearity, but Var(X+Y)=Var(X)+Var(Y)+2Cov(X,Y), so covariance/independence is needed.',
        0.95,
        {'independence_given': False},
    )


# ── Maths V11: extra deterministic handlers from recent Micro-CoT failures ──
# These are conservative, high-precision handlers. They run before Micro-CoT so
# the LLM is not asked to solve routine computations under the 30s game timer.

def _shortcut_mean_after_adding_value(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if 'mean' not in low or not any(t in low for t in ['added', 'add', '8th', 'new number', 'additional']):
        return None
    # Pattern: mean of 7 numbers is 15; when an 8th number is added, mean becomes/decreases to 12.
    m = re.search(
        r'mean\s+of\s+(\d+)\s+numbers?\s+is\s+([-+]?\d+(?:\.\d+)?).*?(?:when\s+an?\s+)?(?:\d+(?:st|nd|rd|th)?\s+)?(?:number\s+is\s+)?added.*?(?:mean\s+(?:is|becomes|decreases\s+to|increases\s+to)\s+)([-+]?\d+(?:\.\d+)?)',
        low,
        flags=re.I | re.S,
    )
    if not m:
        return None
    n = sp.Integer(int(m.group(1)))
    old_mean = sp.Rational(str(m.group(2)))
    new_mean = sp.Rational(str(m.group(3)))
    added_value = sp.simplify((n + 1) * new_mean - n * old_mean)
    opt_id = _unique_option_from_value(question, added_value, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_mean_added_value',
        opt_id,
        f'Old total={n}*{old_mean}; new total={n+1}*{new_mean}; added value={sp.sstr(added_value)}.',
        0.98,
        {'n': int(n), 'old_mean': sp.sstr(old_mean), 'new_mean': sp.sstr(new_mean), 'added_value': sp.sstr(added_value)},
    )


def _shortcut_greatest_odd_factor_factorial(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if 'greatest odd' not in low or 'factor' not in low or '!' not in text:
        return None
    m = re.search(r'(\d+)\s*!', text)
    if not m:
        return None
    n = int(m.group(1))
    if n < 0 or n > 50:
        return None
    value = math.factorial(n)
    while value % 2 == 0:
        value //= 2
    opt_id = _unique_option_from_value(question, sp.Integer(value), tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_greatest_odd_factor_factorial',
        opt_id,
        f'{n}! with all factors of 2 removed gives greatest odd factor {value}.',
        0.98,
        {'n': n, 'greatest_odd_factor': value},
    )


def _shortcut_sum_of_squares_range(question):
    text = normalize_text(get_question_text(question)).replace('−', '-').replace('–', '-')
    low = _normalize_for_text_match(text)
    if 'sum' not in low or '^2' not in text:
        return None
    pattern = re.compile(r'(\d+)\s*\^\s*2\s*\+\s*(\d+)\s*\^\s*2\s*\+\s*(?:\\cdots|\.\.\.|⋯)\s*\+\s*(\d+)\s*\^\s*2', re.I | re.S)
    matches = pattern.findall(text)
    if not matches:
        return None
    a, _, b = matches[-1]
    a, b = int(a), int(b)
    if a > b:
        a, b = b, a
    if b - a > 10000:
        return None
    value = sum(k*k for k in range(a, b + 1))
    opt_id = _unique_option_from_value(question, sp.Integer(value), tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_sum_of_squares_range',
        opt_id,
        f'Sum of squares from {a} to {b} is {value}.',
        0.98,
        {'start': a, 'end': b, 'sum': value},
    )


def _shortcut_equal_base_exponential_sum(question):
    text = normalize_text(get_question_text(question)).replace('−', '-').replace('–', '-')
    low = _normalize_for_text_match(text)
    if 'sum of all possible values of x' not in low or '=' not in text or '^' not in text:
        return None
    m = re.search(r'(\d+)\s*\^\s*\{([^{}]+)\}\s*=\s*(\d+)\s*\^\s*\{([^{}]+)\}', text)
    if not m:
        return None
    b1, e1_text, b2, e2_text = int(m.group(1)), m.group(2), int(m.group(3)), m.group(4)
    if b1 <= 1 or b2 <= 1:
        return None
    k = None
    power = b1
    for exp in range(1, 12):
        if power == b2:
            k = exp
            break
        power *= b1
    if k is None:
        return None
    e1, x = _parse_one_variable_expr(e1_text, 'x')
    e2, _ = _parse_one_variable_expr(e2_text, 'x')
    if e1 is None or e2 is None:
        return None
    sols = [sp.simplify(s) for s in sp.solve(sp.Eq(e1, k * e2), x)]
    if not sols:
        return None
    value = sp.simplify(sum(sols))
    opt_id = _unique_option_from_value(question, value, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_equal_base_exponential_sum',
        opt_id,
        f'{b2}={b1}^{k}; solve {sp.sstr(e1)}={k}({sp.sstr(e2)}); sum of roots={sp.sstr(value)}.',
        0.97,
        {'base': b1, 'power': k, 'solutions': [sp.sstr(s) for s in sols], 'sum': sp.sstr(value)},
    )


def _shortcut_sphere_in_cube_probability(question):
    text = normalize_text(get_question_text(question)).replace('−', '-').replace('–', '-')
    text = text.replace('\\leq', '<=').replace('\\le', '<=').replace('≤', '<=')
    low = _normalize_for_text_match(text)
    compact = low.replace(' ', '')
    if 'probability' not in low or 'x^2+y^2+z^2' not in compact or '<=1' not in compact:
        return None
    if not all(t in compact for t in ['-1<=x<=1', '-1<=y<=1', '-1<=z<=1']):
        # Some rendering inserts spaces/commas; still require the visible [-1,1] cube idea.
        if text.count('-1') < 3 or text.count('1') < 6:
            return None
    value = sp.pi / 6
    opt_id = option_id_by_any_value([value], question, tolerance=1e-6)
    if opt_id is None:
        for opt in get_options(question):
            opt_norm = normalize_text(opt.text).lower().replace(' ', '')
            opt_ascii = opt_norm.replace('\\', '')
            if 'pi}{6' in opt_ascii or 'pi/6' in opt_ascii or 'π/6' in opt_ascii:
                opt_id = int(opt.id)
                break
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_sphere_in_cube_probability',
        opt_id,
        'Unit sphere volume is 4*pi/3 and cube volume is 8, so probability is pi/6.',
        0.98,
        {'sphere_volume': '4*pi/3', 'cube_volume': 8, 'probability': 'pi/6'},
    )


def _shortcut_relative_speed_faster(question):
    text = normalize_text(get_question_text(question)).replace('\\%', '%')
    low = _normalize_for_text_match(text)
    if 'faster' not in low or 'miles further' not in low or 'hour' not in low:
        return None
    pct_m = re.search(r'\$?(\d+(?:\.\d+)?)\s*%\$?\s*faster', text, flags=re.I)
    gap_m = re.search(r'\$?(\d+(?:\.\d+)?)\$?\s*miles?\s+further', text, flags=re.I)
    hour_m = re.search(r'in\s+\$?(\d+(?:\.\d+)?)\$?\s+hours?', text, flags=re.I)
    if hour_m:
        hour_value = hour_m.group(1)
    else:
        word_m = re.search(r'in\s+(one|two|three|four|five|six|seven|eight|nine|ten)\s+hours?', text, flags=re.I)
        word_map = {'one': 1, 'two': 2, 'three': 3, 'four': 4, 'five': 5, 'six': 6, 'seven': 7, 'eight': 8, 'nine': 9, 'ten': 10}
        hour_value = word_map.get(word_m.group(1).lower()) if word_m else None
    if not (pct_m and gap_m and hour_value):
        return None
    pct = sp.Rational(str(pct_m.group(1))) / 100
    gap = sp.Rational(str(gap_m.group(1)))
    hours = sp.Rational(str(hour_value))
    if pct <= 0 or hours <= 0:
        return None
    value = sp.simplify(gap / (pct * hours))
    opt_id = _unique_option_from_value(question, value, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_relative_speed_faster_gap',
        opt_id,
        f'Speed gap is {pct}*v; distance gap={gap} over {hours} hours, so v={sp.sstr(value)} mph.',
        0.97,
        {'percent_faster': sp.sstr(pct), 'distance_gap': sp.sstr(gap), 'hours': sp.sstr(hours), 'speed': sp.sstr(value)},
    )


def _shortcut_confidence_interval_size_change(question):
    text = normalize_text(get_question_text(question)).replace('\\%', '%')
    low = _normalize_for_text_match(text)
    if 'confidence interval' not in low or 'population proportion' not in low or '90%' not in text or '99%' not in text:
        return None
    dist = NormalDist()
    z90 = dist.inv_cdf(1 - (1 - 0.90) / 2)
    z99 = dist.inv_cdf(1 - (1 - 0.99) / 2)
    increase_pct = round((z99 / z90 - 1) * 100)
    opt_id = option_id_by_text(question, ['increase', str(int(increase_pct))])
    if opt_id is None:
        opt_id = option_id_by_text(question, ['increases', str(int(increase_pct))])
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_confidence_interval_z_width_change',
        opt_id,
        f'CI width is proportional to z*. z99/z90 - 1 ≈ {increase_pct}%.',
        0.92,
        {'z90': z90, 'z99': z99, 'increase_percent': increase_pct},
    )


def _shortcut_three_sided_fencing_max_area(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if not all(t in low for t in ['fencing', 'three sides', 'rectangular', 'maximum possible area']):
        return None
    if 'in terms of x' not in low and 'x feet' not in low:
        return None
    x = sp.Symbol('x')
    value = x**2 / 8
    opt_id = option_id_by_any_value([value], question, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_three_sided_fencing_max_area',
        opt_id,
        'With fencing x=2w+l, area=w(x-2w) is maximized at w=x/4, giving x^2/8.',
        0.97,
        {'area': 'x^2/8'},
    )


def _shortcut_binomial_model_concept(question):
    low = _normalize_for_text_match(get_question_text(question))
    if 'binomial probability model' not in low and 'binomial distribution is an appropriate model' not in low:
        return None
    # A binomial model needs fixed n, binary success/failure, independent-like repeated trials.
    for opt in get_options(question):
        opt_low = _normalize_for_text_match(opt.text)
        if (('out of' in opt_low or 'attempts' in opt_low or 'trials' in opt_low) and
            any(t in opt_low for t in ['times', 'success', 'can throw', 'attempt'])):
            return _decision_from_option(
                'math_binomial_model_fixed_trials_concept',
                int(opt.id),
                'Binomial is most reasonable for a fixed number of repeated attempts with success/failure outcome.',
                0.90,
                {'matched_option_text': opt.text},
            )
    return None


def _shortcut_max_acute_angles_convex_polygon(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if 'maximum number of acute angles' not in low or 'convex' not in low or 'gon' not in low:
        return None
    value = sp.Integer(3)
    opt_id = _unique_option_from_value(question, value, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_max_acute_angles_convex_polygon',
        opt_id,
        'Each acute interior angle has exterior angle > 90 degrees; exterior angles sum to 360, so at most 3.',
        0.92,
        {'max_acute_angles': 3},
    )



# ── V12 targeted deterministic handlers from V11 failure analysis ─────────
# These are conservative: they return only when the pattern is explicit and
# the computed value/fact matches exactly one option.

def _shortcut_signed_divisor_count(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if not ('positive and negative integers' in low and 'multiple of' in low):
        return None
    m = re.search(r'(?:is|are)\s+(\d+)\s+a\s+multiple\s+of', low)
    if not m:
        m = re.search(r'multiple\s+of\s+(\d+)', low)
    if not m:
        return None
    n = int(m.group(1))
    value = 2 * int(sp.divisor_count(n))
    opt_id = _unique_option_from_value(question, sp.Integer(value), tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_signed_divisor_count',
        opt_id,
        f'The positive and negative divisors of {n} are counted by 2*d({n})={value}.',
        0.98,
        {'n': n, 'signed_divisor_count': value},
    )


def _shortcut_function_point_square_transform(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if not ('on y = g(x)' in low or 'on y=g(x)' in low or 'on the graph of g' in low):
        return None
    if not ('h(x)' in low and ('g(x)^2' in low or 'g(x)²' in low or 'g(x) squared' in low)):
        return None
    if not ('sum' in low and 'coordinates' in low):
        return None
    m = re.search(r'\(\s*([-+]?\d+(?:\.\d+)?)\s*,\s*([-+]?\d+(?:\.\d+)?)\s*\)', text)
    if not m:
        return None
    x0 = sp.Rational(m.group(1))
    y0 = sp.Rational(m.group(2))
    value = sp.simplify(x0 + y0**2)
    opt_id = _unique_option_from_value(question, value, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_point_transform_g_squared_sum',
        opt_id,
        f'If (x,g(x))=({x0},{y0}), then h({x0})=g({x0})^2={y0**2}; coordinate sum is {value}.',
        0.97,
        {'x': sp.sstr(x0), 'g_x': sp.sstr(y0), 'sum': sp.sstr(value)},
    )


def _shortcut_equilateral_inradius_area(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if 'equilateral triangle' not in low or not any(t in low for t in ['inradius', 'inscribed circle', 'circle inscribed']):
        return None
    # Prefer a radius explicitly mentioned near the circle/inradius wording.
    m = re.search(r'(?:radius|inradius)[^\d]{0,20}(\d+(?:\.\d+)?)', low)
    if not m:
        nums = _extract_number_sequence(low)
        if not nums:
            return None
        r = sp.Rational(str(nums[0]))
    else:
        r = sp.Rational(m.group(1))
    area = sp.simplify(3 * sp.sqrt(3) * r**2)
    opt_id = option_id_by_any_value([area], question, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_equilateral_triangle_area_from_inradius',
        opt_id,
        f'For an equilateral triangle, area=3*sqrt(3)*r^2={area}.',
        0.97,
        {'inradius': sp.sstr(r), 'area': sp.sstr(area)},
    )


def _extract_quadratic_coeffs_from_y_expr(expr_text):
    x = sp.Symbol('x')
    expr, _ = _parse_one_variable_expr(expr_text, 'x')
    if expr is None:
        return None
    try:
        poly = sp.Poly(sp.expand(expr), x)
        if poly.degree() != 2:
            return None
        return poly, x
    except Exception:
        return None


def _shortcut_parabola_line_tangent_parameter(question):
    text = normalize_text(get_question_text(question)).replace('−', '-')
    low = _normalize_for_text_match(text)
    if 'tangent' not in low or 'parabola' not in low:
        return None
    if 'b' not in low:
        return None

    # Robustly extract the two equations from common wording:
    # "line y = 6x + b is tangent to the parabola y = x^2 + 2x + 7"
    line_m = re.search(r'line\s+y\s*=\s*(.+?)(?:\s+is\s+tangent|\s+tangent|,|\.|\?|$)', text, flags=re.I | re.S)
    para_m = re.search(r'parabola\s+y\s*=\s*(.+?)(?:,|\.|\?|$)', text, flags=re.I | re.S)

    # Fallback: collect all y=... spans and classify by x^2 vs b.
    if not line_m or not para_m:
        exprs = re.findall(r'y\s*=\s*([^,\n;.?]+)', text, flags=re.I)
        line_expr = None
        quad_expr = None
        for expr_text in exprs:
            if re.search(r'x\s*(?:\^|\*\*)\s*2|x²', expr_text):
                quad_expr = expr_text
            elif 'b' in expr_text.lower() and 'x' in expr_text.lower():
                line_expr = expr_text
    else:
        line_expr = line_m.group(1)
        quad_expr = para_m.group(1)

    if quad_expr is None or line_expr is None:
        return None

    x = sp.Symbol('x')
    B = sp.Symbol('B')
    quad, _ = _parse_one_variable_expr(quad_expr, 'x')
    if quad is None:
        return None

    line_clean = _math_text_for_parse(line_expr).replace(' ', '').replace('b', 'B')
    try:
        line = parse_expr(line_clean, local_dict={'x': x, 'B': B}, transformations=standard_transformations + (implicit_multiplication_application, convert_xor), evaluate=True)
    except Exception:
        return None

    diff = sp.expand(quad - line)
    poly = sp.Poly(diff, x)
    if poly.degree() != 2:
        return None
    A = poly.coeff_monomial(x**2)
    C = poly.coeff_monomial(x)
    D = poly.coeff_monomial(1)
    discr = sp.expand(C**2 - 4*A*D)
    sol = sp.solve(sp.Eq(discr, 0), B)
    if len(sol) != 1:
        return None
    value = sp.simplify(sol[0])
    opt_id = _unique_option_from_value(question, value, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_parabola_line_tangent_parameter_b',
        opt_id,
        f'Tangency requires discriminant 0 for {sp.sstr(diff)}=0, giving b={value}.',
        0.96,
        {'b': sp.sstr(value), 'discriminant': sp.sstr(discr)},
    )

def _shortcut_right_triangle_perimeter_hypotenuse(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if 'right triangle' not in low or 'hypotenuse' not in low or 'perimeter' not in low:
        return None
    nums = _extract_number_sequence(low)
    if len(nums) < 2:
        return None
    # Use explicit leg and perimeter. In common wording "one leg is 6" and "perimeter is 18".
    leg_m = re.search(r'(?:leg|side)[^\d]{0,20}(\d+(?:\.\d+)?)', low)
    per_m = re.search(r'perimeter[^\d]{0,20}(\d+(?:\.\d+)?)', low)
    if not leg_m or not per_m:
        return None
    a = sp.Rational(leg_m.group(1))
    P = sp.Rational(per_m.group(1))
    S = P - a
    if S <= 0:
        return None
    c = sp.simplify((S + a**2 / S) / 2)
    opt_id = option_id_by_any_value([c, float(sp.N(c))], question, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_right_triangle_hypotenuse_from_leg_perimeter',
        opt_id,
        f'Let other leg be b and hypotenuse c. b+c={S}, c-b=a^2/(b+c)={a**2/S}; c={c}.',
        0.96,
        {'leg': sp.sstr(a), 'perimeter': sp.sstr(P), 'hypotenuse': sp.sstr(c)},
    )


def _shortcut_ci_margin_error_concept(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if 'margin of error' not in low or 'confidence interval' not in low:
        return None
    if not any(t in low for t in ['smaller', 'decrease', 'reduce', 'narrower']):
        return None

    true_labels = []
    blocks = _parse_statement_blocks(text)
    for label, body in blocks:
        b = _normalize_for_text_match(body)
        is_true = None
        if 'smaller confidence' in b or 'lower confidence' in b or 'decrease confidence' in b:
            is_true = True
        elif 'larger confidence' in b or 'higher confidence' in b:
            is_true = False
        elif 'smaller sample standard deviation' in b or 'smaller standard deviation' in b:
            is_true = True
        elif 'larger sample standard deviation' in b or 'larger standard deviation' in b:
            is_true = False
        elif 'larger sample size' in b or 'increase sample size' in b:
            is_true = True
        elif 'smaller sample size' in b or 'decrease sample size' in b:
            is_true = False
        if is_true:
            true_labels.append(label)

    if blocks:
        opt_id = _match_roman_statement_option(question, true_labels)
        if opt_id is not None:
            return _decision_from_option(
                'math_ci_margin_error_decrease_statements',
                opt_id,
                f'Margin of error decreases with lower confidence, smaller standard deviation, or larger sample size. True statements={true_labels}.',
                0.92,
                {'true_statements': true_labels},
            )

    # Text options fallback.
    opt_id = option_id_by_text(question, ['smaller confidence', 'smaller standard deviation'])
    if opt_id is not None:
        return _decision_from_option(
            'math_ci_margin_error_decrease_concept',
            opt_id,
            'Margin of error decreases with lower confidence and smaller standard deviation.',
            0.88,
            {},
        )
    return None


def _shortcut_type_i_error_concept(question):
    low = _normalize_for_text_match(get_question_text(question))
    if 'type i error' not in low and 'type 1 error' not in low:
        return None
    # Type I: reject a true null.
    for opt in get_options(question):
        o = _normalize_for_text_match(opt.text)
        if ('reject' in o and 'true' in o and ('null' in o or 'hypothesis' in o)) or \
           (any(t in o for t in ['halt', 'stop', 'shut down', 'suspend']) and any(t in o for t in ['within specifications', 'meets specifications', 'acceptable', 'sufficient'])):
            return _decision_from_option(
                'math_type_i_error_concept',
                int(opt.id),
                'Type I error means rejecting a true null hypothesis.',
                0.90,
                {'matched_option_text': opt.text},
            )
    return None


def _shortcut_observational_vs_experiment(question):
    low = _normalize_for_text_match(get_question_text(question))
    if not any(t in low for t in ['observational', 'controlled experiment', 'experiment', 'assigned', 'assign', 'randomly assigned']):
        return None
    if not any(t in low for t in ['study', 'studies', 'researcher', 'diet', 'vegetarian', 'meat']):
        return None
    for opt in get_options(question):
        o = _normalize_for_text_match(opt.text)
        if ('first' in o and 'observational' in o and 'second' in o and ('controlled' in o or 'experiment' in o)):
            return _decision_from_option(
                'math_observational_vs_controlled_experiment',
                int(opt.id),
                'A study observing existing groups is observational; assigning treatments/diets is a controlled experiment.',
                0.90,
                {'matched_option_text': opt.text},
            )
    return None


def _shortcut_simple_random_sample_definition(question):
    low = _normalize_for_text_match(get_question_text(question))
    if 'simple random sample' not in low:
        return None
    for opt in get_options(question):
        o = _normalize_for_text_match(opt.text)
        if 'method of selection' in o or ('every' in o and 'equally likely' in o) or ('chosen' in o and 'random' in o and 'method' in o):
            return _decision_from_option(
                'math_simple_random_sample_definition',
                int(opt.id),
                'A simple random sample is defined by the random method of selection.',
                0.88,
                {'matched_option_text': opt.text},
            )
    return None


def _shortcut_permutation_true_false(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if 'permutation' not in low or 'one-to-one' not in low:
        return None
    blocks = _parse_statement_blocks(text)
    if not blocks:
        return None
    # Common pair:
    # I. Every permutation is one-to-one. -> True
    # II. Every function is a permutation iff it is one-to-one. -> False, needs bijective/onto.
    values = []
    for label, body in blocks:
        b = _normalize_for_text_match(body)
        if 'every permutation' in b and 'one-to-one' in b:
            values.append(True)
        elif 'every function' in b and 'permutation' in b and 'one-to-one' in b:
            values.append(False)
        else:
            return None
    # Match options like "True, False".
    for opt in get_options(question):
        o = _normalize_for_text_match(opt.text)
        if len(values) == 2:
            if values == [True, False] and (('true' in o and 'false' in o and o.find('true') < o.find('false')) or o in {'t, f', 'true, false'}):
                return _decision_from_option(
                    'math_permutation_true_false',
                    int(opt.id),
                    'Permutations are one-to-one; one-to-one alone is not enough for an arbitrary function to be a permutation.',
                    0.90,
                    {'truth_values': values},
                )
    return None


def _shortcut_square_divisor_sum(question):
    low = _normalize_for_text_match(get_question_text(question))
    if not ('n^2' in low or 'n²' in low or 'n squared' in low):
        return None
    if not any(t in low for t in ['factor of', 'divides', 'divisor of']):
        return None
    m = re.search(r'(?:factor of|divides|divisor of)\s+(\d+)', low)
    if not m:
        nums = [int(x) for x in re.findall(r'\d+', low)]
        if not nums:
            return None
        N = nums[-1]
    else:
        N = int(m.group(1))
    factors = sp.factorint(N)
    divisors = [1]
    for p, e in factors.items():
        max_exp = e // 2
        divisors = [d * (p ** k) for d in divisors for k in range(max_exp + 1)]
    value = sum(sorted(divisors))
    opt_id = _unique_option_from_value(question, sp.Integer(value), tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_square_divisor_sum',
        opt_id,
        f'If n^2 divides {N}, exponents in n are at most floor(e/2). Possible n={sorted(divisors)}, sum={value}.',
        0.98,
        {'N': N, 'n_values': sorted(divisors), 'sum': value},
    )


def _shortcut_linear_congruence_expression(question):
    text = normalize_text(get_question_text(question)).replace('−', '-')
    low = _normalize_for_text_match(text)
    if 'multiple of' not in low or 'mod' not in low and 'multiple of 7' not in low:
        # This handler is intentionally narrow; most seen cases used "multiples of 7".
        if 'multiples of 7' not in low and 'multiple of 7' not in low:
            return None
    if 'x' not in low or 'y' not in low or ' n' not in f' {low}':
        return None

    m_mod = re.search(r'multiples?\s+of\s+(\d+)|multiple\s+of\s+(\d+)', low)
    if not m_mod:
        return None
    mod = int(next(g for g in m_mod.groups() if g))

    # x-3 and y+3 are multiples of 7 -> x=3, y=-3 mod 7.
    mx = re.search(r'x\s*([+-])\s*(\d+)\s+(?:and\s+)?y\s*([+-])\s*(\d+)\s+are\s+multiples?\s+of', low)
    if not mx:
        return None
    sx, ax, sy, ay = mx.groups()
    x_val = int(ax) if sx == '-' else -int(ax)
    y_val = int(ay) if sy == '-' else -int(ay)

    if not ('x^2' in low or 'x²' in low) or 'xy' not in low or not ('y^2' in low or 'y²' in low):
        return None
    expr_val = (x_val*x_val + x_val*y_val + y_val*y_val) % mod
    n_val = (-expr_val) % mod
    # If the quiz asks for positive n and residue is 0, the smallest positive residue is mod.
    if n_val == 0 and 'positive' in low:
        n_val = mod
    opt_id = _unique_option_from_value(question, sp.Integer(n_val), tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_linear_congruence_expression',
        opt_id,
        f'x≡{x_val}, y≡{y_val} mod {mod}; x^2+xy+y^2≡{expr_val}; n≡{-expr_val} so n={n_val}.',
        0.96,
        {'mod': mod, 'x_residue': x_val % mod, 'y_residue': y_val % mod, 'n': n_val},
    )


def _shortcut_cardinal_rotation_degrees(question):
    low = _normalize_for_text_match(get_question_text(question))
    if not any(d in low for d in ['facing north', 'facing south', 'facing east', 'facing west']):
        return None
    if not any(t in low for t in ['spin', 'spins', 'turn', 'turns', 'rotates', 'rotation']):
        return None
    m_dir = re.search(r'facing\s+(north|east|south|west)', low)
    m_deg = re.search(r'(\d+(?:\.\d+)?)\s*degrees?', low)
    if not m_dir or not m_deg:
        return None
    direction = m_dir.group(1)
    deg = float(m_deg.group(1)) % 360
    right = any(t in low for t in ['right', 'clockwise'])
    left = any(t in low for t in ['left', 'counterclockwise', 'anticlockwise'])
    if not right and not left:
        return None
    if abs(deg % 90) > 1e-9:
        return None
    dirs = ['north', 'east', 'south', 'west']
    idx = dirs.index(direction)
    steps = int(round(deg / 90)) % 4
    new_dir = dirs[(idx + steps) % 4] if right else dirs[(idx - steps) % 4]
    opt_id = option_id_by_text(question, [new_dir])
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_cardinal_rotation_degrees',
        opt_id,
        f'{deg:g} degrees is {steps} quarter-turn(s); from {direction}, turning {"right" if right else "left"} gives {new_dir}.',
        0.98,
        {'start': direction, 'degrees_mod_360': deg, 'direction': new_dir},
    )


def _shortcut_periodic_lcm_inclusive_count(question):
    low = _normalize_for_text_match(get_question_text(question))
    if not any(t in low for t in ['blink', 'flash', 'ring', 'chime']):
        return None
    if not any(t in low for t in ['same time', 'simultaneously', 'together']):
        return None
    intervals = [int(x) for x in re.findall(r'every\s+(\d+)\s+seconds?', low)]
    if len(intervals) < 2:
        return None
    m_min = re.search(r'(\d+)\s+minutes?', low)
    m_sec = re.search(r'(\d+)\s+seconds?', low)
    if m_min:
        total = int(m_min.group(1)) * 60
    elif m_sec:
        total = int(m_sec.group(1))
    else:
        return None
    period = 1
    for v in intervals:
        period = int(sp.ilcm(period, v))
    value = total // period
    if any(t in low for t in ['including the beginning', 'include the beginning', 'beginning and end', 'including start']):
        value += 1
    opt_id = _unique_option_from_value(question, sp.Integer(value), tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_periodic_lcm_inclusive_count',
        opt_id,
        f'Simultaneous period is lcm({intervals})={period}s. Over {total}s with inclusive start count={value}.',
        0.97,
        {'intervals': intervals, 'period': period, 'total_seconds': total, 'count': value},
    )


def _shortcut_modular_group_identity(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if 'identity' not in low or 'modulo' not in low:
        return None
    if not any(t in low for t in ['multiplication', 'under multiplication', 'group']):
        return None
    set_match = re.search(r'\{([^{}]+)\}', text)
    mod_match = re.search(r'modulo\s+(\d+)', low)
    if not set_match or not mod_match:
        return None
    elements = [int(x) for x in re.findall(r'-?\d+', set_match.group(1))]
    mod = int(mod_match.group(1))
    if not elements:
        return None
    identity = None
    for e in elements:
        if all((e*a) % mod == a % mod and (a*e) % mod == a % mod for a in elements):
            identity = e
            break
    if identity is None:
        return None
    opt_id = _unique_option_from_value(question, sp.Integer(identity), tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_modular_group_identity',
        opt_id,
        f'Under multiplication mod {mod}, {identity} leaves every element of {elements} unchanged.',
        0.98,
        {'elements': elements, 'modulus': mod, 'identity': identity},
    )


def _shortcut_gcf_numbers(question):
    low = _normalize_for_text_match(get_question_text(question))
    if not any(t in low for t in ['greatest common factor', 'gcf', 'greatest common divisor', 'gcd']):
        return None
    nums = [int(x) for x in re.findall(r'\d+', low)]
    if len(nums) < 2:
        return None
    # Use the last two explicit numbers in common quiz wording.
    a, b = nums[-2], nums[-1]
    value = int(sp.igcd(a, b))
    opt_id = _unique_option_from_value(question, sp.Integer(value), tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option(
        'math_gcf_numbers',
        opt_id,
        f'gcd({a},{b})={value}.',
        0.98,
        {'a': a, 'b': b, 'gcd': value},
    )


def _shortcut_factor_difference_of_squares(question):
    text = normalize_text(get_question_text(question)).replace('−', '-')
    low = _normalize_for_text_match(text)
    if 'factor' not in low:
        return None
    m = re.search(r'factor\s+\$?([^?.\n]+)', text, flags=re.I)
    if not m:
        return None
    expr_text = m.group(1).strip().strip('$')
    if not re.search(r'x|[a-z]\s*(?:\^|\*\*)\s*2|²', expr_text, flags=re.I):
        return None
    expr, x = _parse_one_variable_expr(expr_text, 'x')
    if expr is None:
        return None
    factored = sp.factor(expr)
    # Compare factored expression to each option by symbolic equivalence.
    matches = []
    for opt in get_options(question):
        opt_expr, _ = _parse_one_variable_expr(opt.text, 'x')
        if opt_expr is None:
            continue
        try:
            opt_raw = normalize_text(opt.text).replace('²', '^2')
            if sp.simplify(opt_expr - expr) == 0 and ('(' in opt_raw or '*' in opt_raw):
                # Prefer the fully factored option; skip answers that still visibly contain x^2.
                if not re.search(r'x\s*(?:\^|\*\*)\s*2', opt_raw, flags=re.I):
                    matches.append(int(opt.id))
        except Exception:
            pass
    matches = sorted(set(matches))
    if len(matches) != 1:
        return None
    return _decision_from_option(
        'math_factor_polynomial',
        matches[0],
        f'Factored {sp.sstr(expr)} as {sp.sstr(factored)} and matched the equivalent factored option.',
        0.94,
        {'expr': sp.sstr(expr), 'factored': sp.sstr(factored)},
    )


# ── V13c: tool-recall deterministic handlers from answer_id logs ─────────────
def _shortcut_modular_expression_remainder(question):
    text = normalize_text(get_question_text(question)).replace('−', '-').replace('–', '-').replace('—', '-')
    low = _normalize_for_text_match(text)
    if 'remainder' not in low or not any(t in low for t in ['divided by', 'modulo', ' mod ']):
        return None
    patterns = [
        r'remainder\s+when\s+(.+?)\s+is\s+divided\s+by\s+(\d+)',
        r'remainder\s+of\s+(.+?)\s+when\s+(?:it\s+is\s+)?divided\s+by\s+(\d+)',
    ]
    for pat in patterns:
        m = re.search(pat, text, flags=re.I | re.S)
        if not m:
            continue
        expr_text = m.group(1).strip().strip(' ?.')
        modulus = int(m.group(2))
        # Reject natural-language fragments; accept only math-like expressions.
        if re.search(r'[A-Za-z]{2,}', expr_text.replace('sqrt', '')):
            continue
        expr = parse_math_expression(expr_text)
        if expr is None:
            continue
        value = int(sp.Mod(sp.Integer(expr) if expr.is_integer else expr, modulus))
        opt_id = _unique_option_from_value(question, sp.Integer(value), tolerance=1e-6)
        if opt_id is None:
            return None
        return _decision_from_option(
            'math_modular_expression_remainder',
            opt_id,
            f'Computed the full expression ({expr_text}) mod {modulus} = {value}.',
            0.99,
            {'expression': expr_text, 'modulus': modulus, 'remainder': value},
        )
    return None


def _shortcut_product_zn_max_order(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if not any(t in low for t in ['maximum possible order', 'largest possible order', 'maximal order']):
        return None
    if not re.search(r'Z[_\s]*\{?\d+\}?', text, flags=re.I) and 'z_' not in low:
        return None
    nums = [int(x) for x in re.findall(r'Z[_\s]*\{?(\d+)\}?', text, flags=re.I)]
    if not nums:
        nums = [int(x) for x in re.findall(r'Z_(\d+)', text, flags=re.I)]
    if len(nums) < 2:
        return None
    value = 1
    for n in nums:
        value = int(sp.ilcm(value, n))
    opt_id = _unique_option_from_value(question, sp.Integer(value), tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option('math_product_zn_max_order', opt_id, f'Maximum element order in a product of cyclic groups divides lcm({nums})={value}.', 0.97, {'moduli': nums, 'lcm': value})


def _shortcut_characteristic_even_integers(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if 'characteristic' not in low or 'ring' not in low:
        return None
    # Infinite rings such as 2Z have characteristic 0.
    if re.search(r'\b2\s*Z\b|2\\mathbb\{?Z\}?|2ℤ|2z\b', text, flags=re.I) or '2z' in low:
        opt_id = _unique_option_from_value(question, sp.Integer(0), tolerance=1e-6)
        if opt_id is None:
            return None
        return _decision_from_option('math_characteristic_infinite_subring', opt_id, 'The additive group 2Z is infinite, so no positive n makes n·a=0 for all a; characteristic is 0.', 0.96, {'ring': '2Z', 'characteristic': 0})
    return None


def _shortcut_least_n_sqrt_integer(question):
    text = normalize_text(get_question_text(question)).replace('−', '-')
    low = _normalize_for_text_match(text)
    if 'sqrt' not in low and 'square root' not in low and '√' not in text:
        return None
    if not any(t in low for t in ['least positive integer', 'smallest positive integer', 'least integer']):
        return None
    if not re.search(r'\bn\b', low):
        return None
    # Covers phrasings like sqrt(18*n*34) is an integer.
    nums = [int(x) for x in re.findall(r'\d+', low)]
    if len(nums) < 2:
        return None
    # Exclude option numbers by preferring numbers in the question text before options.
    qnums = [int(x) for x in re.findall(r'\d+', _normalize_for_text_match(get_question_text(question)))]
    constants = [x for x in qnums if x != 0]
    if len(constants) < 2:
        return None
    product = 1
    for c in constants:
        product *= c
    factors = sp.factorint(product)
    needed = 1
    for p, e in factors.items():
        if e % 2 == 1:
            needed *= int(p)
    opt_id = _unique_option_from_value(question, sp.Integer(needed), tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option('math_least_n_sqrt_integer', opt_id, f'Need n to supply odd prime exponents in {product}; squarefree part={needed}.', 0.97, {'constant_product': product, 'least_n': needed})


def _shortcut_group_order_no_involutions(question):
    low = _normalize_for_text_match(get_question_text(question))
    if 'group' not in low or not any(t in low for t in ['no involution', 'no involutions', 'no elements of order 2', 'no element of order 2']):
        return None
    subgroup_match = re.search(r'subgroup\s+of\s+order\s+(\d+)|subgroup\s+order\s+(\d+)', low)
    if not subgroup_match:
        return None
    subgroup_order = int(next(g for g in subgroup_match.groups() if g))
    candidates = []
    for opt_id, opt_text, opt_val in _option_sympy_values(question):
        if bool(getattr(opt_val, 'is_integer', False)):
            v = int(opt_val)
            if v > 0 and v % subgroup_order == 0 and v % 2 == 1:
                candidates.append(opt_id)
    candidates = sorted(set(candidates))
    if len(candidates) != 1:
        return None
    return _decision_from_option('math_group_order_no_involutions', candidates[0], f'By Lagrange order is a multiple of {subgroup_order}; no involutions implies odd order.', 0.92, {'subgroup_order': subgroup_order})


def _shortcut_torus_equation_standard(question):
    text = normalize_text(get_question_text(question))
    low = _normalize_for_text_match(text)
    if 'torus' not in low and 'rotat' not in low:
        return None
    if 'circle' not in low or 'radius' not in low:
        return None
    # Seen case: circle of radius 1 centered at distance 3 from axis -> R=3, r=1, coefficient 4R^2=36.
    nums = [float(x) for x in re.findall(r'\d+(?:\.\d+)?', low)]
    if not nums:
        return None
    radius = None
    center_dist = None
    mr = re.search(r'radius\s+(\d+(?:\.\d+)?)', low)
    if mr:
        radius = float(mr.group(1))
    # Try center coordinate like (3,0) or distance 3.
    mcenter = re.search(r'center(?:ed)?\s+(?:at\s+)?\(?\s*(\d+(?:\.\d+)?)\s*,\s*0\s*\)?', low)
    if mcenter:
        center_dist = float(mcenter.group(1))
    elif '3' in [str(int(n)) for n in nums]:
        center_dist = 3.0
    if radius is None or center_dist is None:
        return None
    coeff = sp.Integer(int(round(4 * center_dist * center_dist)))
    # Prefer matching options containing the key coefficient and squared x/y term.
    matches = []
    for opt in get_options(question):
        on = _normalize_for_text_match(opt.text).replace(' ', '')
        if str(coeff) in on and ('x^2+y^2' in on or 'x**2+y**2' in on or 'x2+y2' in on or 'x^2+z^2' in on):
            matches.append(int(opt.id))
    matches = sorted(set(matches))
    if len(matches) != 1:
        return None
    return _decision_from_option('math_torus_equation_standard', matches[0], f'Torus coefficient is 4R^2=4*{center_dist:g}^2={coeff}.', 0.86, {'R': center_dist, 'r': radius, 'coefficient': int(coeff)})


def _shortcut_linear_inverse_function(question):
    text = normalize_text(get_question_text(question)).replace('−', '-')
    low = _normalize_for_text_match(text)
    if 'inverse' not in low and 'f^-1' not in low and 'f^{-1}' not in text:
        return None
    m = re.search(r'f\s*\(\s*x\s*\)\s*=\s*([^,;.?]+)', text, flags=re.I)
    if not m:
        return None
    expr, x = _parse_one_variable_expr(m.group(1), 'x')
    if expr is None:
        return None
    poly = sp.Poly(sp.expand(expr), x)
    if poly.degree() != 1:
        return None
    y = sp.Symbol('x')
    inv = sp.solve(sp.Eq(y, expr.subs(x, sp.Symbol('t'))), sp.Symbol('t'))
    if not inv:
        return None
    inv_expr = sp.simplify(inv[0])
    matches = []
    for opt in get_options(question):
        opt_expr, _ = _parse_one_variable_expr(opt.text, 'x')
        if opt_expr is None:
            continue
        try:
            if sp.simplify(opt_expr - inv_expr) == 0:
                matches.append(int(opt.id))
        except Exception:
            pass
    matches = sorted(set(matches))
    if len(matches) != 1:
        return None
    return _decision_from_option('math_linear_inverse_function', matches[0], f'Inverting y={sp.sstr(expr)} gives f^-1(x)={sp.sstr(inv_expr)}.', 0.96, {'inverse': sp.sstr(inv_expr)})


def _shortcut_matrix_entries_i_plus_j_det_sum(question):
    low = _normalize_for_text_match(get_question_text(question))
    if 'det' not in low or not any(t in low for t in ['i+j', 'i + j', 'a_ij', 'entries']):
        return None
    if not any(t in low for t in ['deta', 'det a', 'determinant']):
        return None
    # Seen pattern: det(A)+det(B) for 2x2 and 3x3 matrices with entries i+j.
    dims = []
    for a, b in re.findall(r'(\d+)\s*(?:x|by)\s*(\d+)', low):
        if a == b:
            dims.append(int(a))
    if not dims:
        # Fallback to common A=2x2, B=3x3 wording when both A and B are present.
        if 'a' in low and 'b' in low:
            dims = [2, 3]
    if not dims:
        return None
    dets = []
    for n in dims[:3]:
        M = sp.Matrix([[i + j for j in range(1, n + 1)] for i in range(1, n + 1)])
        dets.append(sp.Integer(M.det()))
    value = sp.simplify(sum(dets))
    opt_id = _unique_option_from_value(question, value, tolerance=1e-6)
    if opt_id is None:
        return None
    return _decision_from_option('math_matrix_entries_i_plus_j_det_sum', opt_id, f'Determinants for dimensions {dims} are {dets}; sum={value}.', 0.9, {'dims': dims, 'dets': [int(d) for d in dets], 'sum': int(value)})


def _shortcut_vowel_vertical_symmetry(question):
    low = _normalize_for_text_match(get_question_text(question))
    if 'vowel' not in low or 'vertical' not in low or 'symmetry' not in low:
        return None
    # In standard block capitals, E is the vowel without vertical-axis symmetry.
    if any(t in low for t in ['not', 'does not', 'without', 'no vertical']):
        opt_id = option_id_by_text(question, ['e'])
        if opt_id is not None:
            return _decision_from_option('math_vowel_vertical_symmetry', opt_id, 'Among standard capital vowels, E lacks vertical-axis symmetry.', 0.8, {'answer': 'E'})
    return None

def try_math_deterministic_shortcuts_and_option_substitution(question):
    handlers = [
        _shortcut_modular_expression_remainder,
        _shortcut_product_zn_max_order,
        _shortcut_characteristic_even_integers,
        _shortcut_least_n_sqrt_integer,
        _shortcut_group_order_no_involutions,
        _shortcut_torus_equation_standard,
        _shortcut_linear_inverse_function,
        _shortcut_matrix_entries_i_plus_j_det_sum,
        _shortcut_vowel_vertical_symmetry,
        _shortcut_signed_divisor_count,
        _shortcut_square_divisor_sum,
        _shortcut_linear_congruence_expression,
        _shortcut_cardinal_rotation_degrees,
        _shortcut_periodic_lcm_inclusive_count,
        _shortcut_modular_group_identity,
        _shortcut_gcf_numbers,
        _shortcut_factor_difference_of_squares,
        _shortcut_function_point_square_transform,
        _shortcut_equilateral_inradius_area,
        _shortcut_parabola_line_tangent_parameter,
        _shortcut_right_triangle_perimeter_hypotenuse,
        _shortcut_ci_margin_error_concept,
        _shortcut_type_i_error_concept,
        _shortcut_observational_vs_experiment,
        _shortcut_simple_random_sample_definition,
        _shortcut_permutation_true_false,
        _shortcut_mean_after_adding_value,
        _shortcut_greatest_odd_factor_factorial,
        _shortcut_sum_of_squares_range,
        _shortcut_equal_base_exponential_sum,
        _shortcut_sphere_in_cube_probability,
        _shortcut_relative_speed_faster,
        _shortcut_confidence_interval_size_change,
        _shortcut_three_sided_fencing_max_area,
        _shortcut_binomial_model_concept,
        _shortcut_max_acute_angles_convex_polygon,
        _shortcut_mean_of_numbers,
        _shortcut_lcm_from_factors,
        _shortcut_constant_function,
        _shortcut_variance_without_independence,
        _shortcut_sqrt_domain_width,
        _shortcut_custom_star_operation,
        _shortcut_matrix_characteristic,
        _shortcut_ring_characteristic,
        _shortcut_rectangle_diagonal,
        _shortcut_positive_integer_sum_cubic,
        _shortcut_quadratic_extremum,
        _shortcut_function_composition,
        _shortcut_compound_doubling,
        _shortcut_coin_turn_game,
        _try_option_substitute_single_equation,
    ]
    errors = []
    for handler in handlers:
        try:
            decision = handler(question)
            if decision is not None:
                return decision, None
        except Exception as exc:
            errors.append(f'{handler.__name__}: {repr(exc)}')
    return None, '; '.join(errors[-3:]) if errors else 'no deterministic shortcut matched'



# ── V13: semantic tool retrieval router ───────────────────────────────
# This is a lightweight retrieval layer for tool selection. It is built before
# the Maths game starts, then used to bias the JSON tool router toward the most
# semantically relevant tools. It does not decide the final answer by itself.
USE_SEMANTIC_TOOL_ROUTER = True
SEMANTIC_TOOL_ROUTER_TOP_K = 5
SEMANTIC_TOOL_ROUTER_MIN_SCORE = 0.18
SEMANTIC_TOOL_ROUTER_CARDS = None
SEMANTIC_TOOL_ROUTER_EMBEDDINGS = None
LAST_SEMANTIC_TOOL_CANDIDATES = []

MATH_ROUTER_STOPWORDS = {
    'the', 'is', 'a', 'an', 'of', 'to', 'in', 'for', 'with', 'and', 'or', 'by', 'on',
    'at', 'from', 'which', 'what', 'following', 'answer', 'option', 'are', 'be', 'as',
    'that', 'this', 'these', 'those', 'if', 'then', 'find', 'compute', 'determine',
}

MATH_ROUTER_SYNONYMS = {
    'arithmetic mean': 'mean',
    'average': 'mean',
    'least positive integer': 'least common multiple',
    'smallest positive integer': 'least common multiple',
    'least number': 'least common multiple',
    'divisible by': 'multiple of',
    'has factors of': 'multiple of',
    'greatest common factor': 'gcd',
    'highest common factor': 'gcd',
    'greatest common divisor': 'gcd',
    'defined for': 'domain',
    'width of the domain': 'domain interval length',
    'cannot be determined': 'insufficient information',
    'not enough information': 'insufficient information',
    '⋆': ' star ',
    '\\star': ' star ',
    '★': ' star ',
    'one-half': '1/2',
    'one-third': '1/3',
}

TOOL_ROUTER_CARD_OVERRIDES = {
    'math_evaluate_expression': {
        'triggers': ['evaluate', 'value of expression', 'simplify numeric expression', 'compute expression'],
        'positive_examples': ['What is the value of 5*8+4?', 'Evaluate i+i^2+...+i^10.'],
        'negative_examples': ['Factor 36-9x^2', 'Find the identity element modulo 10.'],
    },
    'math_solve_equation': {
        'triggers': ['solve equation', 'solve for x', 'system of equations', 'unknown variable', 'linear equation', 'quadratic equation'],
        'positive_examples': ['Solve 2x+3=11.', 'Find x if 3^(x^2+4x+4)=9^(x+2).'],
        'negative_examples': ['Which theorem is true?', 'What is a Type I error?'],
    },
    'math_modular_arithmetic': {
        'triggers': ['remainder', 'units digit', 'power modulo', 'base exponent modulus'],
        'positive_examples': ['What is the remainder when 7^23 is divided by 10?', 'Find the units digit of 3^100.'],
        'negative_examples': ['Find the identity element of a group under multiplication modulo 10.'],
    },
    'math_binomial_probability': {
        'triggers': ['binomial', 'successes', 'at most', 'at least', 'exactly', 'fixed number of trials', 'probability of k'],
        'positive_examples': ['A fair die is rolled 5 times. Probability of at most two sixes?', 'Binomial mean and standard deviation.'],
        'negative_examples': ['Which sampling method is used?', 'Normal distribution percentile.'],
    },
    'math_normal_distribution': {
        'triggers': ['normal distribution', 'normally distributed', 'z score', 'cdf', 'upper tail', 'interquartile range'],
        'positive_examples': ['Demand is normally distributed with mean 2500 and std 225. What is P(X>3000)?'],
        'negative_examples': ['Binomial experiment with n trials.'],
    },
    'math_proportion_z_test': {
        'triggers': ['p-value', 'proportion test', 'hypothesis test', 'z test', 'sample proportion'],
        'positive_examples': ['Test p>0.5 with phat and n; find p-value.'],
        'negative_examples': ['Compute a confidence interval width concept.'],
    },
    'math_geometry': {
        'triggers': ['slope', 'distance', 'area', 'triangle', 'walked east north west south', 'point', 'line'],
        'positive_examples': ['Find the slope between two points.', 'How far from origin after walking east and north?'],
        'negative_examples': ['Permutation order in S_n.'],
    },
    'math_concept_classifier': {
        'triggers': ['type i error', 'type ii error', 'sampling error', 'blocking', 'observational study', 'simple random sample', 'confidence interval', 'correlation'],
        'positive_examples': ['What is a Type I error?', 'Which study is observational?', 'What makes margin of error smaller?'],
        'negative_examples': ['Evaluate a numeric expression.'],
    },
    'math_combinatorics_count': {
        'triggers': ['count', 'number of ways', 'complete graph', 'balls into boxes', 'morse code', 'trees'],
        'positive_examples': ['How many edges does a complete graph with 10 vertices have?'],
        'negative_examples': ['Solve a normal distribution probability.'],
    },
    'math_permutation_max_order': {
        'triggers': ['permutation', 'order', 'symmetric group', 'S_n'],
        'positive_examples': ['What is the largest order of a permutation of 7 objects?'],
        'negative_examples': ['Every permutation is one-to-one true false.'],
    },
    'math_finite_abelian_group_count': {
        'triggers': ['abelian group', 'order n', 'finite abelian groups', 'structurally distinct'],
        'positive_examples': ['How many abelian groups have order 720?'],
        'negative_examples': ['Identity element under multiplication modulo 10.'],
    },
    'math_independent_trials_probability': {
        'triggers': ['independent trials', 'coin', 'sequence', 'true false sequence', 'probability of sequence'],
        'positive_examples': ['Probability of TFFT in independent fair coin flips.'],
        'negative_examples': ['Coin game where players take turns until first heads.'],
    },
}


def normalize_math_router_text(text):
    text = normalize_text(text).lower()
    text = text.replace('−', '-').replace('–', '-').replace('—', '-')
    text = text.replace('²', '^2').replace('³', '^3')
    text = text.replace('×', '*').replace('÷', '/')
    for src, dst in MATH_ROUTER_SYNONYMS.items():
        text = text.replace(src, dst)
    text = re.sub(r'\\(?:left|right|,)', ' ', text)
    text = re.sub(r'[^a-z0-9_+\-*/().^=<>%\s]', ' ', text)
    tokens = re.findall(r'[a-z0-9_+\-*/().^=<>%]+', text)
    tokens = [t for t in tokens if t not in MATH_ROUTER_STOPWORDS]
    return ' '.join(tokens)


def _build_semantic_tool_router_cards():
    cards = []
    for name, spec in TOOL_SPECS.items():
        override = TOOL_ROUTER_CARD_OVERRIDES.get(name, {})
        triggers = override.get('triggers', [])
        positives = override.get('positive_examples', [])
        negatives = override.get('negative_examples', [])
        card_text = '\n'.join([
            f'Tool: {name}',
            f'Description: {spec.description}',
            f'Required args: {spec.required}',
            f'Optional args: {spec.optional}',
            'Triggers: ' + ', '.join(triggers),
            'Positive examples: ' + ' | '.join(positives),
            'Negative examples: ' + ' | '.join(negatives),
        ])
        cards.append({
            'tool': name,
            'description': spec.description,
            'triggers': triggers,
            'positive_examples': positives,
            'negative_examples': negatives,
            'text': normalize_math_router_text(card_text),
        })
    return cards


def warmup_math_semantic_tool_router():
    """Precompute tool-card embeddings before the game timer starts."""
    global SEMANTIC_TOOL_ROUTER_CARDS, SEMANTIC_TOOL_ROUTER_EMBEDDINGS
    if not globals().get('USE_SEMANTIC_TOOL_ROUTER', True):
        return None
    if SEMANTIC_TOOL_ROUTER_EMBEDDINGS is not None:
        return SEMANTIC_TOOL_ROUTER_EMBEDDINGS
    if 'embedding_model' not in globals() or embedding_model is None:
        return None
    SEMANTIC_TOOL_ROUTER_CARDS = _build_semantic_tool_router_cards()
    texts = [card['text'] for card in SEMANTIC_TOOL_ROUTER_CARDS]
    print(f'Warming up semantic tool router: {len(texts)} tool cards...')
    SEMANTIC_TOOL_ROUTER_EMBEDDINGS = embedding_model.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    ).astype('float32')
    return SEMANTIC_TOOL_ROUTER_EMBEDDINGS


def _semantic_keyword_bonus(query_norm, card):
    bonus = 0.0
    hits = []
    for trig in card.get('triggers', []):
        tnorm = normalize_math_router_text(trig)
        if tnorm and tnorm in query_norm:
            bonus += 0.04
            hits.append(trig)
    return min(bonus, 0.16), hits


def rank_semantic_tool_candidates(question, top_k=SEMANTIC_TOOL_ROUTER_TOP_K):
    global LAST_SEMANTIC_TOOL_CANDIDATES
    LAST_SEMANTIC_TOOL_CANDIDATES = []
    if not globals().get('USE_SEMANTIC_TOOL_ROUTER', True):
        return []
    embeddings = warmup_math_semantic_tool_router()
    if embeddings is None or not SEMANTIC_TOOL_ROUTER_CARDS:
        return []
    qtext = get_question_text(question)
    options = ' '.join(str(opt.text) for opt in get_options(question))
    query_norm = normalize_math_router_text(qtext + ' ' + options)
    q_emb = embedding_model.encode([query_norm], normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False).astype('float32')[0]
    base_scores = embeddings @ q_emb
    ranked = []
    for idx, card in enumerate(SEMANTIC_TOOL_ROUTER_CARDS):
        bonus, hits = _semantic_keyword_bonus(query_norm, card)
        score = float(base_scores[idx] + bonus)
        ranked.append({
            'tool': card['tool'],
            'score': round(score, 4),
            'embedding_score': round(float(base_scores[idx]), 4),
            'keyword_bonus': round(float(bonus), 4),
            'keyword_hits': hits[:5],
        })
    ranked.sort(key=lambda x: x['score'], reverse=True)
    candidates = [r for r in ranked[:top_k] if r['score'] >= SEMANTIC_TOOL_ROUTER_MIN_SCORE]
    if not candidates:
        candidates = ranked[:min(top_k, len(ranked))]
    LAST_SEMANTIC_TOOL_CANDIDATES = candidates
    return candidates


def _schemas_for_tool_names(tool_names):
    lines = []
    for name in tool_names:
        spec = TOOL_SPECS.get(name)
        if spec is None:
            continue
        lines.append(f'- {name}: {spec.description}. Required args: {spec.required}. Optional args: {spec.optional}.')
    return '\n'.join(lines)

def build_tool_router_prompt(question, previous_error=None, semantic_candidates=None):
    qtext = get_question_text(question)
    options = '\n'.join(f'{int(opt.id)}. {opt.text}' for opt in get_options(question))
    semantic_candidates = semantic_candidates or []
    candidate_names = [row['tool'] for row in semantic_candidates if row.get('tool') in TOOL_SPECS]
    candidate_schema = _schemas_for_tool_names(candidate_names)
    other_names = [name for name in TOOL_SPECS.keys() if name not in set(candidate_names)]
    other_schema = _schemas_for_tool_names(other_names)
    candidate_block = ''
    if semantic_candidates:
        candidate_lines = [
            f"- {row['tool']} score={row.get('score')} hits={row.get('keyword_hits', [])}"
            for row in semantic_candidates
        ]
        candidate_block = 'Semantic retrieval suggests these candidate tools, in order. Prefer them only if the required arguments are explicit in the question:\n' + '\n'.join(candidate_lines) + '\n\n'
    error_block = f'Previous rejected call: {previous_error}\n' if previous_error else ''
    return f"""/no_think
You are a strict JSON router for multiple-choice Maths questions.

Goal: select one executable tool and provide valid JSON arguments.
Use the semantic candidates as hints, not as proof. Try to choose a tool whenever the question has explicit numeric/symbolic data matching a schema.
Return no_tool only when no listed schema can be filled with concrete arguments from the question.
Do NOT choose a generic tool only because of a keyword. For example:
- "factor" means factor/symbolic equivalence, not evaluate_expression.
- "identity element" in a modular group is not modular_arithmetic.
- conceptual statistics questions usually need math_concept_classifier, not numeric probability.
If no schema fits with explicit data from the question, return {{"mathematical_analysis":"no supported deterministic tool", "tool_name":"no_tool", "arguments":{{}}}}.
No prose outside JSON.

{error_block}{candidate_block}Candidate tool schemas:
{candidate_schema if candidate_schema else '(none)'}

Other allowed tool schemas, use only if clearly better:
{other_schema}

Question:
{qtext}

Options:
{options}

/no_think
JSON:"""


def _tool_router_json_schema():
    return {
        'type': 'object',
        'properties': {
            'mathematical_analysis': {'type': 'string'},
            'tool_name': {'type': 'string', 'enum': sorted(list(TOOL_SPECS.keys()) + ['no_tool'])},
            'arguments': {'type': 'object'},
        },
        'required': ['mathematical_analysis', 'tool_name', 'arguments'],
    }

def run_local_tool_router_json(prompt):
    """Use the unified Qwen3.5-9B Q8_0 model for Maths tool routing."""
    math_llm = get_math_llm() if 'get_math_llm' in globals() else _resolve_llm()
    try:
        response = math_llm.create_chat_completion(
            messages=[{'role': 'user', 'content': prompt}],
            response_format={'type': 'json_object', 'schema': _tool_router_json_schema()},
            max_tokens=MATH_STRUCTURED_MAX_NEW_TOKENS,
            temperature=0.0,
            top_p=1.0,
        )
        return response['choices'][0]['message']['content'].strip()
    except Exception:
        return run_local_llm(
            prompt,
            max_new_tokens=MATH_STRUCTURED_MAX_NEW_TOKENS,
            stop=['<|im_end|>', '\n\nThe', '\n\nWait'],
            llm=math_llm,
        )



def llm_tool_router(question):
    start = time.time()
    semantic_candidates = rank_semantic_tool_candidates(question) if globals().get('USE_SEMANTIC_TOOL_ROUTER', True) else []
    prompt = build_tool_router_prompt(question, semantic_candidates=semantic_candidates)
    raw = run_local_tool_router_json(prompt)
    call, spec, args, error = parse_validated_tool_call(question, raw)
    candidate_json = json.dumps(semantic_candidates, ensure_ascii=False)
    if error is not None:
        validation = 'no_tool_selected' if error == 'no_tool_selected' else 'parse_or_validation_failed'
        _append_tool_trace(
            'llm_validated_tool_router', False, start,
            error=error, raw=raw, validation=validation,
            explanation='semantic_candidates=' + candidate_json[:700]
        )
        return None
    decision, exec_error = execute_validated_tool_call(question, {'tool': spec.name, 'args': args}, raw=raw)
    _append_tool_trace(
        'llm_validated_tool_router',
        decision is not None,
        start,
        error=exec_error,
        call={'tool': spec.name, 'args': args, 'semantic_candidates': semantic_candidates},
        raw=raw,
        validation='accepted' if decision is not None else 'tool_execution_failed',
        explanation=(decision.explanation if decision is not None else 'semantic_candidates=' + candidate_json[:700]),
    )
    return decision


try:
    MATH_MICRO_COT_GRAMMAR = LlamaGrammar.from_string(
        'root ::= line "\\n" line "\\n" line "\\n" "FINAL_CHOICE: " [0-3]\n'
        'line ::= char char*\n'
        'char ::= [A-Za-z0-9 .,;:=+*/()<>|_^%-]\n'
    ) if 'LlamaGrammar' in globals() else None
except Exception as exc:
    MATH_MICRO_COT_GRAMMAR = None
    print('Micro-CoT grammar unavailable; falling back to unconstrained Micro-CoT:', repr(exc))


def _map_answer_payload_to_option(payload, valid_ids, question=None):
    """Map a disobedient ANSWER payload to an option without confusing values with ids.

    The preferred prompt is ANSWER_ID: <0-3>. If the model instead writes a
    mathematical value such as ANSWER: 2/3 or ANSWER: 2.40, this helper maps that
    value/text to the corresponding option. A bare digit from ANSWER: is treated
    as an id only after value/text matching fails.
    """
    if question is None:
        return None
    payload = normalize_text(payload).strip().strip(' .,:;')
    if not payload:
        return None

    # Remove common prefixes while preserving the actual content.
    payload = re.sub(r'^(?:option|choice|id)\s*[:#\-]?\s*', '', payload, flags=re.I).strip()

    # Exact option text match first.
    payload_norm = _normalize_for_text_match(payload)
    matches = []
    for opt in get_options(question):
        if int(opt.id) in valid_ids and payload_norm == _normalize_for_text_match(opt.text):
            matches.append(int(opt.id))
    matches = sorted(set(matches))
    if len(matches) == 1:
        return matches[0]

    # Numeric/symbolic value match. This is the critical path for outputs like
    # ANSWER: 2/3, ANSWER: 2.40, ANSWER: sqrt(3), etc.
    for candidate in [payload, payload.replace('ANSWER_ID', '').replace('ANSWER', '')]:
        candidate = candidate.strip(' :=')
        val = _try_parse_math_value(candidate)
        if val is None:
            val = parse_math_expression(candidate) if 'parse_math_expression' in globals() else None
        if val is not None:
            oid = option_id_from_value(val, question, tolerance=1e-4)
            if oid is not None and oid in valid_ids:
                return oid

    # Number sequence fallback for option text like "z = 2.40" when payload is "2.40".
    nums = _extract_number_sequence(payload)
    if nums:
        oid = option_id_from_number_sequence(nums, question, tolerance=1e-3)
        if oid is not None and oid in valid_ids:
            return oid

    # Textual fallback for things like "I and II only".
    oid = option_id_from_text(payload, valid_ids, question=question)
    if oid is not None:
        return oid

    return None


def parse_micro_cot_choice(text, valid_ids, question=None):
    """Parse answer-id Micro-CoT output.

    Preferred output is exactly: ANSWER_ID: <0-3>.
    We distinguish ANSWER_ID from ANSWER because ANSWER: 2 may be a mathematical
    value, not option id 2. If the model disobeys and writes ANSWER: 2/3 or
    ANSWER: 2.40, map that value to the option text/value before treating digits
    as ids.
    """
    valid_ids = {int(x) for x in valid_ids}
    raw = normalize_text(text).strip()

    # 1) Preferred explicit option-id labels. Require a single option id, not a value.
    answer_id_patterns = [
        r'\bANSWER[\s_\-]*ID\s*(?:is|:|=)?\s*([0-3])\b(?!\s*(?:/|\.\d))',
        r'\bFINAL[\s_\-]*ANSWER[\s_\-]*ID\s*(?:is|:|=)?\s*([0-3])\b(?!\s*(?:/|\.\d))',
        r'\bFINAL[\s_\-]*OPTION(?:[\s_\-]*ID)?\s*(?:is|:|=)?\s*([0-3])\b(?!\s*(?:/|\.\d))',
        r'\bOPTION[\s_\-]*ID\s*(?:is|:|=)?\s*([0-3])\b(?!\s*(?:/|\.\d))',
    ]
    for pattern in answer_id_patterns:
        matches = list(re.finditer(pattern, raw, flags=re.I))
        if matches:
            option_id = int(matches[-1].group(1))
            if option_id in valid_ids:
                return option_id, 'answer_id_explicit_id'

    # 2) If the model wrote ANSWER: <payload>, treat payload as value/text first.
    # This fixes cases like ANSWER: 2/3, ANSWER: 2.40, or ANSWER: 2 where 2 is
    # the correct mathematical value but not the option id.
    payload_patterns = [
        r'\bANSWER\s*(?:is|:|=)\s*([^\n\r]+)',
        r'\bFINAL[\s_\-]*ANSWER\s*(?:is|:|=)\s*([^\n\r]+)',
        r'\bCHOICE\s*(?:is|:|=)\s*([^\n\r]+)',
    ]
    for pattern in payload_patterns:
        matches = list(re.finditer(pattern, raw, flags=re.I))
        if matches:
            payload = matches[-1].group(1).strip()
            mapped = _map_answer_payload_to_option(payload, valid_ids, question=question)
            if mapped is not None:
                return mapped, 'answer_payload_mapped_to_option'
            # If it is exactly a bare option id and not mappable as a value/text, accept it.
            if re.fullmatch(r'[0-3]', payload):
                option_id = int(payload)
                if option_id in valid_ids:
                    return option_id, 'answer_payload_bare_id_after_no_value_match'

    # 3) If the entire output is a bare id, accept it.
    cleaned = _clean_answer_text(raw)
    if re.fullmatch(r'[0-3]', cleaned):
        option_id = int(cleaned)
        if option_id in valid_ids:
            return option_id, 'answer_id_bare_id'

    # 4) Tail fallback: first map tail to option value/text, then accept option phrases.
    tail = '\n'.join([line for line in raw.splitlines() if line.strip()][-3:])
    if question is not None:
        mapped = _map_answer_payload_to_option(tail, valid_ids, question=question)
        if mapped is not None:
            return mapped, 'answer_tail_text_or_value_mapped_to_option'

    m = re.search(r'\b(?:option|choice|answer)\s*([0-3])\b', tail, flags=re.I)
    if m:
        option_id = int(m.group(1))
        if option_id in valid_ids:
            return option_id, 'answer_tail_option_phrase'

    return None, 'no_answer_id_parse'

MATH_MICRO_COT_SYSTEM = """/no_think
You answer multiple-choice Maths questions.

Use the answer options as constraints. For numeric or symbolic options, test/substitute the four options whenever possible.
Do any reasoning internally and output only the option id, not the mathematical value.

Important: return the option id. If the correct value is 2 and it appears as option 1, return ANSWER_ID: 1, not ANSWER_ID: 2.

Return exactly one line:
ANSWER_ID: <0-3>"""


def run_local_math_micro_cot(prompt, valid_ids, question=None):
    valid_ids = {int(x) for x in valid_ids}
    math_llm = get_math_llm() if 'get_math_llm' in globals() else _resolve_llm()

    full_prompt = f"""<|im_start|>system
{MATH_MICRO_COT_SYSTEM}<|im_end|>
<|im_start|>user
{prompt}<|im_end|>
<|im_start|>assistant
ANSWER_ID:"""

    common_kwargs = dict(
        max_tokens=MATH_MICRO_COT_MAX_TOKENS,
        temperature=0.0,
        top_p=1.0,
        top_k=40,
        repeat_penalty=1.05,
        stop=['<|im_end|>', '<|endoftext|>', '<|im_start|>', '\n\n'],
    )

    try:
        out = math_llm(full_prompt, **common_kwargs)
        raw = 'ANSWER_ID:' + out['choices'][0]['text'].strip()
    except Exception as exc:
        raw = '[micro_cot_error] ' + repr(exc)

    choice, parse_method = parse_micro_cot_choice(raw, valid_ids, question=question)

    if choice is None:
        choice = int(list(valid_ids)[0])
        parse_method = 'fallback_first_option_after_no_answer_id_parse'

    return choice, raw, parse_method


def retrieve_math_textbook_context(question):
    query = get_question_text(question)
    result_lists = [index.search(query, top_k=TOP_K_TEXTBOOK_BM25) for index in textbook_sparse_indexes.values()]
    result_lists.extend(index.search(query, top_k=TOP_K_DENSE) for index in textbook_dense_indexes.values())
    fused = rrf_fusion(result_lists)
    return rerank(query, fused)



def build_math_direct_prompt(question, docs=None):
    qtext = get_question_text(question)
    options = '\n'.join(f'{int(opt.id)}. {opt.text}' for opt in get_options(question))
    context = '\n\n'.join(
        f'[MATH DOC {i} | {doc.get("source", "unknown")} | score={doc.get("reranker_score", 0):.3f}]\n{doc["text"][:420]}'
        for i, doc in enumerate((docs or [])[:3], start=1)
    )
    context_block = f'\nTextbook context, only if needed for definitions/theorems:\n{context}\n' if context else ''
    return f"""Question:
{qtext}

Options:
{options}
{context_block}
Solve by reverse-checking the answer options whenever possible.
For numeric or symbolic options, substitute/test each option and choose the option id that satisfies the question.
Use textbook context only for definitions/theorems, not for routine calculations.
Return only one line: ANSWER_ID: <0-3>.
Do not return the mathematical value. Return the option id. For example, if the correct value is 2 but it is option 1, return ANSWER_ID: 1."""


def verify_micro_cot_answer(raw_output, question):
    """Extract the computed value from Micro-CoT reasoning and match to options."""
    import re
    # Find numbers/expressions in the reasoning (last line before FINAL_CHOICE)
    lines = [l.strip() for l in raw_output.split('\n') if l.strip() and not l.strip().startswith('FINAL')]
    if not lines:
        return None

    last_reasoning = lines[-1]
    # Extract numbers from the last reasoning line
    candidates = re.findall(r'[-+]?\d*\.?\d+(?:/\d+)?', last_reasoning)
    # Also try to find symbolic expressions like sqrt(3)/3
    symbolic = re.findall(r'(?:√|sqrt)\(?(\d+)\)?/(\d+)', last_reasoning)

    for opt in get_options(question):
        opt_text = str(opt.text).replace('$', '').replace('\\', '').strip()
        opt_lower = opt_text.lower()

        # Direct text match in reasoning
        for line in lines:
            line_clean = line.replace('$', '').replace('\\', '').lower()
            if opt_lower in line_clean:
                return int(opt.id)

        # Numeric match
        try:
            opt_val = float(sp.sympify(_math_text_for_parse(opt_text)))
            for c in candidates:
                try:
                    c_val = float(sp.sympify(c))
                    if abs(c_val - opt_val) < 0.05:
                        return int(opt.id)
                except:
                    pass
        except:
            pass

    return None


def llm_choose_math_option_direct(question):
    valid_ids = {int(opt.id) for opt in get_options(question)}
    docs = retrieve_math_textbook_context(question)
    prompt = build_math_direct_prompt(question, docs=docs)
    option_id, raw, parse_method = run_local_math_micro_cot(prompt, valid_ids, question=question)
    parsed = option_id is not None

    # Optional verifier. Disabled by default in V6 because the previous logs showed
    # that verify_override often changed correct-looking choices for weak reasons.
    if globals().get('USE_MATH_MICRO_COT_VERIFY_OVERRIDE', False):
        verified_id = verify_micro_cot_answer(raw, question)
        if verified_id is not None and verified_id in valid_ids:
            if verified_id != option_id:
                raw += f'\n[verify_override: {option_id}->{verified_id}]'
            option_id = verified_id

    if option_id is None:
        option_id = int(get_options(question)[0].id)
    return option_id, {
        'strategy': 'math_micro_cot_v13_answer_only_qwen25_math7b_lazy_gguf' if parsed else 'math_micro_cot_v13_answer_only_invalid_output_fallback_first_option',
        'decision_source': 'math_micro_cot',
        'confidence': 0.48 if parsed else 0.15,
        'raw_llm_output': raw,
        'micro_cot_parse_method': parse_method,
        'retrieved_context': docs,
        'math_tool_trace': json.dumps(LAST_MATH_TOOL_TRACE, ensure_ascii=False),
        'fallback_used': 'math_micro_cot_fallback' if parsed else 'first_option_invalid_math_micro_cot_output',
        **retrieval_score_summary(docs),
    }

def try_math_tools(question, use_llm_router=True):
    global LAST_MATH_TOOL_TRACE
    LAST_MATH_TOOL_TRACE = []

    # V6 layer: deterministic shortcuts and option substitution before any LLM call.
    if globals().get('USE_MATH_DETERMINISTIC_SHORTCUTS', True) or globals().get('USE_MATH_OPTION_SUBSTITUTION', True):
        start = time.time()
        decision, error = try_math_deterministic_shortcuts_and_option_substitution(question)
        _append_tool_trace(
            'deterministic_shortcuts_option_substitution',
            decision is not None,
            start,
            error=error,
            call=decision.validated_tool_call if decision is not None else None,
            validation='accepted' if decision is not None else 'no_match',
            explanation=decision.explanation if decision is not None else None,
        )
        if decision is not None:
            return decision

    deterministic_call = route_math_tool_deterministically(question)
    if deterministic_call is not None:
        start = time.time()
        decision, error = execute_validated_tool_call(question, deterministic_call)
        _append_tool_trace('deterministic_validated_router', decision is not None, start, error=error, call=deterministic_call, validation='accepted' if decision else 'failed', explanation=decision.explanation if decision is not None else None)
        if decision is not None:
            decision.raw_tool_call = json.dumps(deterministic_call, ensure_ascii=False)
            decision.validated_tool_call = deterministic_call
            return decision

    if use_llm_router:
        return llm_tool_router(question)
    return None



In [16]:


# ── Python Executor fallback for Maths V6 ──────────────────────────────
# This is now a constrained natural-language → Python/SymPy translator.
# It is still a fallback, but it is asked to prefer option substitution and print
# OPTION_ID: <id> whenever one of the four options satisfies the generated check.
import subprocess, tempfile, textwrap, ast

USE_MATH_NL_CODE_TRANSLATOR = True


def run_python_sandbox(code: str, timeout: int = 10) -> str:
    """Execute Python code in a subprocess with timeout. Returns stdout or error."""
    wrapped = textwrap.dedent("""\
import sympy as sp
import math
from fractions import Fraction
from itertools import combinations, permutations, product
from functools import reduce
from statistics import NormalDist

""" + code + "\n")
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(wrapped)
        f.flush()
        try:
            result = subprocess.run(
                ['python3', f.name],
                capture_output=True, text=True, timeout=timeout
            )
            output = result.stdout.strip()
            if not output and result.stderr:
                return f"[ERROR] {result.stderr.strip()[:500]}"
            return output
        except subprocess.TimeoutExpired:
            return "[ERROR] timeout"
        except Exception as exc:
            return f"[ERROR] {exc}"


def _executor_options_literal(question):
    return {int(opt.id): str(opt.text) for opt in get_options(question)}


def build_python_executor_prompt(question):
    qtext = get_question_text(question)
    options = _executor_options_literal(question)
    return f'''# You translate a multiple-choice Maths word problem into Python/SymPy.
# Prefer OPTION SUBSTITUTION: test the four options directly whenever possible.
# Print exactly one of:
#   OPTION_ID: <0-3>       if a unique option satisfies the computation/check
#   VALUE: <exact value>   if you compute a final value but not an option id
# No explanations, no markdown.
# Question: {qtext}
# Options dict: {options!r}
import sympy as sp
from sympy import *
from fractions import Fraction
import math
options = {options!r}

def parse_num(s):
    s = str(s).replace('$','').replace(',','').replace('−','-').replace('^','**')
    s = s.replace('√', 'sqrt')
    if s.endswith('%'):
        return sp.sympify(s[:-1]) / 100
    return sp.sympify(s)

'''


def extract_python_code(text):
    # Prefer fenced code if present.
    fence = re.search(r'```(?:python)?\s*(.*?)```', text, flags=re.I | re.S)
    if fence:
        text = fence.group(1)
    lines = text.split('\n')
    code_lines = []
    for line in lines:
        stripped = line.strip()
        if not stripped:
            if code_lines:
                code_lines.append(line)
            continue
        if stripped.startswith(('*', '>', 'You ', 'The ', 'To ', 'This ', 'Note', 'Here', 'I ', 'We ', 'Step', '```')):
            continue
        code_lines.append(line)
    # Cut after the last print statement — ignore trailing explanations/comments.
    last_print = -1
    for i, line in enumerate(code_lines):
        if 'print(' in line and not line.strip().startswith('#'):
            last_print = i
    if last_print >= 0:
        code_lines = code_lines[:last_print + 1]
    return '\n'.join(code_lines).strip()


def _parse_option_id_from_output(output, valid_ids):
    if not output:
        return None
    m = re.search(r'\b(?:OPTION_ID|FINAL_CHOICE|CHOICE)\s*[:=]\s*([0-3])\b', output, flags=re.I)
    if m:
        option_id = int(m.group(1))
        return option_id if option_id in valid_ids else None
    return None


def match_executor_output(output: str, question, tolerance=0.05):
    valid_ids = {int(opt.id) for opt in get_options(question)}
    if not output or output.startswith('[ERROR]'):
        return None, output

    labeled_option = _parse_option_id_from_output(output, valid_ids)
    if labeled_option is not None:
        return labeled_option, f'OPTION_ID:{labeled_option}'

    # Prefer VALUE: ... if present, otherwise use the first non-empty line.
    value_match = re.search(r'\bVALUE\s*[:=]\s*(.+)', output, flags=re.I)
    output_line = value_match.group(1).strip() if value_match else output.strip().split('\n')[0].strip()
    output_line = output_line.strip('{}[]() ')

    # Try exact option text first.
    low = output_line.lower().strip()
    for opt in get_options(question):
        opt_clean = str(opt.text).replace('$', '').replace('\\', '').strip().lower()
        if low == opt_clean or low == opt_clean.replace(',', ''):
            return int(opt.id), output_line

    # Numeric/symbolic match against options.
    try:
        candidate_expr = sp.sympify(_math_text_for_parse(output_line).replace(',', ''))
    except Exception:
        candidate_expr = None

    if candidate_expr is not None:
        for opt in get_options(question):
            opt_text = str(opt.text).replace('$', '').replace(',', '').replace('\\', '').strip()
            opt_text = opt_text.replace('−', '-').replace('^', '**')
            try:
                if opt_text.endswith('%'):
                    opt_expr = sp.sympify(_math_text_for_parse(opt_text[:-1])) / 100
                    opt_expr_alt = sp.sympify(_math_text_for_parse(opt_text[:-1]))
                    if _numeric_values_close(candidate_expr, opt_expr, tolerance) or _numeric_values_close(candidate_expr, opt_expr_alt, tolerance):
                        return int(opt.id), output_line
                else:
                    opt_expr = sp.sympify(_math_text_for_parse(opt_text))
                    if sp.simplify(opt_expr - candidate_expr) == 0 or _numeric_values_close(opt_expr, candidate_expr, tolerance):
                        return int(opt.id), output_line
            except Exception:
                pass

    return None, output_line


def python_executor_fallback(question):
    start = time.time()
    prompt = build_python_executor_prompt(question)

    math_llm = get_math_llm() if 'get_math_llm' in globals() else None
    raw_code = run_local_llm(
        prompt,
        max_new_tokens=320,
        stop=['<|im_end|>', '<|endoftext|>', '\n\n\n# Question:', '# Answer:'],
        temperature=0.0,
        llm=math_llm,
    )

    code = extract_python_code(raw_code)
    options = _executor_options_literal(question)
    executable = f"""
import sympy as sp
from sympy import *
from fractions import Fraction
import math
from itertools import combinations, permutations, product
options = {options!r}

def parse_num(s):
    s = str(s).replace('$','').replace(',','').replace('−','-').replace('^','**')
    s = s.replace('√', 'sqrt')
    if s.endswith('%'):
        return sp.sympify(s[:-1]) / 100
    return sp.sympify(s)

{code}
"""

    if len(code.strip()) < 5:
        return None, {
            'strategy': 'math_python_executor_no_code',
            'decision_source': 'python_executor',
            'confidence': 0.15,
            'raw_llm_output': f'[RAW]\n{raw_code}\n[EXTRACTED]\n{code}',
            'math_tool_trace': json.dumps(LAST_MATH_TOOL_TRACE, ensure_ascii=False),
            'fallback_used': 'python_executor_no_code',
        }

    output = run_python_sandbox(executable, timeout=8)
    option_id, matched = match_executor_output(output, question)

    # If first line did not match, try all output lines.
    if option_id is None and output and not output.startswith('[ERROR]'):
        for line in output.strip().split('\n'):
            option_id, matched = match_executor_output(line.strip(), question)
            if option_id is not None:
                break

    _append_tool_trace(
        'python_executor_nl_to_code', option_id is not None, start,
        error=None if option_id is not None else f'no match: {matched}',
        call=executable[:800], raw=output[:800],
        explanation=f'Executor output: {matched}'
    )

    return option_id, {
        'strategy': 'math_python_executor_nl_to_code_qwen25_math7b' if option_id is not None else 'math_python_executor_nl_to_code_no_match',
        'decision_source': 'python_executor',
        'confidence': 0.78 if option_id is not None else 0.15,
        'raw_llm_output': f'[CODE]\n{executable}\n[OUTPUT]\n{output}\n[RAW_MODEL]\n{raw_code}',
        'math_tool_trace': json.dumps(LAST_MATH_TOOL_TRACE, ensure_ascii=False),
        'fallback_used': 'python_executor_nl_to_code',
    }



In [17]:


# ── Wikipedia API retrieval fallback ─────────────────────────────────
import requests

WIKI_API_URL = "https://en.wikipedia.org/w/api.php"
WIKI_HEADERS = {'User-Agent': 'PoliMillionaire/1.0 (NLP course project; polimi.it)'}
WIKI_DELAY = 0.5


def _wiki_api(params, retries=2):
    for attempt in range(retries):
        try:
            time.sleep(WIKI_DELAY)
            r = requests.get(WIKI_API_URL, params=params, headers=WIKI_HEADERS, timeout=8)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 429:
                time.sleep(2)
                continue
        except Exception:
            pass
    return None


def _extract_relevant_chunks(full_text, question, options, chunk_size=500, max_chunks=3):
    keywords = set()
    for opt in options:
        for word in re.findall(r'\w{4,}', str(opt).lower()):
            keywords.add(word)
    for word in re.findall(r'\w{4,}', question.lower()):
        if word not in {'which', 'following', 'what', 'that', 'this', 'with', 'from', 'about', 'does', 'have', 'been', 'were', 'their', 'there', 'best', 'describes', 'primary', 'reason'}:
            keywords.add(word)
    paragraphs = [p.strip() for p in full_text.split('\n') if len(p.strip()) > 50]
    scored = []
    for p in paragraphs:
        p_low = p.lower()
        score = sum(1 for kw in keywords if kw in p_low)
        if score > 0:
            scored.append((score, p))
    scored.sort(key=lambda x: -x[0])
    chunks = [p for _, p in scored[:max_chunks]]
    return '\n\n'.join(chunks)[:chunk_size * max_chunks]


def wiki_retrieve(question_text, options, max_chars=2000):
    option_texts = [str(getattr(o, 'text', o)) for o in options]
    q_clean = re.sub(r'[\'\"\\$]', '', question_text)[:80]
    data = _wiki_api({
        'action': 'query', 'list': 'search', 'srsearch': q_clean,
        'srlimit': 3, 'format': 'json',
    })
    if not data:
        return []
    results = data.get('query', {}).get('search', [])
    if not results:
        return []
    titles = '|'.join(res['title'] for res in results[:2])
    data2 = _wiki_api({
        'action': 'query', 'titles': titles, 'prop': 'extracts',
        'explaintext': True, 'format': 'json',
    })
    if not data2:
        return []
    docs = []
    for p in data2.get('query', {}).get('pages', {}).values():
        full_text = p.get('extract', '')
        if not full_text:
            continue
        relevant = _extract_relevant_chunks(full_text, question_text, option_texts)
        if relevant:
            docs.append({
                'title': p.get('title', ''),
                'text': relevant[:max_chars],
                'source': 'wikipedia',
                'reranker_score': 0.5,
            })
    return docs

# Categories that benefit from Wikipedia fallback
WIKI_CATEGORIES = {'Entertainment', 'Ancient History and Politics', 'News'}

print('Wikipedia retrieval ready.')

# ── Google News RSS retrieval for News category ─────────────────────
from xml.etree import ElementTree

def google_news_search(query, max_results=5):
    try:
        url = f"https://news.google.com/rss/search?q={requests.utils.quote(query)}&hl=en&gl=US&ceid=US:en"
        r = requests.get(url, headers=WIKI_HEADERS, timeout=8)
        if r.status_code != 200:
            return []
        root = ElementTree.fromstring(r.content)
        docs = []
        for item in root.findall('.//item')[:max_results]:
            title = item.findtext('title', '')
            desc = item.findtext('description', '')
            clean_desc = re.sub(r'<[^>]+>', '', desc).replace('&nbsp;', ' ').strip()
            text = f"{title}. {clean_desc}" if clean_desc else title
            if text:
                docs.append({'text': text, 'source': 'google_news', 'reranker_score': 0.4})
        return docs
    except Exception:
        return []

WIKI_CATEGORIES = {'Entertainment', 'Ancient History and Politics'}
NEWS_CATEGORIES = {'News'}

print('Google News RSS ready.')



Wikipedia retrieval ready.
Google News RSS ready.




## 11. Routing policy

Maths uses deterministic shortcuts and option substitution first, then validated tools/JSON router, then a constrained Python/SymPy executor, and only then Micro-CoT fallback. Non-Maths questions use global RAG, with option-wise retrieval enabled for Entertainment and weak-evidence factual questions.



In [18]:


def answer_strategy(question, competition_name: str):
    """Routing policy used by the game loop."""
    valid_ids = {int(opt.id) for opt in get_options(question)}

    if competition_name == MATH_COMPETITION_NAME:
        decision = try_math_tools(question, use_llm_router=True)
        if decision is not None and int(decision.option_id) in valid_ids:
            return int(decision.option_id), {
                'strategy': decision.strategy,
                'decision_source': 'math_validated_tool',
                'confidence': float(decision.confidence),
                'math_llm_specialized': globals().get('MATH_LLM_IS_SPECIALIZED', False),
                'math_llm_device': globals().get('MATH_LLM_LOAD_DEVICE', None),
                'explanation': decision.explanation,
                'raw_llm_output': decision.raw_tool_call,
                'validated_tool_call': json.dumps(decision.validated_tool_call, ensure_ascii=False) if decision.validated_tool_call else None,
                'tool_validated': True,
                'tool_rejected_reason': None,
                'retrieved_context': [],
                'math_tool_trace': json.dumps(LAST_MATH_TOOL_TRACE, ensure_ascii=False),
                'fallback_used': None,
                'semantic_tool_candidates_json': json.dumps(LAST_SEMANTIC_TOOL_CANDIDATES, ensure_ascii=False),
                'semantic_tool_top_candidate': LAST_SEMANTIC_TOOL_CANDIDATES[0]['tool'] if LAST_SEMANTIC_TOOL_CANDIDATES else None,
                'prompt_version': PROMPT_VERSION,
            }

        # Python executor fallback before Micro-CoT
        exec_option_id, exec_meta = python_executor_fallback(question)
        if exec_option_id is not None and exec_option_id in valid_ids:
            exec_meta['tool_validated'] = False
            exec_meta['tool_rejected_reason'] = LAST_MATH_TOOL_TRACE[-1].get('error') if LAST_MATH_TOOL_TRACE else 'no_tool_match'
            exec_meta['prompt_version'] = PROMPT_VERSION
            exec_meta['semantic_tool_candidates_json'] = json.dumps(LAST_SEMANTIC_TOOL_CANDIDATES, ensure_ascii=False)
            exec_meta['semantic_tool_top_candidate'] = LAST_SEMANTIC_TOOL_CANDIDATES[0]['tool'] if LAST_SEMANTIC_TOOL_CANDIDATES else None
            exec_meta['math_llm_specialized'] = globals().get('MATH_LLM_IS_SPECIALIZED', False)
            exec_meta['math_llm_device'] = globals().get('MATH_LLM_LOAD_DEVICE', None)
            return exec_option_id, exec_meta

        option_id, meta = llm_choose_math_option_direct(question)
        meta['tool_validated'] = False
        meta['tool_rejected_reason'] = LAST_MATH_TOOL_TRACE[-1].get('error') if LAST_MATH_TOOL_TRACE else 'no_tool_match'
        meta['prompt_version'] = PROMPT_VERSION
        meta['semantic_tool_candidates_json'] = json.dumps(LAST_SEMANTIC_TOOL_CANDIDATES, ensure_ascii=False)
        meta['semantic_tool_top_candidate'] = LAST_SEMANTIC_TOOL_CANDIDATES[0]['tool'] if LAST_SEMANTIC_TOOL_CANDIDATES else None
        meta['math_llm_specialized'] = globals().get('MATH_LLM_IS_SPECIALIZED', False)
        meta['math_llm_device'] = globals().get('MATH_LLM_LOAD_DEVICE', None)
        return option_id, meta

    # ── Knowledge categories: local retrieval + external fallback ──
    # Non-Maths sections must use the general model only. If the lazy Maths model
    # is still resident from a previous Maths run, unload it before continuing.
    if 'qwen_math_llm' in globals() and qwen_math_llm is not None and 'unload_math_llm' in globals():
        unload_math_llm()

    docs = retrieve_and_rerank(get_question_text(question))

    # Wikipedia for Entertainment/History
    if competition_name in WIKI_CATEGORIES or (docs and docs[0].get('reranker_score', 0) < 1.0):
        try:
            wiki_docs = wiki_retrieve(get_question_text(question), get_options(question))
            if wiki_docs:
                docs = docs + wiki_docs
        except Exception:
            pass

    # Google News for News category
    if competition_name in NEWS_CATEGORIES:
        try:
            news_docs = google_news_search(get_question_text(question)[:80])
            if news_docs:
                docs = docs + news_docs
        except Exception:
            pass

    if should_use_option_retrieval(competition_name, docs):
        option_evidence, option_summary = retrieve_option_evidence(question)
        option_id, meta = llm_choose_option_with_option_evidence(question, docs, option_evidence, option_summary, competition_name)
    else:
        option_id, meta = llm_choose_option(question, docs, competition_name)

    if option_id not in valid_ids:
        option_id = int(get_options(question)[0].id)
        meta['strategy'] = 'invalid_option_id_fallback_first_option'
        meta['fallback_used'] = 'first_option_invalid_option_id'
        meta['confidence'] = 0.15
    meta['retrieved_context'] = docs
    meta['prompt_version'] = PROMPT_VERSION
    return option_id, meta





## 12. Dummy tests



In [19]:


class DummyOption:
    def __init__(self, id, text):
        self.id = id
        self.text = text

class DummyQuestion:
    def __init__(self, text, options, qid=0, level=1):
        self.id = qid
        self.text = text
        self.options = options
        self.level = level

q = DummyQuestion(
    text='Who was the first president of the United States?',
    options=[
        DummyOption(1, 'Abraham Lincoln'),
        DummyOption(2, 'George Washington'),
        DummyOption(3, 'Thomas Jefferson'),
        DummyOption(4, 'John Adams'),
    ],
)

option_id, meta = answer_strategy(q, 'Ancient History and Politics')
print('Predicted:', option_id)
print('Strategy:', meta.get('strategy'))
print('Raw LLM:', meta.get('raw_llm_output'))
for d in meta.get('retrieved_context', [])[:3]:
    print('DOC:', d.get('source'), d.get('reranker_score'), d['text'][:250])



BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/9 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/9 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/11 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/11 [00:00<?, ?it/s]

Predicted: 2
Strategy: hybrid_rag_option_evidence_qwen35_gguf_gbnf_adaptive
Raw LLM: 2
DOC: simplewiki 10.440590858459473 The first inauguration of George Washington as the president of the United States took place on April 30, 1789. The inauguration was the beginning of the first term of George Washington as president. John Adams had already taken office as vice presid
DOC: simplewiki 9.524194717407227 wrote the Constitution of the United States, and all of the states eventually agreed to it and joined the new government. of President George Washington]] Presidency On January 7, 1789, aged 56, Washington was elected as the first president of the Un
DOC: simplewiki 8.04497241973877 The first inauguration of Thomas Jefferson took place on March 4, 1801. Jefferson was sworn-in by Supreme Court Chief Justice John Marshall. Jefferson became the third president of the United States. It was the first presidential inauguration held in


In [20]:


q_math = DummyQuestion(
    text='What is the value of the expression 5*8+4?',
    options=[
        DummyOption(1, '40'),
        DummyOption(2, '42'),
        DummyOption(3, '44'),
        DummyOption(4, '48'),
    ],
)

option_id, meta = answer_strategy(q_math, 'Maths')
print('Predicted:', option_id)
print('Meta:', meta)



Predicted: 3
Meta: {'strategy': 'tool_math_evaluate_expression', 'decision_source': 'math_validated_tool', 'confidence': 0.96, 'math_llm_specialized': False, 'math_llm_device': 'unified_qwen35_q8', 'explanation': "Evaluated '5*8+4' = 44.", 'raw_llm_output': '{"tool": "math_evaluate_expression", "args": {"expression": "5*8+4"}}', 'validated_tool_call': '{"tool": "math_evaluate_expression", "args": {"expression": "5*8+4"}}', 'tool_validated': True, 'tool_rejected_reason': None, 'retrieved_context': [], 'math_tool_trace': '[{"tool": "deterministic_shortcuts_option_substitution", "matched": false, "latency": 0.0008234977722167969, "error": "no deterministic shortcut matched", "validation": "no_match"}, {"tool": "deterministic_validated_router", "matched": true, "latency": 0.004210472106933594, "error": null, "call": {"tool": "math_evaluate_expression", "args": {"expression": "5*8+4"}}, "validation": "accepted", "explanation": "Evaluated \'5*8+4\' = 44."}]', 'fallback_used': None, 'semantic



## 13. PoliMillionaire API loop skeleton



In [21]:


# V3 deterministic tool smoke tests. Run before API games.
def _assert_tool_choice(question, call, expected_id):
    decision, error = execute_validated_tool_call(question, call)
    assert error is None, error
    assert decision is not None, call
    assert int(decision.option_id) == int(expected_id), (decision, call)
    return decision

walk_q = DummyQuestion(
    text='A person walked 3 miles to the east, then turned north and walked 10 miles, then turned west and walked 6 miles, and finally turned south and walked 16 miles. Approximately how far is the person from his starting point in miles?',
    options=[DummyOption(0, '3.4'), DummyOption(1, '9.2'), DummyOption(2, '6.7'), DummyOption(3, '12.8')],
)
_assert_tool_choice(walk_q, {'tool': 'math_geometry', 'args': {'operation': 'cardinal_walk_distance', 'movements': walk_q.text}}, 2)

normal_q = DummyQuestion(
    text='Demand is normally distributed with mean 2500 and standard deviation 225. What is P(X > 3000)?',
    options=[DummyOption(0, '0.0132'), DummyOption(1, '0.9869'), DummyOption(2, '0.1667'), DummyOption(3, '0.8333')],
)
_assert_tool_choice(normal_q, {'tool': 'math_normal_distribution', 'args': {'operation': 'tail_probability', 'mean': 2500, 'std': 225, 'score': 3000}}, 0)

binom_q = DummyQuestion(
    text='We roll a fair 6-sided die 5 times. What is the probability that we get a 6 in at most 2 of the rolls?',
    options=[DummyOption(0, '\\frac{625}{648}'), DummyOption(1, '\\frac{25}{648}'), DummyOption(2, '\\frac{125}{648}'), DummyOption(3, '\\frac{1}{648}')],
)
_assert_tool_choice(binom_q, {'tool': 'math_binomial_probability', 'args': {'operation': 'at_most', 'n': 5, 'p': 1/6, 'k': 2}}, 0)

inverse_q = DummyQuestion(
    text='The numbers x and y are inversely proportional. When x+y=42 and x is twice y. What is y when x=-8?',
    options=[DummyOption(0, '-49'), DummyOption(1, '-7'), DummyOption(2, '40'), DummyOption(3, '-40')],
)
_assert_tool_choice(inverse_q, {'tool': 'math_solve_equation', 'args': {'equations': ['x+y=42', 'x-2*y=0', 'k-x*y=0', 'w+k/8=0'], 'variables': ['x', 'y', 'k', 'w'], 'target': 'w'}}, 0)

print('V3 deterministic Maths tool smoke tests passed.')


# V8 safe handler smoke tests.
def _assert_shortcut_choice(question, expected_id):
    decision, error = try_math_deterministic_shortcuts_and_option_substitution(question)
    assert error is None or decision is not None, error
    assert decision is not None, question.text
    assert int(decision.option_id) == int(expected_id), (decision, question.text)
    return decision

mean_q = DummyQuestion(
    text='Ten students took a quiz. Their scores were 45, 55, 50, 70, 65, 80, 40, 90, 70, and 85. What is the mean?',
    options=[DummyOption(0, '62'), DummyOption(1, '65'), DummyOption(2, '70'), DummyOption(3, '75')],
)
_assert_shortcut_choice(mean_q, 1)

lcm_q = DummyQuestion(
    text='What is the smallest positive integer that has factors of 16, 15, and 12?',
    options=[DummyOption(0, '60'), DummyOption(1, '120'), DummyOption(2, '180'), DummyOption(3, '240')],
)
_assert_shortcut_choice(lcm_q, 3)

constant_q = DummyQuestion(
    text='If f(x)=2 for all real numbers x, what is f(x+2)?',
    options=[DummyOption(0, '0'), DummyOption(1, '2'), DummyOption(2, '4'), DummyOption(3, 'x+2')],
)
_assert_shortcut_choice(constant_q, 1)

star_q = DummyQuestion(
    text='Let a ⋆ b = a^b - ab. If 2 ⋆ x = 22, find x.',
    options=[DummyOption(0, '11'), DummyOption(1, '5'), DummyOption(2, '6'), DummyOption(3, '22')],
)
_assert_shortcut_choice(star_q, 1)

sqrt_domain_q = DummyQuestion(
    text='What is the width of the domain of h(x) = sqrt(25-x^2) + sqrt(-(x-2))?',
    options=[DummyOption(0, '5'), DummyOption(1, '7'), DummyOption(2, '10'), DummyOption(3, '12')],
)
_assert_shortcut_choice(sqrt_domain_q, 1)

variance_q = DummyQuestion(
    text='Suppose X and Y are random variables with E(X)=37, var(X)=5, E(Y)=62, var(Y)=12. What are E(X+Y) and Var(X+Y)?',
    options=[DummyOption(0, 'E=99, Var=17'), DummyOption(1, 'E=99, Var cannot be determined'), DummyOption(2, 'E=25, Var=7'), DummyOption(3, 'Cannot be determined')],
)
_assert_shortcut_choice(variance_q, 1)


# V12 targeted handler smoke tests from recent failure analysis.
v12_cases = [
    (
        DummyQuestion(
            text='How many positive and negative integers is 12 a multiple of?',
            options=[DummyOption(0, '6'), DummyOption(1, '4'), DummyOption(2, '12'), DummyOption(3, '3')],
        ),
        2,
    ),
    (
        DummyQuestion(
            text='If (3,6) is on y = g(x), and h(x)=g(x)^2, what is the sum of the coordinates of the point on h?',
            options=[DummyOption(0, '36'), DummyOption(1, '39'), DummyOption(2, '42'), DummyOption(3, '9')],
        ),
        1,
    ),
    (
        DummyQuestion(
            text='An equilateral triangle has an inscribed circle with radius 2. What is its area?',
            options=[DummyOption(0, '12*sqrt(3)'), DummyOption(1, '16*sqrt(3)'), DummyOption(2, '8*sqrt(3)'), DummyOption(3, '4*sqrt(3)')],
        ),
        0,
    ),
    (
        DummyQuestion(
            text='A line y = 6x + b is tangent to the parabola y = x^2 + 2x + 7. What is b?',
            options=[DummyOption(0, '2'), DummyOption(1, '3'), DummyOption(2, '4'), DummyOption(3, '5')],
        ),
        1,
    ),
    (
        DummyQuestion(
            text='A right triangle has one leg 6 and perimeter 18. What is the hypotenuse?',
            options=[DummyOption(0, '7'), DummyOption(1, '15/2'), DummyOption(2, '8'), DummyOption(3, '9')],
        ),
        1,
    ),
    (
        DummyQuestion(
            text='Which statements decrease the margin of error of a confidence interval? I. smaller confidence level II. smaller sample standard deviation III. smaller sample size',
            options=[DummyOption(0, 'I only'), DummyOption(1, 'II only'), DummyOption(2, 'I and II'), DummyOption(3, 'I, II, and III')],
        ),
        2,
    ),
    (
        DummyQuestion(
            text='What is the sum of all positive integer values of n such that n^2 is a factor of 1200?',
            options=[DummyOption(0, '39'), DummyOption(1, '40'), DummyOption(2, '42'), DummyOption(3, '45')],
        ),
        2,
    ),
    (
        DummyQuestion(
            text='x-3 and y+3 are multiples of 7. Find the smallest positive n so x^2+xy+y^2+n is a multiple of 7.',
            options=[DummyOption(0, '4'), DummyOption(1, '5'), DummyOption(2, '6'), DummyOption(3, '7')],
        ),
        1,
    ),
    (
        DummyQuestion(
            text='A person facing north spins right 2250 degrees. Which direction is he facing?',
            options=[DummyOption(0, 'north'), DummyOption(1, 'east'), DummyOption(2, 'south'), DummyOption(3, 'west')],
        ),
        1,
    ),
    (
        DummyQuestion(
            text='Three lights blink every 2 seconds, every 3 seconds, and every 5 seconds. During a 7 minute dance, how many times do they blink together including the beginning and end?',
            options=[DummyOption(0, '14'), DummyOption(1, '15'), DummyOption(2, '16'), DummyOption(3, '30')],
        ),
        1,
    ),
    (
        DummyQuestion(
            text='G = {2,4,6,8} under multiplication modulo 10. What is the identity element?',
            options=[DummyOption(0, '2'), DummyOption(1, '4'), DummyOption(2, '6'), DummyOption(3, '8')],
        ),
        2,
    ),
]
for case_q, expected_id in v12_cases:
    _assert_shortcut_choice(case_q, expected_id)

print('V12 targeted deterministic Maths handler smoke tests passed.')



V3 deterministic Maths tool smoke tests passed.
V12 targeted deterministic Maths handler smoke tests passed.


In [22]:


# Fill these before running.
API_URL = 'http://131.175.15.22:51111/'

# Colab Secret names. Change these if your secrets use different names.
USERNAME_SECRET_NAME = 'USERNAME'
PASSWORD_SECRET_NAME = 'PASSWORD'

# Optional manual fallback. Leave as None when using Colab Secrets.
USERNAME = None
PASSWORD = None

# Number of full game attempts to run for each competition/category.
N_ATTEMPTS_PER_COMPETITION = 5

# Single cumulative log file for this notebook version.
RUN_LOG_PATH = LOG_DIR / 'run_qwen35_q8_unified_option_substitution_semantic_router_v13c_answer_id_tool_recall.csv'


def _read_colab_secret(secret_name):
    if not secret_name:
        return None
    try:
        if 'userdata' in globals() and userdata is not None:
            return userdata.get(secret_name)
    except Exception as e:
        print(f'Could not read Colab secret {secret_name}:', repr(e))
    return None


def setup_client():
    from millionaire_client import MillionaireClient
    username = USERNAME or _read_colab_secret(USERNAME_SECRET_NAME)
    password = PASSWORD or _read_colab_secret(PASSWORD_SECRET_NAME)
    if username is None or password is None:
        raise ValueError(
            'Set USERNAME/PASSWORD manually or create Colab Secrets named '
            f'{USERNAME_SECRET_NAME!r} and {PASSWORD_SECRET_NAME!r}'
        )
    client = MillionaireClient(API_URL)
    client.login(username, password)
    return client


def get_competitions(client):
    competitions = client.competitions.list_all()
    for comp in competitions:
        print(comp.id, comp.name, getattr(comp, 'max_levels', None))
    return competitions


def get_competition_names(client):
    return {comp.id: comp.name for comp in get_competitions(client)}


def _serialize_retrieved_context(meta):
    return json.dumps([
        {
            'source': d.get('source'),
            'idx': d.get('idx'),
            'reranker_score': d.get('reranker_score'),
            'text': d.get('text', '')[:500],
        }
        for d in meta.get('retrieved_context', [])
    ], ensure_ascii=False)


def _retrieved_docs(meta):
    docs = meta.get('retrieved_context', []) if isinstance(meta, dict) else []
    return docs if isinstance(docs, list) else []


def _retrieval_sources(meta):
    sources = sorted({str(d.get('source')) for d in _retrieved_docs(meta) if d.get('source')})
    return json.dumps(sources, ensure_ascii=False)


def _textbook_docs(meta):
    return [d for d in _retrieved_docs(meta) if str(d.get('source', '')).startswith('textbook_')]


def _textbook_context_summary(meta):
    docs = _textbook_docs(meta)
    sources = sorted({str(d.get('source')) for d in docs if d.get('source')})
    top_doc = docs[0] if docs else {}
    return {
        'textbook_context_used': bool(docs),
        'textbook_context_count': len(docs),
        'textbook_context_sources': json.dumps(sources, ensure_ascii=False),
        'textbook_top_source': top_doc.get('source'),
        'textbook_top_reranker_score': top_doc.get('reranker_score'),
    }


def append_logs(df, output_csv=RUN_LOG_PATH):
    if df is None or df.empty:
        return
    output_csv = Path(output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)
    write_header = not output_csv.exists() or output_csv.stat().st_size == 0
    df.to_csv(output_csv, mode='a', header=write_header, index=False)


def _meta_get(meta, key, default=None):
    return meta.get(key, default) if isinstance(meta, dict) else default


def run_competition(client, comp_id, competition_names, attempt_number=None, run_id=None):
    competition_name = competition_names[comp_id]
    logs = []
    session_started_at = time.strftime('%Y-%m-%d %H:%M:%S')

    try:
        game = client.game.start(competition_id=comp_id)
    except Exception as e:
        return pd.DataFrame([{
            'run_id': run_id,
            'attempt_number': attempt_number,
            'session_started_at': session_started_at,
            'session_id': None,
            'competition_id': comp_id,
            'competition_name': competition_name,
            'error_message': repr(e),
        }])

    while game.in_progress:
        question = game.current_question
        if question is None:
            break

        time_remaining_before = getattr(game, 'time_remaining', None)
        game_current_level_before = getattr(game, 'current_level', None)
        start = time.time()
        try:
            option_id, meta = answer_strategy(question, competition_name)
            latency = time.time() - start
            result = game.answer(option_id)
            textbook_summary = _textbook_context_summary(meta)
            logs.append({
                'run_id': run_id,
                'attempt_number': attempt_number,
                'session_started_at': session_started_at,
                'session_id': getattr(game, 'session_id', None),
                'competition_id': comp_id,
                'competition_name': competition_name,
                'question_id': getattr(question, 'id', None),
                'question_level': getattr(question, 'level', None),
                'game_current_level_before': game_current_level_before,
                'time_remaining_before': time_remaining_before,
                'question_text': get_question_text(question),
                'options_json': json.dumps([(int(o.id), o.text) for o in get_options(question)], ensure_ascii=False),
                'chosen_option_id': option_id,
                'correct': getattr(result, 'correct', None),
                'timed_out': getattr(result, 'timed_out', None),
                'game_over': getattr(result, 'game_over', None),
                'earned_amount': getattr(result, 'earned_amount', None),
                'latency_seconds': latency,
                'strategy': _meta_get(meta, 'strategy'),
                'decision_source': _meta_get(meta, 'decision_source'),
                'confidence': _meta_get(meta, 'confidence'),
                'explanation': _meta_get(meta, 'explanation'),
                'raw_llm_output': _meta_get(meta, 'raw_llm_output'),
                'prompt_version': _meta_get(meta, 'prompt_version', PROMPT_VERSION),
                'semantic_tool_top_candidate': _meta_get(meta, 'semantic_tool_top_candidate'),
                'semantic_tool_candidates_json': _meta_get(meta, 'semantic_tool_candidates_json'),
                'retrieved_context': _serialize_retrieved_context(meta),
                'retrieval_sources': _retrieval_sources(meta),
                'retrieval_top_score': _meta_get(meta, 'retrieval_top_score'),
                'retrieval_second_score': _meta_get(meta, 'retrieval_second_score'),
                'retrieval_margin': _meta_get(meta, 'retrieval_margin'),
                'option_retrieval_top_score': _meta_get(meta, 'option_retrieval_top_score'),
                'option_retrieval_second_score': _meta_get(meta, 'option_retrieval_second_score'),
                'option_retrieval_margin': _meta_get(meta, 'option_retrieval_margin'),
                'option_evidence_scores_json': _meta_get(meta, 'option_evidence_scores_json'),
                'option_evidence_json': _meta_get(meta, 'option_evidence_json'),
                'tool_validated': _meta_get(meta, 'tool_validated'),
                'validated_tool_call': _meta_get(meta, 'validated_tool_call'),
                'tool_rejected_reason': _meta_get(meta, 'tool_rejected_reason'),
                'math_tool_trace': _meta_get(meta, 'math_tool_trace'),
                'fallback_used': _meta_get(meta, 'fallback_used'),
                'micro_cot_parse_method': _meta_get(meta, 'micro_cot_parse_method'),
                **textbook_summary,
                'error_message': None,
            })
        except Exception as e:
            logs.append({
                'run_id': run_id,
                'attempt_number': attempt_number,
                'session_started_at': session_started_at,
                'session_id': getattr(game, 'session_id', None),
                'competition_id': comp_id,
                'competition_name': competition_name,
                'question_id': getattr(question, 'id', None),
                'question_level': getattr(question, 'level', None),
                'game_current_level_before': game_current_level_before,
                'time_remaining_before': time_remaining_before,
                'question_text': get_question_text(question),
                'options_json': json.dumps([(int(o.id), o.text) for o in get_options(question)], ensure_ascii=False),
                'retrieval_sources': '[]',
                'math_tool_trace': None,
                'fallback_used': 'exception',
                'error_message': repr(e),
            })
            break

    return pd.DataFrame(logs)


def warmup_math_model_if_needed(comp_name, warmed=False):
    if comp_name == MATH_COMPETITION_NAME and not warmed:
        if 'warmup_math_semantic_tool_router' in globals():
            print('Warming up semantic tool router before starting the Maths game timer...')
            _ = warmup_math_semantic_tool_router()
        if 'get_math_llm' in globals():
            print('Warming up unified Qwen3.5-9B Q8_0 before starting the Maths game timer...')
            _ = get_math_llm()
        return True
    return warmed


def run_all_competitions(client, attempts_per_competition=N_ATTEMPTS_PER_COMPETITION, output_csv=RUN_LOG_PATH):
    competition_names = get_competition_names(client)
    all_logs = []
    run_id = time.strftime('%Y%m%d_%H%M%S')
    math_model_warmed = False

    for attempt in range(1, attempts_per_competition + 1):
        for comp_id, comp_name in competition_names.items():
            print(f'Run {run_id} | attempt {attempt}/{attempts_per_competition} | {comp_id}: {comp_name}')
            math_model_warmed = warmup_math_model_if_needed(comp_name, math_model_warmed)
            df_logs = run_competition(
                client,
                comp_id,
                competition_names,
                attempt_number=attempt,
                run_id=run_id,
            )
            append_logs(df_logs, output_csv=output_csv)
            all_logs.append(df_logs)
            print(f'Appended {len(df_logs)} rows to {output_csv}')
            time.sleep(1.0)
            cleanup_memory()

    if not all_logs:
        return pd.DataFrame()
    return pd.concat(all_logs, ignore_index=True)





## 14. **Start Game**



In [23]:


client = setup_client()
#df_logs = run_all_competitions(
  #  client,
    #attempts_per_competition=N_ATTEMPTS_PER_COMPETITION,
#)
#print(RUN_LOG_PATH)



In [24]:


import pandas as pd

try:
    df = pd.read_csv(RUN_LOG_PATH, on_bad_lines='skip')
    df_keep = df[df['competition_name'] == 'Maths']
    df_keep.to_csv(RUN_LOG_PATH, index=False)
    print(f"Filtrate {len(df) - len(df_keep)} righe, tenute {len(df_keep)} per 'Maths' competitions.")
except FileNotFoundError:
    print(f"Error: The log file '{RUN_LOG_PATH}' was not found.")
    print("This file is generated by the 'run_all_competitions' function in the previous cell.")
    print("Please ensure that the previous cell (cell RshF6lLQ7HZe) has executed successfully to create this log file.")
    print("If cell RshF6lLQ7HZe was recently uncommented, re-run it to generate the file.")
    df = pd.DataFrame() # Initialize an empty DataFrame to prevent further errors
except Exception as e:
    print(f"An unexpected error occurred: {e}")
    df = pd.DataFrame()



Error: The log file '/content/drive/MyDrive/nlp26/logs/run_qwen35_q8_unified_option_substitution_semantic_router_v13c_answer_id_tool_recall.csv' was not found.
This file is generated by the 'run_all_competitions' function in the previous cell.
Please ensure that the previous cell (cell RshF6lLQ7HZe) has executed successfully to create this log file.
If cell RshF6lLQ7HZe was recently uncommented, re-run it to generate the file.


In [25]:


# Run only specific competitions
SELECTED_COMPETITIONS = {'Maths'}
SELECTED_ATTEMPTS = 15

def run_selected_competitions(client, competitions, attempts=5):
    competition_names = get_competition_names(client)
    all_logs = []
    run_id = time.strftime('%Y%m%d_%H%M%S_selected')
    math_model_warmed = False

    for attempt in range(1, attempts + 1):
        for comp_id, comp_name in competition_names.items():
            if comp_name not in competitions:
                continue
            print(f'Run {run_id} | attempt {attempt}/{attempts} | {comp_id}: {comp_name}')
            math_model_warmed = warmup_math_model_if_needed(comp_name, math_model_warmed)
            df = run_competition(client, comp_id, competition_names, attempt_number=attempt, run_id=run_id)
            append_logs(df, output_csv=RUN_LOG_PATH)
            all_logs.append(df)
            earned = df['earned_amount'].iloc[-1] if len(df) > 0 and 'earned_amount' in df.columns else 0
            earned_str = f"${int(earned):,}" if earned else "$0"
            print(f'Appended {len(df)} rows on {comp_name}: reached {earned_str}')
            time.sleep(1.0)
            cleanup_memory()

    return pd.concat(all_logs, ignore_index=True) if all_logs else pd.DataFrame()

client = setup_client()
df = run_selected_competitions(client, SELECTED_COMPETITIONS, SELECTED_ATTEMPTS)



0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15
4 Philosophy and Psychology 15
5 News 15
Run 20260529_143503_selected | attempt 1/15 | 3: Maths
Warming up semantic tool router before starting the Maths game timer...
Warming up semantic tool router: 17 tool cards...
Warming up unified Qwen3.5-9B Q8_0 before starting the Maths game timer...
Appended 4 rows on Maths: reached $300
Run 20260529_143503_selected | attempt 2/15 | 3: Maths


BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/45 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/45 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/40 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/40 [00:00<?, ?it/s]

Appended 5 rows on Maths: reached $500


/tmp/ipykernel_40107/2362576127.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(all_logs, ignore_index=True) if all_logs else pd.DataFrame()
